# Jev × diffusion: attention control for hands


A chronological research notebook using **Stable Diffusion 1.5** and Jev. The later experiments continue from the original **seed 123** image.


## Reading guide

Use the **Table of Contents on the left** to jump between sections.


- 1 · Environment and model
- 2 · Attention implementation and observations
- 3 · Initial downsampling study
- 4 · Seed 123: three-round continuation
- 5 · Seed 123: 50 attention decisions
- 6 · Actual architecture audit


**Start with the full 50-decision progression** for the latest images, then the architecture audit for the available model components. Expand folded code to inspect implementation; the controller prompt cells remain open.


| Stage | What was controlled | Reading the outcome |
|---|---|---|
| Initial study | Spatial key/value retention | Historical four-mode, three-seed experiment |
| Three-round continuation | Feature-driven retention choices | All nine decisions selected full attention; no established anatomy repair |
| 50-decision run | Eight additive attention-logit biases, jointly updated | Full chronological record, including holds and regressions; visual change is not proof of anatomical improvement |


**Record integrity:** original experiment code, execution counts, outputs, and chronological order are preserved. Formatting does not rerun diffusion or Jev. Execution counts reflect the interactive research history. Existing setup errors remain in folded outputs. Running all cells again would repeat paid API calls and generation.



<a id="environment"></a>

## 1 · Environment and model

Dependencies, fixed settings, cached model weights, credentials loaded privately, and observer setup. Installation logs are folded.

In [ ]:
# 1. Install the hand-region observer before importing numerical libraries
%pip -q install 'mediapipe>=0.10.30,<0.11'
print('Hand observer dependency installed. Diffusers and model weights are already cached.')

### Original experimental brief

**Goal:** test whether Jev can choose useful interventions inside SD1.5 while generating an open hand. The controller is the experiment, not an optional extra.

**Actuator:** ToDo-style spatial key/value downsampling, keeping all queries and text cross-attention intact. Actions include full attention, uniform downsampling, and preserving dense tokens around detected hand regions. Registers play a different role and are not needed for this actuator. This notebook implements the core operation with installed Diffusers primitives; it is not an exact reproduction of the repository benchmarks.

**Loop:** current latent → short counterfactual diffusion branches → decoded predicted-clean previews → hand landmarks plus a local visual-language observer → goal, uncertain anatomy observations, action effects and recent history → Jev probability distributions → selected attention policy and branch → continue diffusion.

Compare identical seeds under **full attention**, **fixed uniform downsampling**, **deterministic counterfactual routing**, and **Jev counterfactual routing**. Both routers receive the same observer data and choose from the same actions. Preserve every result and every TypeSafe response.

This uses both mechanisms: a conditional operation inside attention, selected by feedback during the diffusion loop. Changing information density does not itself enforce five-finger anatomy; we will inspect whether the intervention actually helps.

The visual observer's finger counts and defect reports are fallible model claims. MediaPipe predicts a fixed 21-landmark template, so 21 points do not prove five correct fingers. Final image grids and explicit uncertainty are essential.

Sources: [ToDo repository](https://github.com/ethansmith2000/ImprovedTokenMerge), [paper](https://arxiv.org/abs/2402.13573), [MediaPipe](https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker/python), [TypeSafe API](https://docs.typesafe.ai/api).

The key is loaded from man.env without printing it. Only synthetic image observations and experiment metrics are sent to TypeSafe; visual inference runs on this GPU.

In [ ]:
# 2. Independent runtime, cached SD1.5 weights, and fixed experimental settings
import sys,time,json,math,hashlib,importlib.metadata as metadata
from pathlib import Path
import numpy as np,pandas as pd,matplotlib.pyplot as plt
from PIL import Image,ImageDraw
import torch
import torch.nn.functional as F
from diffusers import StableDiffusionPipeline,DDIMScheduler
from diffusers.models.attention_processor import AttnProcessor2_0
from IPython.display import display,Markdown,HTML,Image as DisplayImage
import mediapipe as mp,requests
DEVICE,DTYPE='cuda',torch.float16
SIZE,STEPS,CFG=512,36,7.5
SEEDS=[17,42,123]
MODES=['full','uniform_kv','fixed_roi','feedback_roi']
CHECKPOINTS=[6,12,18,24]
PROMPT='Studio photograph of a single human hand, open palm facing camera, all five fingers spread apart and fully visible, wrist entering from bottom, centered, plain dark gray background, realistic skin, sharp focus'
NEGATIVE='extra hands, extra fingers, missing fingers, fused fingers, deformed, text, watermark, blurry, cropped'
OUT=Path('results')/time.strftime('hands_attention_%Y%m%d_%H%M%S');OUT.mkdir(parents=True,exist_ok=True)
MODEL_ID='stable-diffusion-v1-5/stable-diffusion-v1-5'
pipe=StableDiffusionPipeline.from_pretrained(MODEL_ID,torch_dtype=DTYPE,use_safetensors=True).to(DEVICE)
pipe.scheduler=DDIMScheduler.from_config(pipe.scheduler.config);pipe.scheduler.set_timesteps(STEPS,device=DEVICE)
pipe.set_progress_bar_config(disable=True)
print('GPU:',torch.cuda.get_device_name(0))
print('Versions:',{p:metadata.version(p) for p in ['torch','diffusers','transformers','mediapipe','numpy']})
print('Outputs:',OUT)

In [ ]:
# 3. Download Google's hand landmarker; use it only to locate regions
ASSET=Path('models')/'hand_landmarker.task';ASSET.parent.mkdir(exist_ok=True)
ASSET_URL='https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'
if not ASSET.exists():
    response=requests.get(ASSET_URL,timeout=60);response.raise_for_status();ASSET.write_bytes(response.content)
landmarker=mp.tasks.vision.HandLandmarker.create_from_options(mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(ASSET)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,num_hands=2,
    min_hand_detection_confidence=.5,min_hand_presence_confidence=.5))
def locate_hands(image):
    result=landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=np.ascontiguousarray(np.asarray(image.convert('RGB')))))
    boxes=[];points=[]
    for hand in result.hand_landmarks:
        xy=np.array([[p.x,p.y] for p in hand]);points.append(xy.tolist())
        low=xy.min(0);high=xy.max(0);margin=.04+.15*(high-low)
        boxes.append(np.concatenate([np.maximum(0,low-margin),np.minimum(1,high+margin)]).tolist())
    return {'detected_hands':len(boxes),'boxes':boxes,'landmarks':points}
print('Hand detector ready. It predicts 21 template landmarks per detection, NOT a finger count.')
print('Asset SHA256:',hashlib.sha256(ASSET.read_bytes()).hexdigest())

In [ ]:
# 4. Load Jev credentials privately and a local visual observer
from dotenv import dotenv_values
from transformers import AutoProcessor,Qwen2VLForConditionalGeneration
_env=dotenv_values('man.env')
_keys=[v for k,v in _env.items() if v and any(s in k.upper() for s in ['TYPESAFE','JEV'])]
if not _keys: _keys=[v for k,v in _env.items() if v and 'KEY' in k.upper()]
assert len(_keys)==1,'Expected an unambiguous TypeSafe key in man.env'
_api_key=_keys[0];del _env,_keys
MODES=['full','uniform_kv','deterministic','jev']
CHECKPOINTS=[8,16,24];HORIZON=4
VLM_ID='Qwen/Qwen2-VL-2B-Instruct'
vlm_processor=AutoProcessor.from_pretrained(VLM_ID,min_pixels=128*28*28,max_pixels=256*28*28)
vlm_processor.tokenizer.padding_side='left'
vlm=Qwen2VLForConditionalGeneration.from_pretrained(VLM_ID,torch_dtype=torch.bfloat16,use_safetensors=True,attn_implementation='sdpa').to(DEVICE).eval()
print('TypeSafe key loaded:',bool(_api_key),'(hidden)')
print('Local visual observer ready:',VLM_ID)
print('Both controllers will use identical candidate observations. VLM judgments are fallible.')

In [ ]:
# 5. Supply MediaPipe's missing container graphics libraries, then retry
import subprocess
for command in [['apt-get','update','-qq'],['apt-get','install','-y','-qq','libgles2','libegl1','libgl1']]:
    status=subprocess.run(command,capture_output=True,text=True)
    if status.returncode: raise RuntimeError('System dependency install failed: '+status.stderr[-1000:])
landmarker=mp.tasks.vision.HandLandmarker.create_from_options(mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=str(ASSET)),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,num_hands=2,
    min_hand_detection_confidence=.5,min_hand_presence_confidence=.5))
def locate_hands(image):
    result=landmarker.detect(mp.Image(image_format=mp.ImageFormat.SRGB,data=np.ascontiguousarray(np.asarray(image.convert('RGB')))))
    boxes=[];points=[]
    for hand in result.hand_landmarks:
        xy=np.array([[p.x,p.y] for p in hand]);points.append(xy.tolist())
        low=xy.min(0);high=xy.max(0);margin=.04+.15*(high-low)
        boxes.append(np.concatenate([np.maximum(0,low-margin),np.minimum(1,high+margin)]).tolist())
    return {'detected_hands':len(boxes),'boxes':boxes,'landmarks':points}
print('MediaPipe ready:',locate_hands(Image.new('RGB',(512,512),'gray'))['detected_hands'],'hands on blank image')

<a id="attention"></a>

## 2 · Attention implementation and observations

The original downsampling processor, DDIM identity check, and local visual observer. These are the foundations used by the initial study.

In [ ]:
# 6. Conditional ToDo-style attention: dense queries, sparse spatial keys/values
# Original extension: keep all fine tokens in protected blocks and correct token mass.
POLICY={'action':'full','step':0,'boxes':[],'phase':'test'}
ATTENTION_LOG=[]; INDEX_CACHE={}
ACTIONS=['full','kv2','hand_kv2','hand_kv4']
def kv_indices(n,factor,boxes,device):
    cache_key=(n,factor,tuple(tuple(round(float(v),5) for v in b) for b in boxes),str(device))
    if cache_key in INDEX_CACHE:return INDEX_CACHE[cache_key]
    side=math.isqrt(n);assert side*side==n and side%factor==0
    yy,xx=np.mgrid[:side,:side];protected=np.zeros((side,side),dtype=bool)
    for x0,y0,x1,y1 in boxes:
        protected |= ((xx+.5)/side>=x0)&((xx+.5)/side<=x1)&((yy+.5)/side>=y0)&((yy+.5)/side<=y1)
    blocks=np.arange(n).reshape(side,side).reshape(side//factor,factor,side//factor,factor).transpose(0,2,1,3).reshape(-1,factor*factor)
    preserve=protected.ravel()[blocks].any(1)
    fine=blocks[preserve].ravel();coarse=blocks[~preserve,0]
    idx=np.concatenate([fine,coarse]);mass=np.concatenate([np.ones(len(fine)),np.full(len(coarse),factor*factor)])
    order=np.argsort(idx);idx=idx[order];mass=mass[order]
    answer=(torch.tensor(idx,device=device,dtype=torch.long),torch.tensor(np.log(mass),device=device,dtype=DTYPE))
    INDEX_CACHE[cache_key]=answer;return answer
class ConditionalKVProcessor:
    def __init__(self,name):self.name=name;self.base=AttnProcessor2_0()
    def __call__(self,attn,hidden_states,encoder_hidden_states=None,attention_mask=None,temb=None,*args,**kwargs):
        assert encoder_hidden_states is None,'Only replace spatial self-attention processors'
        n=hidden_states.shape[1];action=POLICY['action'];idx=None;logmass=None
        eligible=hidden_states.ndim==3 and attn.to_q.in_features in (320,640)
        active=8<=POLICY['step']<28 and action!='full' and eligible
        if active:
            assert attn.group_norm is None and attn.spatial_norm is None and not attn.norm_cross
            boxes=POLICY['boxes'] if action.startswith('hand_') else []
            # If a hand-preserving action lacks a location, keep full information.
            if not action.startswith('hand_') or boxes:
                factor=4 if action=='hand_kv4' else 2
                idx,logmass=kv_indices(n,factor,boxes,hidden_states.device)
        nk=n if idx is None else len(idx)
        ATTENTION_LOG.append({'step':POLICY['step'],'phase':POLICY['phase'],'layer':self.name,'action':action,'q_tokens':n,'kv_tokens':nk,'heads':attn.heads})
        if idx is None or nk==n:
            return self.base(attn,hidden_states,attention_mask=attention_mask,temb=temb)
        context=hidden_states.index_select(1,idx)
        bias=logmass[None,None,:].expand(hidden_states.shape[0],1,-1)
        return self.base(attn,hidden_states,encoder_hidden_states=context,attention_mask=bias,temb=temb)
original_processors=dict(pipe.unet.attn_processors)
patched={name:ConditionalKVProcessor(name) if name.endswith('attn1.processor') else processor for name,processor in original_processors.items()}
pipe.unet.set_attn_processor(patched.copy())
print('Patched spatial self-attention layers:',sum(isinstance(p,ConditionalKVProcessor) for p in pipe.unet.attn_processors.values()))
print('Text cross-attention unchanged. Full Q; selectively reduced K/V; log(area) mass correction.')

In [ ]:
# 7. DDIM sampler and a meaningful attention identity check
with torch.inference_mode():
    tokens=pipe.tokenizer([NEGATIVE,PROMPT],padding='max_length',max_length=77,truncation=True,return_tensors='pt').to(DEVICE)
    embeddings=pipe.text_encoder(tokens.input_ids)[0]
@torch.inference_mode()
def initial_latent(seed):
    return torch.randn((1,4,SIZE//8,SIZE//8),generator=torch.Generator(device=DEVICE).manual_seed(seed),device=DEVICE,dtype=DTYPE)
@torch.inference_mode()
def step_latent(z,i,action='full',boxes=None,phase='main'):
    POLICY.update(action=action,step=i,boxes=boxes or [],phase=phase)
    eps=pipe.unet(z.repeat(2,1,1,1),pipe.scheduler.timesteps[i],encoder_hidden_states=embeddings).sample
    uncond,cond=eps.chunk(2);eps=uncond+CFG*(cond-uncond)
    output=pipe.scheduler.step(eps,pipe.scheduler.timesteps[i],z,eta=0.)
    return output.prev_sample,output.pred_original_sample
@torch.inference_mode()
def decode(z):
    x=(pipe.vae.decode(z/pipe.vae.config.scaling_factor).sample/2+.5).clamp(0,1)
    return [Image.fromarray((a*255).round().astype('uint8')) for a in x.float().cpu().permute(0,2,3,1).numpy()]
ztest=initial_latent(17)
za,_=step_latent(ztest,8,'full',phase='validation')
pipe.unet.set_attn_processor(original_processors.copy())
zb,_=step_latent(ztest,8,'full',phase='validation')
pipe.unet.set_attn_processor(patched.copy())
identity_error=float((za-zb).abs().max());assert identity_error==0
ATTENTION_LOG.clear()
_,_=step_latent(ztest,8,'kv2',phase='validation')
display(pd.DataFrame(ATTENTION_LOG).groupby(['q_tokens','kv_tokens']).size().rename('layers'))
print('Full-attention wrapper vs original max error:',identity_error)
print('Dense query counts preserved; keys/values reduced only in eligible spatial layers.')
ATTENTION_LOG.clear()
RESULTS={};BASE_PATHS={}
def run_plain(seed,mode):
    start=time.perf_counter();z=initial_latent(seed);checkpoints={};ATTENTION_LOG.clear()
    for i in range(STEPS):
        if i in CHECKPOINTS:checkpoints[i]=z.clone()
        z,_=step_latent(z,i,'kv2' if mode=='uniform_kv' else 'full')
    image=decode(z)[0];torch.cuda.synchronize();seconds=time.perf_counter()-start
    image.save(OUT/f'{seed}_{mode}.png')
    RESULTS[(seed,mode)]={'image':image,'seconds':seconds,'attention':list(ATTENTION_LOG),'decisions':[]}
    if mode=='full':BASE_PATHS[seed]=checkpoints
    print(f'{mode} seed {seed}: {seconds:.2f}s')
    return image
display(run_plain(17,'full'))

In [ ]:
# 8. Convert images into explicit, uncertain observations for Jev
import re
VISION_QUESTION='Examine the hand anatomy in this image. Return ONLY a compact JSON object with keys: visible_digits (integer total visibly distinct digits on the main hand, counting thumb, or null if not countable), fused_or_extra (true/false/null), complete_hand (true/false/null), uncertain (true/false), description (one short sentence describing visible defects or uncertainty). Do not assume five digits because a hand should have five. If image is blurry or ambiguous, set uncertain true. Do not judge aesthetics.'
VLM_LOG=[]
@torch.inference_mode()
def visual_reports(images):
    messages=[{'role':'user','content':[{'type':'image'},{'type':'text','text':VISION_QUESTION}]}]
    template=vlm_processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=vlm_processor(text=[template]*len(images),images=images,padding=True,return_tensors='pt').to(DEVICE)
    generated=vlm.generate(**inputs,max_new_tokens=120,do_sample=False)
    texts=vlm_processor.batch_decode(generated[:,inputs.input_ids.shape[1]:],skip_special_tokens=True,clean_up_tokenization_spaces=False)
    reports=[]
    for raw in texts:
        try:
            obj=json.loads(re.search(r'\{[\s\S]*\}',raw).group(0))
            digits=obj.get('visible_digits')
            if type(digits) is not int or not 0<=digits<=10:digits=None
            obj={'visible_digits':digits,'fused_or_extra':obj.get('fused_or_extra') if type(obj.get('fused_or_extra')) is bool else None,'complete_hand':obj.get('complete_hand') if type(obj.get('complete_hand')) is bool else None,'uncertain':obj.get('uncertain') is not False,'description':str(obj.get('description','')),'parse_ok':True,'raw':raw}
        except Exception:
            obj={'visible_digits':None,'fused_or_extra':None,'complete_hand':None,'uncertain':True,'description':'Observer output could not be parsed.','parse_ok':False,'raw':raw}
        reports.append(obj)
    VLM_LOG.extend(reports);return reports
baseline_observation=visual_reports([RESULTS[(17,'full')]['image']])[0]
print('VLM observation (a model claim, not ground truth):');display(baseline_observation)
print('Landmarker detections:',locate_hands(RESULTS[(17,'full')]['image'])['detected_hands'])

<a id="initial-study"></a>

## 3 · Initial downsampling study

Historical comparison across full attention, uniform downsampling, deterministic control, and Jev. Seeds 17, 42, and 123 were retained; this section is not a new comparison run.

In [ ]:
# 9. Same-latent counterfactuals: expose consequences, not just action names
vlm.generation_config.temperature=1.;vlm.generation_config.top_p=1.;vlm.generation_config.top_k=50
PROBE_CACHE={}
def score_cost(log):
    denom=sum(x['heads']*x['q_tokens']**2 for x in log)
    return sum(x['heads']*x['q_tokens']*x['kv_tokens'] for x in log)/max(denom,1)
def clean_report(report):return {k:v for k,v in report.items() if k!='raw'}
def probe_attention(z,i,seed,history,tag):
    start=time.perf_counter()
    _,current_x0=step_latent(z,i,'full',phase='observation')
    current=decode(current_x0)[0];localization=locate_hands(current);boxes=localization['boxes']
    branches={};images=[];costs={}
    for action in ACTIONS:
        branch=z.clone();mark=len(ATTENTION_LOG)
        for j in range(i,i+HORIZON):branch,x0=step_latent(branch,j,action,boxes,phase='probe')
        costs[action]=score_cost(ATTENTION_LOG[mark:]);branches[action]=branch
        preview=decode(x0)[0];images.append(preview);preview.save(OUT/f'{tag}_step{i}_{action}.png')
    reports=visual_reports([current]+images)
    ref=np.asarray(images[0],dtype=np.float32)/255
    region=np.zeros((SIZE,SIZE),dtype=bool)
    for x0,y0,x1,y1 in boxes:region[int(y0*SIZE):int(np.ceil(y1*SIZE)),int(x0*SIZE):int(np.ceil(x1*SIZE))]=True
    if not region.any():region[:]=True
    outcomes={}
    for action,image,report in zip(ACTIONS,images,reports[1:]):
        arr=np.asarray(image,dtype=np.float32)/255;det=locate_hands(image)
        outcomes[action]={'vision':clean_report(report),'detected_hands':det['detected_hands'],
            'pixel_mae_vs_full':round(float(np.abs(arr-ref).mean()),6),
            'hand_region_mae_vs_full':round(float(np.abs(arr-ref)[region].mean()),6),
            'attention_score_ratio':round(costs[action],5),
            'effective_policy':'full attention fallback: no hand location' if action.startswith('hand_') and not boxes else action}
    state={'goal':'One complete open human hand with five separate natural digits including the thumb. Preserve anatomy first; save attention computation when evidence suggests it is safe.',
        'seed':seed,'step':i,'progress':round(i/STEPS,3),'lookahead_steps':HORIZON,
        'scene':{'vision':clean_report(reports[0]),'detected_hands':localization['detected_hands'],'hand_boxes':boxes},
        'candidate_outcomes':outcomes,'history':history[-2:],
        'measurement_limits':'Vision-language descriptions and digit counts are fallible claims, not ground truth. Null means unknown. MediaPipe predicts a fixed landmark template and does not validate finger count. Pixel closeness to full attention measures preservation, not correctness. Attention score ratio estimates QK work, not total runtime.'}
    result={'branches':branches,'images':images,'state':state,'boxes':boxes,'seconds':time.perf_counter()-start}
    if i==CHECKPOINTS[0]:PROBE_CACHE[seed]=result
    print(f'Probe seed={seed} step={i}: {result["seconds"]:.2f}s; hand regions={len(boxes)}',flush=True)
    return result
probe17=probe_attention(BASE_PATHS[17][8],8,17,[],'seed17_first_probe')
display(pd.DataFrame({a:{'VLM_digits':o['vision']['visible_digits'],'uncertain':o['vision']['uncertain'],'hand_detections':o['detected_hands'],'attention_ratio':o['attention_score_ratio'],'drift':o['hand_region_mae_vs_full'],'description':o['vision']['description']} for a,o in probe17['state']['candidate_outcomes'].items()}).T)
fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,a,img in zip(axes,ACTIONS,probe17['images']):ax.imshow(img);ax.set_title(a);ax.axis('off')
plt.tight_layout();fig.savefig(OUT/'first_counterfactuals.png',dpi=150);plt.show()

In [ ]:
# 10. Real Jev calls: small anatomy judgments, combined in code
QUESTIONS={}
for action in ACTIONS:
    prefix=f'Use candidate_outcomes.{action}.vision and its description. These are fallible observations of one hand, not ground truth. Null fields mean unknown. '
    QUESTIONS[action+'_five']={'type':'noul','instructions':prefix+'Does the evidence support exactly five distinct visible digits including the thumb? Do not infer five merely because it is a hand. Ambiguous or contradictory evidence should be near 0.5.'}
    QUESTIONS[action+'_separate']={'type':'noul','instructions':prefix+'Does the evidence support separated, naturally formed digits without fusion or extra digit artifacts? Unknown is near 0.5, not yes.'}
    QUESTIONS[action+'_complete']={'type':'noul','instructions':prefix+'Does the evidence support one complete hand rather than a cropped, missing, or duplicated hand? Unknown is near 0.5.'}
API_LOG=[]
def aggregate_candidate(outcome,p):
    anatomy=.55*p['five']+.30*p['separate']+.15*p['complete']
    certainty=.75 if outcome['vision']['uncertain'] else 1.
    return certainty*anatomy-.4*outcome['hand_region_mae_vs_full']+.10*(1-outcome['attention_score_ratio'])
def deterministic_route(state):
    scores={}
    for a,o in state['candidate_outcomes'].items():
        v=o['vision'];digits=v['visible_digits']
        p={'five':.5 if digits is None else float(digits==5),
           'separate':.5 if v['fused_or_extra'] is None else float(not v['fused_or_extra']),
           'complete':.5 if v['complete_hand'] is None else float(v['complete_hand'])}
        scores[a]=aggregate_candidate(o,p)
    return max(scores,key=scores.get),{'scores':scores,'router':'deterministic'}
def jev_route(state):
    start=time.perf_counter()
    for attempt in range(3):
        response=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_api_key},json={'model':'jev-latest','state':state,'questions':QUESTIONS},timeout=60)
        if response.status_code not in (429,529):break
        time.sleep(2**attempt)
    if response.status_code!=200:raise RuntimeError('TypeSafe HTTP '+str(response.status_code)+'; response withheld to protect credentials')
    data=response.json();answers=data['answers'];scores={}
    for a,o in state['candidate_outcomes'].items():
        p={q:float(answers[a+'_'+q]['noul']) for q in ['five','separate','complete']}
        assert all(0<=x<=1 for x in p.values())
        scores[a]=aggregate_candidate(o,p)
    selected=max(scores,key=scores.get)
    audit={'model':data.get('model'),'answers':answers,'scores':scores,'selected':selected,'usage':data.get('usage'),'seconds':time.perf_counter()-start}
    API_LOG.append(audit);return selected,audit
first_jev_action,first_jev_audit=jev_route(probe17['state'])
first_det_action,first_det_audit=deterministic_route(probe17['state'])
print('Jev model:',first_jev_audit['model'],'| selected:',first_jev_action,'| deterministic:',first_det_action)
display(pd.DataFrame({'jev':first_jev_audit['scores'],'deterministic':first_det_audit['scores']}))
print('Jev probabilities:');display(pd.DataFrame({a:{q:first_jev_audit['answers'][a+'_'+q]['noul'] for q in ['five','separate','complete']} for a in ACTIONS}).T)

In [ ]:
# 11. Let Jev control the diffusion trajectory, then observe and replan
# The candidate action runs for four probe steps; its selected latent is committed.
# Continue the same action until the next checkpoint, then measure new futures.
def run_control(seed,mode):
    assert mode in ['deterministic','jev']
    start=time.perf_counter();z=initial_latent(seed);i=0;action='full';boxes=[]
    history=[];decisions=[];trajectory_ratios=[];ATTENTION_LOG.clear()
    while i<STEPS:
        if i in CHECKPOINTS:
            probe=probe_attention(z,i,seed,history,f'{seed}_{mode}')
            state=probe['state']
            chosen,audit=jev_route(state) if mode=='jev' else deterministic_route(state)
            action=chosen;boxes=probe['boxes'];z=probe['branches'][chosen].clone()
            outcome=state['candidate_outcomes'][chosen]
            trajectory_ratios.extend([outcome['attention_score_ratio']]*HORIZON)
            decisions.append({'step':i,'action':chosen,'state':state,'router':audit})
            history.append({'step':i,'action':chosen,'observed_future':outcome})
            print(f'{mode} seed {seed} step {i}: {chosen}',flush=True)
            i+=HORIZON
        else:
            mark=len(ATTENTION_LOG);z,_=step_latent(z,i,action,boxes)
            trajectory_ratios.append(score_cost(ATTENTION_LOG[mark:]));i+=1
    image=decode(z)[0];torch.cuda.synchronize();seconds=time.perf_counter()-start
    assert len(trajectory_ratios)==STEPS
    image.save(OUT/f'{seed}_{mode}.png')
    logs=list(ATTENTION_LOG)
    RESULTS[(seed,mode)]={'image':image,'seconds':seconds,'attention':logs,'decisions':decisions,'trajectory_attention_ratio':float(np.mean(trajectory_ratios))}
    payload={'seed':seed,'mode':mode,'seconds':seconds,'trajectory_attention_ratio':float(np.mean(trajectory_ratios)),'decisions':decisions,'attention':logs}
    serialized=json.dumps(payload,indent=2);assert _api_key not in serialized
    (OUT/f'{seed}_{mode}_audit.json').write_text(serialized)
    print(f'Completed {mode} seed {seed}: {seconds:.2f}s (includes probes, vision and API)',flush=True)
    return image
run_plain(17,'uniform_kv')
for mode in ['deterministic','jev']:run_control(17,mode)
fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,mode in zip(axes,MODES):ax.imshow(RESULTS[(17,mode)]['image']);ax.set_title(mode);ax.axis('off')
plt.tight_layout();fig.savefig(OUT/'seed17_comparison.png',dpi=160);plt.show()

In [ ]:
# 12. Replicate the same Jev policy on the remaining predeclared seeds
for seed in SEEDS[1:]:
    run_plain(seed,'full')
    run_plain(seed,'uniform_kv')
    for mode in ['deterministic','jev']:run_control(seed,mode)
print('All 12 final images generated. No seeds were discarded.')

<a id="initial-evidence"></a>

### Evaluation and intervention evidence

In [ ]:
# 13. Evaluate every final image without pretending model claims are ground truth
rows=[];FINAL_OBSERVATIONS={}
for seed in SEEDS:
    images=[RESULTS[(seed,m)]['image'] for m in MODES]
    reports=visual_reports(images)
    reference=np.asarray(images[0],dtype=np.float32)/255
    for mode,img,v in zip(MODES,images,reports):
        run=RESULTS[(seed,mode)];d=locate_hands(img)
        ratio=run.get('trajectory_attention_ratio',score_cost(run['attention']))
        row={'seed':seed,'mode':mode,'seconds_including_control':run['seconds'],
             'trajectory_self_attention_ratio':ratio,'actual_unet_calls':len(run['attention'])/16,
             'pixel_mae_vs_full':float(np.abs(np.asarray(img,dtype=np.float32)/255-reference).mean()),
             'hand_detections':d['detected_hands'],'vlm_visible_digits':v['visible_digits'],
             'vlm_fused_or_extra':v['fused_or_extra'],'vlm_uncertain':v['uncertain'],'vlm_description':v['description']}
        rows.append(row);FINAL_OBSERVATIONS[f'{seed}_{mode}']={'vision':v,'localization':d}
metrics_df=pd.DataFrame(rows);metrics_df.to_csv(OUT/'metrics.csv',index=False)
display(metrics_df.round(4))
actions_df=pd.DataFrame([{'seed':seed,'mode':mode,'step':d['step'],'action':d['action']} for (seed,mode),run in RESULTS.items() for d in run['decisions']])
actions_df.to_csv(OUT/'actions.csv',index=False)
print('Actual routing decisions:');display(actions_df.pivot(index=['seed','mode'],columns='step',values='action'))
summary=metrics_df.groupby('mode',sort=False)[['seconds_including_control','trajectory_self_attention_ratio','actual_unet_calls','pixel_mae_vs_full']].mean()
print('Descriptive averages; one timing run per seed, not a rigorous speed benchmark:');display(summary.round(4))
summary.to_csv(OUT/'summary.csv')
fig,axes=plt.subplots(len(SEEDS),len(MODES),figsize=(16,12))
for r,seed in enumerate(SEEDS):
    for c,mode in enumerate(MODES):
        axes[r,c].imshow(RESULTS[(seed,mode)]['image']);axes[r,c].set_title(f'{mode} | seed {seed}');axes[r,c].axis('off')
plt.tight_layout();fig.savefig(OUT/'all_seeds_comparison.png',dpi=160,bbox_inches='tight');plt.show()
# Human review fields remain blank until someone actually judges the image.
review=pd.DataFrame([{'seed':s,'mode':m,'image':f'{s}_{m}.png','visible_digits_human':None,'anatomy_correct_human':None,'notes':''} for s in SEEDS for m in MODES])
review.to_csv(OUT/'human_review.csv',index=False)
assert len(RESULTS)==12 and len(API_LOG)==10
assert all(img['image'].size==(512,512) for img in RESULTS.values())
print('Verified: 12 final images, 9 Jev trajectory decisions plus 1 demonstration call.')

In [ ]:
# 14. Show exactly what Jev changed and archive the evidence
boxes=probe17['boxes'];chosen=first_jev_action
factor=4 if chosen=='hand_kv4' else 2
idx,_=kv_indices(64*64,factor,boxes if chosen.startswith('hand_') else [],DEVICE)
retained=np.zeros(4096);retained[idx.cpu().numpy()]=1
fig,axes=plt.subplots(1,3,figsize=(13,4))
axes[0].imshow(probe17['images'][0]);axes[0].set_title('Same-latent full-attention future')
axes[1].imshow(retained.reshape(64,64),cmap='gray',vmin=0,vmax=1);axes[1].set_title('K/V locations retained by Jev action')
axes[2].imshow(probe17['images'][ACTIONS.index(chosen)]);axes[2].set_title('Jev selected: '+chosen)
for ax in axes:ax.axis('off')
plt.tight_layout();fig.savefig(OUT/'jev_attention_intervention.png',dpi=160,bbox_inches='tight');plt.show()
paired=actions_df.pivot(index=['seed','step'],columns='mode',values='action')
different=int((paired.jev!=paired.deterministic).sum())
print('Different action-sequence entries:',different,'of',len(paired))
config={'model':MODEL_ID,'vlm':VLM_ID,'versions':{p:metadata.version(p) for p in ['torch','diffusers','transformers','mediapipe','numpy']},
    'seeds':SEEDS,'size':SIZE,'steps':STEPS,'cfg':CFG,'prompt':PROMPT,'negative':NEGATIVE,
    'checkpoints':CHECKPOINTS,'horizon':HORIZON,'active_attention_window':[8,28],
    'actions':ACTIONS,'attention':'Full queries; nearest spatial K/V subsampling on channels 320/640; dense protected blocks; log(area) attention mass correction. Text cross-attention unchanged.',
    'observer_prompt':VISION_QUESTION,'jev_questions':QUESTIONS,
    'utility':'certainty*(.55*p_five+.30*p_separate+.15*p_complete) - .4*hand_mae + .10*(1-attention_ratio); certainty .75 for uncertain observer, else 1',
    'asset_sha256':hashlib.sha256(ASSET.read_bytes()).hexdigest(),'full_wrapper_identity_error':identity_error,
    'source':'https://github.com/ethansmith2000/ImprovedTokenMerge'}
for name,value in [('config.json',config),('typesafe_responses.json',API_LOG),('visual_observer_responses.json',VLM_LOG),('final_observations.json',FINAL_OBSERVATIONS)]:
    payload=json.dumps(value,indent=2);assert _api_key not in payload;(OUT/name).write_text(payload)
print('Actual TypeSafe calls:',len(API_LOG))
print('API token usage:',{k:sum(x.get('usage',{}).get(k,0) for x in API_LOG) for k in ['input_tokens','output_tokens']})
POLICY.update(action='full',step=0,boxes=[],phase='idle')

In [ ]:
# 15. Research note: what Jev did, what it did not establish
import zipfile
summary_lines=['| Method | Wall seconds | Selected-trajectory self-attention ratio | Actual UNet calls |','|---|---:|---:|---:|']
for mode,row in summary.iterrows():summary_lines.append(f'| {mode} | {row.seconds_including_control:.2f} | {row.trajectory_self_attention_ratio:.3f} | {row.actual_unet_calls:.0f} |')
report='\n'.join(['# Jev controlling SD1.5 attention: hands pilot','',
'Jev is the experimental controller. This notebook made ten real TypeSafe calls: one demonstration and nine decisions during three sampling trajectories. At each checkpoint Jev received measured counterfactual outcomes, local visual-model descriptions, hand locations, phase, goal and short history. Twelve small probability judgments were combined in code to select an actual attention operation and commit its latent branch.','',
'## How the pieces fit','',
'Use both: conditional K/V downsampling inside attention, with Jev selecting that condition from feedback during diffusion. Static downsampling needs no outer controller. The closed loop is what lets policy change as the sample develops. SD1.5 has no internal register tokens; no register steering is claimed here.','',
'Actions: full attention; uniform 2x2 K/V sampling; preserve dense hand regions and sample the rest 2x2 or 4x4. Queries and text cross-attention stay intact. This is a ToDo-inspired implementation with regional protection and token-mass correction, not a reproduction of its published speed benchmark.','',
'## Observed results','']+summary_lines+['',
f'Jev and the deterministic router produced different action-sequence entries at {different} of nine matched seed/checkpoint positions. Once trajectories diverge, later observer states differ too, so this is not nine identical-state policy tests.',
'', 'For seed 17 both controllers protected the hand while downsampling the background at step 8, then restored full attention at steps 16 and 24. Jev made different later choices on seeds 42 and 123. These are real changes to sampling, not changes to the text prompt.',
'', 'Visual inspection: the first seed already appears to have five visible digits. Seed 42 still contains an unwanted second hand under all methods. The pilot does not demonstrate reliable finger correction or a clear quality advantage from Jev. All images and candidate previews are retained.',
'', 'The attention ratio counts head-weighted QK pairs along the committed generation only. It excludes discarded probes, vision inference and networking; it is not overall FLOP reduction or a wall-clock speedup. Controller timings include all those costs. The controller used 75 UNet calls rather than 36, so this exploratory system is slower end to end.',
'', '## Evidence and limitations','',
'Full-attention behavior of the wrapper matched the original UNet exactly in a numerical check. Uniform reduction retained 4096 queries while reducing eligible K/V sets to 1024 tokens. Three fixed seeds, one simple prompt, and one timing trial per condition are a feasibility test, not a statistical benchmark.',
'', 'Qwen2-VL observations are fallible, and null values are retained as unknown. MediaPipe supplies a 21-landmark template, not a verified finger count. Jev receives text descriptions rather than pixels and cannot recover visual defects the observers omit. Both routers use the same available observer fields, action menu and utility weights; the deterministic router uses fixed field-to-score mappings while Jev estimates the probabilities from the structured context and descriptions. No trained anatomy verifier, context ablation, blinded human ratings, or learned register atlas was used.',
'', 'human_review.csv intentionally has blank human labels; automated model counts have not been presented as independently verified correctness.',
'', '## What to try next','',
'Test more difficult hand poses and seeds; add a stronger anatomy observer with calibrated uncertainty; compare fixed hand protection and random routing at equal compute. Give Jev corrective actions such as hand-pose conditioning, local inpainting or rollback/resampling, rather than only changing information density. Explicitly detect extra hands and use earlier interventions when layout is already wrong.',
'', '## Files and reproduction','',
'The notebook contains the executed cells. Results include final PNGs, counterfactual previews, masks, metrics.csv, actions.csv, all TypeSafe probabilities, full per-run state/action logs, observer responses, settings and model versions. The setup needed libgles2, libegl1 and libgl1 in this container (installed in cell 5). Keep man.env beside the notebook. Its contents are never saved in this bundle.',
'', 'Sources: https://github.com/ethansmith2000/ImprovedTokenMerge ; https://arxiv.org/abs/2402.13573 ; https://docs.typesafe.ai/api ; https://huggingface.co/Qwen/Qwen2-VL-2B-Instruct ; https://developers.google.com/edge/mediapipe/solutions/vision/hand_landmarker/python'])
assert _api_key not in report
(OUT/'README.md').write_text(report)
zip_path=OUT.with_suffix('.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUT.iterdir()):
        if path.is_file():bundle.write(path,arcname=OUT.name+'/'+path.name)
display(Markdown(report))
file_base='/files'+str(OUT.resolve())
display(HTML(f'<p><a href="{file_base}/all_seeds_comparison.png" target="_blank">Final hand comparison</a> | <a href="{file_base}/jev_attention_intervention.png" target="_blank">See Jev token selection</a> | <a href="/files{zip_path.resolve()}" download>Download all results and logs</a></p>'))
print('Result bundle:',zip_path,'| MiB:',round(zip_path.stat().st_size/2**20,2))

<a id="initial-images"></a>

### Initial study: complete image grid

In [ ]:
# 16. Final visual result — Jev is the controller in the rightmost column
display(Markdown('### Jev controlling attention inside SD1.5\nRows: seeds **17, 42, 123**. Columns: **full → uniform K/V downsampling → deterministic controller → Jev controller**. Exact actions and returned probabilities are saved above.'))
display(DisplayImage(filename=str(OUT/'all_seeds_comparison.png'),width=660))
display(HTML(f'<a href="{file_base}/all_seeds_comparison.png" target="_blank">Full-resolution comparison</a> | <a href="/files{zip_path.resolve()}" download>Images, measurements and actual Jev responses</a>'))

<a id="continuation"></a>

## 4 · Seed 123: three-round continuation

Jev-only continuation from the original seed 123 image. The prompt, feature-driven retention mechanism, every round, and the complete readout follow.

In [ ]:
# 17. Jev-only continuation of seed 123: visible controller instruction
JEV_CONTINUATION_PROMPT = """Choose the next operation from the available candidates using the current model state and observed transition history. Favor progress toward the requested image that is supported by the supplied evidence. Account for uncertainty and delayed effects. Architectural measurements describe activity, not semantic labels: do not assume a component represents fingers. Preserve the subject's skin tone, pose, sleeve and background while seeking a coherent single hand with naturally connected, distinct digits. Do not reward mere sharpness or change. When evidence does not distinguish candidates, prefer the default transition. The sampling budget is fixed; there is no early-stop action."""
print(JEV_CONTINUATION_PROMPT)
import inspect
print(inspect.getsource(ConditionalKVProcessor.__call__))


In [ ]:
# 18. Architecture as context: feature-driven K/V retention, no anatomical masks
from datetime import datetime
REFINE_OUT = OUT / ('jev123_continuation_' + datetime.now().strftime('%H%M%S'))
REFINE_OUT.mkdir(parents=True, exist_ok=False)
REFINE_ORIGINAL_PROCESSORS = dict(pipe.unet.attn_processors)
REFINE_CONTROL = {'action':'default', 'capture':False}
REFINE_LAYERS = {}
REFINE_ACTIONS = {
 'default': 'Retain every spatial key/value token in every self-attention layer.',
 'novel_up': 'In up-block self-attention, retain a spatial coverage grid plus tokens most different from their local feature neighborhood; keep all query tokens.',
 'novel_down': 'Apply the same feature-based K/V retention in down-block self-attention; keep all query tokens.',
 'novel_both': 'Apply feature-based K/V retention in both down- and up-block self-attention; keep all query tokens.'
}
class FeatureContextProcessor:
    def __init__(self, name, base): self.name,self.base=name,base
    def __call__(self, attn, hidden_states, encoder_hidden_states=None, attention_mask=None, temb=None, *args, **kwargs):
        assert encoder_hidden_states is None
        b,n,c=hidden_states.shape; side=int(n**0.5)
        action=REFINE_CONTROL['action']
        eligible=(side*side==n and n>=256)
        selected=eligible and ((action in ('novel_up','novel_both') and self.name.startswith('up_blocks')) or (action in ('novel_down','novel_both') and self.name.startswith('down_blocks')))
        context=None; retained=n
        if eligible and (selected or REFINE_CONTROL['capture']):
            # Conditional branch's internal features; no pixel boxes or semantic labels.
            features=hidden_states[-1].float().T.reshape(1,c,side,side)
            local=torch.nn.functional.avg_pool2d(features,3,stride=1,padding=1,count_include_pad=False)
            novelty=(features-local).square().mean(1).flatten()
            if selected:
                coverage=torch.arange(n,device=hidden_states.device).reshape(side,side)[::2,::2].flatten()
                distinctive=novelty.topk(max(1,n//4)).indices
                indices=torch.unique(torch.cat([coverage,distinctive]),sorted=True)
                context=hidden_states.index_select(1,indices);retained=len(indices)
            if REFINE_CONTROL['capture']:
                REFINE_LAYERS[self.name]={'tokens':n,'channels':c,'retained_kv':retained,
                  'feature_rms':round(float(features.square().mean().sqrt()),4),
                  'local_difference_mean':round(float(novelty.mean()),4),
                  'local_difference_p90':round(float(torch.quantile(novelty,.9)),4)}
        return self.base(attn,hidden_states,encoder_hidden_states=context,attention_mask=attention_mask,temb=temb)

def install_refinement_processors():
    replacements={}
    for name,processor in REFINE_ORIGINAL_PROCESSORS.items():
        replacements[name]=FeatureContextProcessor(name,getattr(processor,'base',processor)) if name.endswith('attn1.processor') else processor
    pipe.unet.set_attn_processor(replacements)

@torch.inference_mode()
def refinement_step(z,i,action,capture=False):
    REFINE_CONTROL.update(action=action,capture=capture)
    prediction=pipe.unet(z.repeat(2,1,1,1),pipe.scheduler.timesteps[i],encoder_hidden_states=embeddings).sample
    u,c=prediction.chunk(2)
    prediction=u+CFG*(c-u)
    result=pipe.scheduler.step(prediction,pipe.scheduler.timesteps[i],z,eta=0.)
    return result.prev_sample,result.pred_original_sample

@torch.inference_mode()
def image_latent(image):
    pixels=torch.from_numpy(np.asarray(image.convert('RGB')).copy()).permute(2,0,1)[None].to(device=pipe.device,dtype=pipe.unet.dtype)/127.5-1
    return pipe.vae.encode(pixels).latent_dist.mode()*pipe.vae.config.scaling_factor

# Default must match the existing full-attention path before any new Jev decision.
check_z=BASE_PATHS[123][24].clone()
reference,_=step_latent(check_z,24,'full')
install_refinement_processors()
try:
    checked,_=refinement_step(check_z,24,'default',True)
    error=float((checked-reference).abs().max())
    assert error<1e-5, error
    changed,_=refinement_step(check_z,24,'novel_up',True)
    assert torch.isfinite(changed).all()
    print('Default max error:',error,'| feature-retention latent change:',float((changed-checked).abs().mean()))
    display(pd.DataFrame(REFINE_LAYERS).T)
finally:
    pipe.unet.set_attn_processor(dict(REFINE_ORIGINAL_PROCESSORS))
REFINE_START=RESULTS[(123,'jev')]['image'].copy()
REFINE_START.save(REFINE_OUT/'round_00_start.png')
display(Markdown('**Starting image: the existing Jev result, seed 123.**'))
display(REFINE_START)


In [ ]:
# 19. Jev-only loop: three sequential image-to-image refinement rounds
# The branches below are choices inside this controller, not comparison-study runs.
# Jev supplies one holistic preference probability per candidate; no anatomy-weight formula.
REFINE_QUESTIONS={a:{'type':'noul','instructions':JEV_CONTINUATION_PROMPT +
    ' Considering all candidates in state.candidates, is candidate '+a+
    ' the best next continuation? Return your confidence as a probability. Treat observer descriptions as fallible. Architectural activity is not evidence of anatomical correctness by itself. There is no reward for reducing compute. Unknown or indistinguishable outcomes should receive similar probabilities.'}
    for a in REFINE_ACTIONS}
REFINE_HISTORY=[];REFINE_DECISIONS=[];REFINE_IMAGES=[REFINE_START.copy()]
REFINE_ROUND_REPORTS=[]

def choose_refinement(state):
    for attempt in range(3):
        response=requests.post('https://api.typesafe.ai/v1/systemone',
            headers={'Authorization':'Bearer '+_api_key},
            json={'model':'jev-latest','state':state,'questions':REFINE_QUESTIONS},timeout=60)
        if response.status_code not in (429,529):break
        time.sleep(2**attempt)
    if response.status_code!=200:raise RuntimeError('TypeSafe HTTP '+str(response.status_code)+'; response withheld')
    data=response.json()
    scores={a:float(data['answers'][a]['noul']) for a in REFINE_ACTIONS}
    assert all(np.isfinite(v) and 0<=v<=1 for v in scores.values())
    chosen=max(scores,key=scores.get)
    # Explicit default rule, not an early stop: ordinary denoising still advances.
    fallback=scores[chosen]-scores['default']<=.03
    if fallback:chosen='default'
    return chosen,{'model':data.get('model'),'answers':data['answers'],'scores':scores,
                   'selected':chosen,'default_within_0.03':fallback,'usage':data.get('usage')}

@torch.inference_mode()
def continue_seed123():
    current=REFINE_START.copy()
    install_refinement_processors()
    try:
        for round_no,strength in enumerate([.42,.34,.26],1):
            pipe.scheduler.set_timesteps(STEPS,device=pipe.device)
            start=STEPS-int(STEPS*strength)
            z0=image_latent(current)
            generator=torch.Generator(device=pipe.device).manual_seed(123000+round_no)
            noise=torch.randn(z0.shape,generator=generator,device=z0.device,dtype=z0.dtype)
            z=pipe.scheduler.add_noise(z0,noise,pipe.scheduler.timesteps[start:start+1])
            interval=int(np.ceil((STEPS-start)/3));i=start
            while i<STEPS:
                end=min(i+interval,STEPS)
                REFINE_LAYERS.clear()
                _,current_clean=refinement_step(z,i,'default',True)
                current_arch=dict(REFINE_LAYERS)
                current_preview=decode(current_clean)[0]
                branches={};previews=[];measurements={}
                for action in REFINE_ACTIONS:
                    branch=z.clone();REFINE_LAYERS.clear()
                    for j in range(i,end):
                        branch,clean=refinement_step(branch,j,action,capture=(j==end-1))
                    assert torch.isfinite(branch).all()
                    branches[action]=branch
                    preview=decode(clean)[0];previews.append(preview)
                    preview.save(REFINE_OUT/f'round_{round_no:02d}_step_{i:02d}_{action}.png')
                    measurements[action]={'operation':REFINE_ACTIONS[action],
                        'measured_layers':dict(REFINE_LAYERS),
                        'pixel_change_from_round_input':round(float(np.abs(np.asarray(preview,dtype=float)-np.asarray(current,dtype=float)).mean()/255),5)}
                descriptions=visual_reports([current_preview]+previews)
                for action,description in zip(REFINE_ACTIONS,descriptions[1:]):
                    measurements[action]['visual_observation']=description
                state={'controller_instruction':JEV_CONTINUATION_PROMPT,
                    'goal':'Refine the seed-123 hand: natural distinct connected digits, preserve the existing subject and composition.',
                    'round':round_no,'total_rounds':3,'diffusion_step':i,'total_steps':STEPS,
                    'candidate_interval_steps':end-i,'remaining_steps_this_round':STEPS-i,
                    'architecture':{'model':'SD1.5 UNet','query_tokens':'all retained','text_cross_attention':'unchanged',
                        'selection_rule':'Union of 2x2 spatial coverage grid and top 25 percent of tokens by squared feature deviation from their 3x3 local mean. Selection recomputed per layer and step. No finger boxes or semantic segmentation.',
                        'current_layers':current_arch},
                    'current_visual_observation':descriptions[0],'candidates':measurements,
                    'recent_history':REFINE_HISTORY[-3:],
                    'limits':'Descriptions and digit counts are unverified local VLM outputs. Raw feature novelty is not semantic importance. Pixel change is not improvement. No trained semantic component atlas. Default is selected within 0.03 of the highest score; it still advances diffusion.'}
                chosen,audit=choose_refinement(state)
                z=branches[chosen].clone()
                record={'round':round_no,'step':i,'end_step':end,'state':state,'jev':audit}
                REFINE_DECISIONS.append(record)
                REFINE_HISTORY.append({'round':round_no,'step':i,'chosen':chosen,
                    'predicted_clean_observation':measurements[chosen]['visual_observation']})
                payload=json.dumps(REFINE_DECISIONS,indent=2)
                assert _api_key not in payload
                (REFINE_OUT/'decisions.json').write_text(payload)
                print(f'Round {round_no}, steps {i}:{end}: Jev selected {chosen}; probabilities {audit["scores"]}',flush=True)
                i=end
            current=decode(z)[0]
            current.save(REFINE_OUT/f'round_{round_no:02d}_selected.png')
            REFINE_IMAGES.append(current.copy())
            report=visual_reports([current])[0];REFINE_ROUND_REPORTS.append(report)
            REFINE_HISTORY.append({'round':round_no,'completed_round_observation':report})
            display(Markdown(f'**Jev refinement round {round_no}/3** — image-to-image strength {strength}; same evolving image.'))
            display(current)
            print('Unverified observer:',report['description'],flush=True)
        return current
    finally:
        pipe.unet.set_attn_processor(dict(REFINE_ORIGINAL_PROCESSORS))
        POLICY.update(action='full',boxes=[])

refine_started=time.perf_counter()
REFINE_FINAL=continue_seed123()
print('Completed Jev-only refinement in',round(time.perf_counter()-refine_started,2),'seconds')


<a id="three-round-progress"></a>

### Three-round progression and readout

In [ ]:
# 20. Complete chronological record — no best-round selection or discarded attempts
assert len(REFINE_IMAGES)==4 and len(REFINE_DECISIONS)==9, 'Loop did not finish; inspect the preceding cell.'
from PIL import ImageDraw
# Every completed round, in order, including the unmodified starting image.
progression=Image.new('RGB',(4*384,414),'white')
for k,image in enumerate(REFINE_IMAGES):
    progression.paste(image.resize((384,384)),(384*k,30))
    ImageDraw.Draw(progression).text((384*k+12,9),'Original seed 123' if k==0 else f'Round {k} — chronological, not ranked',fill='black')
progression.save(REFINE_OUT/'all_rounds_progression.png')
display(Markdown('### Entire Jev progression: original → round 1 → round 2 → round 3\nEvery completed round is retained. The final image is simply the last round, not a selected best attempt.'))
display(DisplayImage(filename=str(REFINE_OUT/'all_rounds_progression.png'),width=960))
# Also expose all nine committed predicted-clean checkpoint previews.
checkpoints=Image.new('RGB',(3*320,3*352),'white')
for index,decision in enumerate(REFINE_DECISIONS):
    r=decision['round'];i=decision['step'];a=decision['jev']['selected']
    preview=Image.open(REFINE_OUT/f'round_{r:02d}_step_{i:02d}_{a}.png').convert('RGB')
    x=(index%3)*320;y=(index//3)*352
    checkpoints.paste(preview.resize((320,320)),(x,y+32))
    ImageDraw.Draw(checkpoints).text((x+8,y+9),f'Round {r}, steps {i}:{decision["end_step"]}, {a}',fill='black')
checkpoints.save(REFINE_OUT/'all_selected_checkpoints.png')
display(Markdown('### Every selected checkpoint, in execution order\nThese are predicted-clean previews during sampling, not separate finished images.'))
display(DisplayImage(filename=str(REFINE_OUT/'all_selected_checkpoints.png'),width=780))
manifest={'source':'Existing RESULTS[(123, jev)] image; no regenerated starting image',
    'round_strengths':[.42,.34,.26],'noise_seeds':[123001,123002,123003],
    'steps':STEPS,'cfg':CFG,'controller_prompt':JEV_CONTINUATION_PROMPT,
    'questions':REFINE_QUESTIONS,'actions':REFINE_ACTIONS,'round_observations':REFINE_ROUND_REPORTS,
    'selection':'Highest Jev probability, default within 0.03; all rounds committed, no best-round filter',
    'limitations':'Whole-image img2img can change pose and identity. No anatomy boxes. Feature novelty is not semantic understanding. Jev observes text reports, not image pixels. No claim of verified anatomical improvement.'}
serialized=json.dumps(manifest,indent=2);assert _api_key not in serialized
(REFINE_OUT/'manifest.json').write_text(serialized)
REFINE_FINAL.save(REFINE_OUT/'last_round.png')
refinement_zip=REFINE_OUT.with_suffix('.zip')
with zipfile.ZipFile(refinement_zip,'w',zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(REFINE_OUT.iterdir()):
        if path.is_file():bundle.write(path,arcname=REFINE_OUT.name+'/'+path.name)
refinement_base='/files'+str(REFINE_OUT.resolve())
display(HTML(f'<a href="{refinement_base}/all_rounds_progression.png" target="_blank">All rounds, full resolution</a> | <a href="{refinement_base}/all_selected_checkpoints.png" target="_blank">All nine selected checkpoints</a> | <a href="/files{refinement_zip.resolve()}" download>All candidate images and exact Jev inputs/outputs</a>'))
print('Saved to:',REFINE_OUT)



In [ ]:
# 21. Readout of this single run; no reruns or best-attempt filter
selected_actions=[d['jev']['selected'] for d in REFINE_DECISIONS]
print('Jev choices, chronologically:',selected_actions)
display(Markdown('**Run completed: 3/3 rounds, 9/9 Jev decisions.** All nine selected full attention. The token-retention alternatives were evaluated but never committed. Therefore this progression shows repeated image-to-image denoising under Jev selection, not evidence of a successful attention intervention. Visual inspection shows warmer/redder skin and changed detail; reliable finger repair is not established. All rounds and all 36 candidate previews are retained.'))
display(DisplayImage(filename=str(REFINE_OUT/'all_rounds_progression.png'),width=700))
display(HTML(f'<a href="{refinement_base}/all_rounds_progression.png" target="_blank">Open complete progression</a> | <a href="/files{refinement_zip.resolve()}" download>Download every candidate and Jev decision</a>'))


<a id="vector50"></a>

## 5 · Seed 123: 50 attention decisions

A fixed 50-call trajectory. Jev judgments drive eight joint bias updates across two 16×16 self-attention modules and four selected heads. This run changes logits before softmax; it does not implement a general architecture-editing agent.

In [ ]:
# 22. Jev edits a vector of attention relationships: fixed 50-decision protocol
VECTOR_BUDGET = 50  # Enforced by Python, not delegated to the prompt.
VECTOR_CONTROLLER = """You are assigning simultaneous edits to the named attention relationships in this state. Each address identifies an actual layer, head, source-token group and destination-token group. Inspect their measured attention mass, spatial footprints, current logit biases, and the downstream evidence from joint perturbations. Infer useful changes from measured consequences, not from layer names or attention strength alone. The task is a physically coherent hand consistent with the source image. Distinct fingertips alone do not establish a coherent hand: connections to the palm, articulation, proportions and pose must agree. Consider the other relationships when judging each edit because row normalization couples their effects. Observations are fallible; do not invent anatomical meanings for token groups. High uncertainty should produce similar support for opposing edits."""
VECTOR_OBSERVER = """Describe this generated hand critically in at most 110 words. Trace the visible digit structures into the palm; discuss connections, separation, joints, proportions and pose consistency. Mention duplicated, fused, truncated or ambiguous structures if visible. Do not infer correctness from a count of five tips. Describe skin appearance, sleeve and background changes only when visible. Distinguish visible evidence from uncertainty. Use image-relative positions when describing defects. Do not assume the image meets its prompt."""
print('CONTROLLER:', VECTOR_CONTROLLER)
print('VISUAL OBSERVER:', VECTOR_OBSERVER)
print('Fixed protocol: same original seed-123 Jev image; one 100-step DDIM schedule entered at step 50; one joint edit decision per remaining step; 50 decisions, no best-frame selection.')
import inspect
print('Existing attention names:', [n for n in pipe.unet.attn_processors if n.endswith('attn1.processor')])
print(inspect.getsource(next(p.base for p in pipe.unet.attn_processors.values() if hasattr(p,'base')).__call__))
print(inspect.getsource(visual_reports))


In [ ]:
# 23. Real attention-logit access: eight coupled edits across two layers and four heads
VECTOR_OUT=OUT/('jev123_vector50_'+datetime.now().strftime('%H%M%S'))
VECTOR_OUT.mkdir(parents=True,exist_ok=False)
VECTOR_SAVED_PROCESSORS=dict(pipe.unet.attn_processors)
VECTOR_SAVED_SCHEDULER=pipe.scheduler
from diffusers import DDIMScheduler
VECTOR_SCHEDULER=DDIMScheduler.from_config(pipe.scheduler.config)
VECTOR_SCHEDULER.set_timesteps(100,device=pipe.device)
VECTOR_TARGETS=['down_blocks.2.attentions.1.transformer_blocks.0.attn1.processor',
                'up_blocks.1.attentions.1.transformer_blocks.0.attn1.processor']
VECTOR_GROUPS={};VECTOR_GRAPH={};VECTOR_SLOTS=[];VECTOR_WEIGHTS=np.zeros(8,dtype=float)
VECTOR_CAPTURE=True;VECTOR_INITIALIZE=True

@torch.inference_mode()
def cluster_tokens(features,k=6):
    x=torch.nn.functional.normalize(features.float(),dim=-1)
    centers=[x[x.square().sum(-1).argmax()]]
    for _ in range(k-1):
        similarity=x@torch.stack(centers).T
        centers.append(x[similarity.max(-1).values.argmin()])
    centers=torch.stack(centers)
    for _ in range(15):
        labels=(x@centers.T).argmax(-1)
        centers=torch.stack([torch.nn.functional.normalize(x[labels==g].mean(0),dim=0) if (labels==g).any() else centers[g] for g in range(k)])
    assert len(labels.unique())==k,'Empty feature group; stop before experiment.'
    return labels

class JointLogitProcessor:
    def __init__(self,name):self.name=name
    def __call__(self,attn,hidden_states,encoder_hidden_states=None,attention_mask=None,temb=None,*args,**kwargs):
        assert encoder_hidden_states is None and hidden_states.ndim==3 and attention_mask is None
        assert attn.group_norm is None and attn.spatial_norm is None
        residual=hidden_states;b,n,c=hidden_states.shape;heads=attn.heads
        q=attn.to_q(hidden_states);k=attn.to_k(hidden_states);v=attn.to_v(hidden_states)
        d=k.shape[-1]//heads
        q=q.view(b,n,heads,d).transpose(1,2);k=k.view(b,n,heads,d).transpose(1,2);v=v.view(b,n,heads,d).transpose(1,2)
        if attn.norm_q is not None:q=attn.norm_q(q)
        if attn.norm_k is not None:k=attn.norm_k(k)
        if VECTOR_INITIALIZE and self.name not in VECTOR_GROUPS:
            VECTOR_GROUPS[self.name]=cluster_tokens(hidden_states[-1])
        labels=VECTOR_GROUPS.get(self.name)
        bias=None
        for index,slot in enumerate(VECTOR_SLOTS):
            if slot['layer']!=self.name or abs(VECTOR_WEIGHTS[index])<1e-8:continue
            if bias is None:bias=torch.zeros((b,heads,n,n),device=q.device,dtype=q.dtype)
            mask=(labels[:,None]==slot['source_group']) & (labels[None,:]==slot['destination_group'])
            # Edit the conditional branch only; unconditional and cross-attention remain unchanged.
            bias[-1,slot['head']]+=mask.to(q.dtype)*float(VECTOR_WEIGHTS[index])
        out=torch.nn.functional.scaled_dot_product_attention(q,k,v,attn_mask=bias,dropout_p=0.,is_causal=False)
        if VECTOR_CAPTURE and labels is not None:
            graph={}
            for h in (0,1):
                logits=(q[-1,h].float()@k[-1,h].float().T)/(d**.5)
                if bias is not None:logits=logits+bias[-1,h].float()
                p=logits.softmax(-1)
                matrix=torch.stack([torch.stack([p[labels==a][:,labels==g].sum(-1).mean() for g in range(6)]) for a in range(6)])
                graph[str(h)]={'group_attention_mass':matrix.cpu().numpy().round(5).tolist(),
                    'mean_row_entropy':round(float(-(p*p.clamp_min(1e-12).log()).sum(-1).mean()),5),
                    'output_rms':round(float(out[-1,h].float().square().mean().sqrt()),5)}
            VECTOR_GRAPH[self.name]=graph
        out=out.transpose(1,2).reshape(b,n,heads*d).to(q.dtype)
        out=attn.to_out[1](attn.to_out[0](out))
        if attn.residual_connection:out=out+residual
        return out/attn.rescale_output_factor

def install_vector_processors():
    replacements={n:(JointLogitProcessor(n) if n in VECTOR_TARGETS else getattr(p,'base',p)) for n,p in VECTOR_SAVED_PROCESSORS.items()}
    pipe.unet.set_attn_processor(replacements)

@torch.inference_mode()
def vector_prediction(z,i,weights,capture=True):
    global VECTOR_WEIGHTS,VECTOR_CAPTURE
    VECTOR_WEIGHTS=np.asarray(weights,dtype=float);VECTOR_CAPTURE=capture
    pred=pipe.unet(z.repeat(2,1,1,1),VECTOR_SCHEDULER.timesteps[i],encoder_hidden_states=embeddings).sample
    u,c=pred.chunk(2)
    return u+CFG*(c-u)

@torch.inference_mode()
def vector_step(z,i,weights,capture=True):
    eps=vector_prediction(z,i,weights,capture)
    result=VECTOR_SCHEDULER.step(eps,VECTOR_SCHEDULER.timesteps[i],z,eta=0.)
    return result.prev_sample,result.pred_original_sample,eps

VECTOR_SOURCE=RESULTS[(123,'jev')]['image'].copy()
VECTOR_SOURCE.save(VECTOR_OUT/'000_original.png')
VECTOR_Z0=image_latent(VECTOR_SOURCE)
g=torch.Generator(device=pipe.device).manual_seed(1235050)
VECTOR_NOISE=torch.randn(VECTOR_Z0.shape,device=VECTOR_Z0.device,dtype=VECTOR_Z0.dtype,generator=g)
VECTOR_INITIAL=VECTOR_SCHEDULER.add_noise(VECTOR_Z0,VECTOR_NOISE,VECTOR_SCHEDULER.timesteps[50:51])
torch.save({'encoded_original':VECTOR_Z0.cpu(),'initial_noisy_latent':VECTOR_INITIAL.cpu(),'noise':VECTOR_NOISE.cpu()},VECTOR_OUT/'immutable_start.pt')
# Compare the zero-edit processor to the original full-attention computation.
POLICY.update(action='full')
with torch.inference_mode():
    raw=pipe.unet(VECTOR_INITIAL.repeat(2,1,1,1),VECTOR_SCHEDULER.timesteps[50],encoder_hidden_states=embeddings).sample
    u,c=raw.chunk(2);original_prediction=u+CFG*(c-u)
install_vector_processors()
try:
    zero_prediction=vector_prediction(VECTOR_INITIAL,50,np.zeros(8))
    vector_identity_error=float((zero_prediction-original_prediction).abs().max())
    assert vector_identity_error<1e-4,vector_identity_error
    # Fixed heads 0 and 1; pick two competing destination groups for a source with strong group preference.
    for layer in VECTOR_TARGETS:
        for h in (0,1):
            matrix=np.asarray(VECTOR_GRAPH[layer][str(h)]['group_attention_mass'])
            source=int(matrix.var(axis=1).argmax())
            destinations=np.argsort(matrix[source])[::-1]
            destinations=[int(x) for x in destinations if x!=source][:2]
            for destination in destinations:
                VECTOR_SLOTS.append({'id':f'E{len(VECTOR_SLOTS)}','layer':layer,'head':h,
                    'source_group':source,'destination_group':destination})
    assert len(VECTOR_SLOTS)==8
    probe_prediction=vector_prediction(VECTOR_INITIAL,50,np.array([.5,-.5]*4))
    intervention_effect=float((probe_prediction-zero_prediction).abs().mean())
    assert intervention_effect>0 and torch.isfinite(probe_prediction).all()
finally:
    pipe.unet.set_attn_processor(dict(VECTOR_SAVED_PROCESSORS))
VECTOR_INITIALIZE=False
print('Zero-edit identity error:',vector_identity_error,'| joint-edit prediction change:',intervention_effect)
display(pd.DataFrame(VECTOR_SLOTS))
print('Token groups are fixed feature clusters from the initial noisy state, not finger boxes. Heads and address-selection rule are fixed before the run.')


<a id="grounding"></a>

### Spatial grounding and exact questions

In [ ]:
# 24. Spatial grounding, measured graph context, and exact per-coordinate questions
@torch.inference_mode()
def vector_observe(images,question=VECTOR_OBSERVER,max_tokens=180):
    messages=[{'role':'user','content':[{'type':'image'},{'type':'text','text':question}]}]
    template=vlm_processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs=vlm_processor(text=[template]*len(images),images=images,padding=True,return_tensors='pt').to(DEVICE)
    generated=vlm.generate(**inputs,max_new_tokens=max_tokens,do_sample=False)
    return vlm_processor.batch_decode(generated[:,inputs.input_ids.shape[1]:],skip_special_tokens=True,clean_up_tokenization_spaces=False)

VECTOR_FOOTPRINTS={}
fig,axes=plt.subplots(2,6,figsize=(15,5))
for row,layer in enumerate(VECTOR_TARGETS):
    labels=VECTOR_GROUPS[layer].cpu().numpy().reshape(16,16)
    group_images=[]
    for group in range(6):
        mask=labels==group
        enlarged=np.asarray(Image.fromarray(mask.astype('uint8')*255).resize(VECTOR_SOURCE.size,Image.Resampling.NEAREST))>0
        source=np.asarray(VECTOR_SOURCE,dtype=float)
        dimmed=source*.18;dimmed[enlarged]=source[enlarged]
        grounded=Image.fromarray(dimmed.astype('uint8'));group_images.append(grounded)
        grounded.save(VECTOR_OUT/f'footprint_layer{row}_group{group}.png')
        ys,xs=np.where(mask)
        occupancy=mask.reshape(4,4,4,4).mean(axis=(1,3)).round(3).tolist()
        VECTOR_FOOTPRINTS[layer+'|'+str(group)]={'tokens':int(mask.sum()),
            'centroid_xy':[round(float(xs.mean()/15),3),round(float(ys.mean()/15),3)],
            'occupancy_4x4_rows_top_to_bottom':occupancy}
        axes[row,group].imshow(grounded);axes[row,group].set_title(f'Layer {row}, group {group}');axes[row,group].axis('off')
    captions=vector_observe(group_images,'The image is dimmed except for a set of model token locations. In at most 40 words describe what is visible in the brighter locations and where they are. Do not infer a semantic purpose for the tokens. If the bright region is fragmented or ambiguous, say so.',90)
    for group,caption in enumerate(captions):
        VECTOR_FOOTPRINTS[layer+'|'+str(group)]['unverified_source_footprint_description']=caption
plt.tight_layout();fig.savefig(VECTOR_OUT/'token_group_atlas.png',dpi=150);plt.show()
VECTOR_SOURCE_OBSERVATION=vector_observe([VECTOR_SOURCE])[0]
print('Unverified source observation:',VECTOR_SOURCE_OBSERVATION)
VECTOR_QUESTIONS={}
for slot in VECTOR_SLOTS:
    for direction in ['strengthen','weaken']:
        VECTOR_QUESTIONS[slot['id']+'_'+direction]={'type':'noul','instructions':
            VECTOR_CONTROLLER+' For relationship '+slot['id']+', does the supplied evidence support '+direction+
            'ing its logit bias by 0.20 as one part of the simultaneous edit vector? Use the joint-probe deltas and outcomes, the actual group connections and the accumulated history. Joint probes do not isolate individual causal effects. Evaluate structural plausibility, not merely visible digit count. Opposing options may both be uncertain; do not force an edit.'}
print('Exact example question:',VECTOR_QUESTIONS['E0_strengthen']['instructions'])
print('Mapping: strengthen support minus weaken support > 0.10 => +0.20; < -0.10 => -0.20; otherwise hold. Each accumulated bias is bounded to [-1.5,1.5]. All eight edits applied together.')
# Deterministic orthogonal joint probes rotate across eight directions, not hand-designed anatomy edits.
VECTOR_DIRECTIONS=np.array([[1 if (r&c).bit_count()%2==0 else -1 for c in range(8)] for r in range(8)],dtype=float)
vector_manifest={'budget':50,'schedule_steps':100,'start_index':50,'noise_seed':1235050,
    'source':'Original RESULTS[(123, jev)] image, not the prior refinement result',
    'controller':VECTOR_CONTROLLER,'observer':VECTOR_OBSERVER,'questions':VECTOR_QUESTIONS,
    'slots':VECTOR_SLOTS,'footprints':VECTOR_FOOTPRINTS,'probe_directions':VECTOR_DIRECTIONS.tolist(),
    'probe_horizon':3,'probe_bias_amplitude':.35,'edit_step':.20,'preference_deadband':.10,'bias_bound':1.5,
    'group_rule':'Six fixed cosine-feature clusters at each target layer, initialized from first noisy state; heads 0 and 1; source with maximum group-mass variance; top two off-diagonal destinations.',
    'limitations':'Only eight chosen relationships are editable. Spatial footprint captions are fallible. Multi-coordinate judgments are simultaneous but not a jointly optimized solution. No learned anatomy prior or ground-truth verifier.'}
blob=json.dumps(vector_manifest,indent=2);assert _api_key not in blob
(VECTOR_OUT/'manifest.json').write_text(blob)
np.savez(VECTOR_OUT/'group_labels.npz',**{f'layer{k}':VECTOR_GROUPS[n].cpu().numpy() for k,n in enumerate(VECTOR_TARGETS)})


<a id="live-loop"></a>

### Executed 50-decision loop

In [ ]:
# 25. Fifty Jev calls; eight simultaneous attention edits per call; all states retained
import copy
VECTOR_RECORDS=[];VECTOR_FRAMES=[VECTOR_SOURCE.copy()];VECTOR_MEMORY=[]
VECTOR_STATS={'decisions':0,'coordinate_updates':[0]*8,'positive_updates':[0]*8,'negative_updates':[0]*8,
              'mean_abs_joint_noise_prediction_change':0.}

def vector_api(state):
    for attempt in range(5):
        response=requests.post('https://api.typesafe.ai/v1/systemone',
            headers={'Authorization':'Bearer '+_api_key},
            json={'model':'jev-latest','state':state,'questions':VECTOR_QUESTIONS},timeout=90)
        if response.status_code not in (429,529,502,503,504):break
        time.sleep(min(2**attempt,8))
    if response.status_code!=200:raise RuntimeError('TypeSafe HTTP '+str(response.status_code)+'; response body withheld')
    data=response.json();support={};delta=[]
    for slot in VECTOR_SLOTS:
        key=slot['id'];up=float(data['answers'][key+'_strengthen']['noul']);down=float(data['answers'][key+'_weaken']['noul'])
        assert np.isfinite(up) and np.isfinite(down) and 0<=up<=1 and 0<=down<=1
        support[key]={'strengthen':up,'weaken':down}
        delta.append(.2 if up-down>.10 else (-.2 if down-up>.10 else 0.))
    return np.asarray(delta),{'model':data.get('model'),'support':support,'answers':data['answers'],'usage':data.get('usage')}

@torch.inference_mode()
def vector_trial(z,index,weights,horizon=3):
    branch=z.clone();graph=None;first_eps=None
    for j in range(index,min(index+horizon,100)):
        branch,clean,eps=vector_step(branch,j,weights,capture=(j==index))
        if j==index:graph=copy.deepcopy(VECTOR_GRAPH);first_eps=eps.clone()
    return decode(clean)[0],graph,first_eps

def frame_strip(frames,labels,width=144):
    sheet=Image.new('RGB',(len(frames)*width,width+25),'white')
    for k,(frame,label) in enumerate(zip(frames,labels)):
        sheet.paste(frame.resize((width,width)),(k*width,25))
        ImageDraw.Draw(sheet).text((k*width+5,6),label,fill='black')
    return sheet

@torch.inference_mode()
def run_vector50():
    global VECTOR_STATS
    z=VECTOR_INITIAL.clone();weights=np.zeros(8);previous_observation=VECTOR_SOURCE_OBSERVATION
    install_vector_processors()
    live=display(VECTOR_SOURCE.resize((384,384)),display_id=True)
    try:
        for decision,index in enumerate(range(50,100),1):
            direction=VECTOR_DIRECTIONS[(decision-1)%8]
            candidates={'current':weights.copy(),'probe_plus':np.clip(weights+.35*direction,-1.5,1.5),
                        'probe_minus':np.clip(weights-.35*direction,-1.5,1.5)}
            trials={};images=[]
            for name,candidate in candidates.items():
                preview,graph,eps=vector_trial(z,index,candidate)
                preview.save(VECTOR_OUT/f'{decision:03d}_{name}_lookahead.png')
                trials[name]={'weights':candidate.tolist(),'graph':graph,'eps':eps};images.append(preview)
            observations=vector_observe(images)
            baseline_eps=trials['current']['eps']
            probe_evidence={}
            for (name,trial),observation in zip(trials.items(),observations):
                probe_evidence[name]={'bias_vector':trial['weights'],
                    'delta_from_current':(np.asarray(trial['weights'])-weights).round(4).tolist(),
                    'predicted_clean_observation_after_steps':min(3,100-index),
                    'unverified_visual_observation':observation,
                    'mean_abs_noise_prediction_change':round(float((trial['eps']-baseline_eps).abs().mean()),7)}
            addresses=[]
            for k,slot in enumerate(VECTOR_SLOTS):
                layer=slot['layer'];h=str(slot['head']);a=slot['source_group'];b=slot['destination_group']
                addresses.append({**slot,'current_logit_bias':round(float(weights[k]),3),
                    'source_footprint':VECTOR_FOOTPRINTS[layer+'|'+str(a)],
                    'destination_footprint':VECTOR_FOOTPRINTS[layer+'|'+str(b)],
                    'measured_attention_mass':{name:trial['graph'][layer][h]['group_attention_mass'][a][b] for name,trial in trials.items()}})
            state={'task':'Develop a physically coherent hand from this same source; preserve pose, skin appearance, sleeve and background where compatible with correcting structure.',
                'source_observation_unverified':VECTOR_SOURCE_OBSERVATION,
                'current_observation_unverified':previous_observation,
                'diffusion_state':{'timestep':int(VECTOR_SCHEDULER.timesteps[index]),'noise_to_signal_ratio':round(float(((1-VECTOR_SCHEDULER.alphas_cumprod[VECTOR_SCHEDULER.timesteps[index]])/VECTOR_SCHEDULER.alphas_cumprod[VECTOR_SCHEDULER.timesteps[index]]).sqrt()),4)},
                'editable_relationships':addresses,'current_attention_graphs':trials['current']['graph'],
                'joint_probe_evidence':probe_evidence,'recent_committed_outcomes':VECTOR_MEMORY[-3:],
                'accumulated_measured_history':copy.deepcopy(VECTOR_STATS),
                'semantics':'Rows of group matrices are source groups; columns are total probability mass assigned to destination groups. Group labels and spatial token membership are fixed. Strengthen/weaken changes logit bias by 0.20. All chosen coordinate edits are applied simultaneously. Context has no manually drawn hand or finger boxes.',
                'uncertainty':'The local visual model can miss defects and hallucinate anatomy. Its reports are observations, not verified labels. Correlated probes measure joint effects, not individual causal attribution. Larger internal changes are not necessarily improvements. No stop decision is requested.'}
            proposed,audit=vector_api(state)
            next_weights=np.clip(weights+proposed,-1.5,1.5)
            applied=next_weights-weights
            next_z,clean,actual_eps=vector_step(z,index,next_weights,True)
            assert torch.isfinite(next_z).all()
            joint_effect=float((actual_eps-baseline_eps).abs().mean())
            committed=decode(clean)[0]
            committed.save(VECTOR_OUT/f'{decision:03d}_committed_preview.png')
            observation=vector_observe([committed])[0]
            record={'decision':decision,'schedule_index':index,'state':state,'jev':audit,
                'previous_weights':weights.tolist(),'proposed_delta':proposed.tolist(),'applied_delta':applied.round(5).tolist(),
                'committed_weights':next_weights.round(5).tolist(),'joint_noise_prediction_change':joint_effect,
                'committed_graph':copy.deepcopy(VECTOR_GRAPH),'committed_observation_unverified':observation}
            VECTOR_RECORDS.append(record);VECTOR_FRAMES.append(committed.copy())
            VECTOR_STATS['decisions']=decision
            for k,d in enumerate(applied):
                VECTOR_STATS['coordinate_updates'][k]+=int(abs(d)>1e-6)
                VECTOR_STATS['positive_updates'][k]+=int(d>1e-6)
                VECTOR_STATS['negative_updates'][k]+=int(d< -1e-6)
            VECTOR_STATS['mean_abs_joint_noise_prediction_change']+= (joint_effect-VECTOR_STATS['mean_abs_joint_noise_prediction_change'])/decision
            VECTOR_MEMORY.append({'timestep':int(VECTOR_SCHEDULER.timesteps[index]),'before':previous_observation,
                'applied_vector':applied.round(3).tolist(),'after':observation,'noise_prediction_change':round(joint_effect,7)})
            blob=json.dumps(record,indent=2);assert _api_key not in blob
            (VECTOR_OUT/f'{decision:03d}_decision.json').write_text(blob)
            torch.save({'latent':next_z.cpu(),'weights':next_weights,'decision':decision},VECTOR_OUT/f'{decision:03d}_checkpoint.pt')
            z=next_z;weights=next_weights;previous_observation=observation
            live.update(committed.resize((384,384)))
            print(f'Decision {decision:02d}/50 | edited {int(np.count_nonzero(abs(applied)>1e-6))}/8 | bias={np.round(weights,2).tolist()} | measured joint effect={joint_effect:.6f}',flush=True)
            if decision%5==0:
                strip=frame_strip(VECTOR_FRAMES[-5:],[f'Decision {j}' for j in range(decision-4,decision+1)])
                strip.save(VECTOR_OUT/f'progress_{decision-4:02d}_{decision:02d}.png')
                display(strip)
        final=decode(z)[0];final.save(VECTOR_OUT/'050_final_decoded.png')
        return final
    finally:
        pipe.unet.set_attn_processor(dict(VECTOR_SAVED_PROCESSORS))
        POLICY.update(action='full',boxes=[])

vector_run_started=time.perf_counter()
VECTOR_FINAL=run_vector50()
print('FINISHED:',len(VECTOR_RECORDS),'decisions in',round(time.perf_counter()-vector_run_started,1),'seconds. No attempts discarded.')


<a id="all-decisions"></a>

### All 50 decisions: progression, edit history, and exports

In [ ]:
# 26. Export every decision in chronological order, including holds and regressions
assert len(VECTOR_RECORDS)==50 and len(VECTOR_FRAMES)==51, 'The 50-decision run must finish first.'
vector_rows=[]
for record in VECTOR_RECORDS:
    row={'decision':record['decision'],'timestep':record['state']['diffusion_state']['timestep'],
         'edited_coordinates':int(np.count_nonzero(np.abs(record['applied_delta'])>1e-6)),
         'joint_noise_prediction_change':record['joint_noise_prediction_change']}
    for k,slot in enumerate(VECTOR_SLOTS):
        key=slot['id'];row[key+'_bias']=record['committed_weights'][k]
        row[key+'_delta']=record['applied_delta'][k]
        row[key+'_strengthen_support']=record['jev']['support'][key]['strengthen']
        row[key+'_weaken_support']=record['jev']['support'][key]['weaken']
    vector_rows.append(row)
VECTOR_TABLE=pd.DataFrame(vector_rows)
VECTOR_TABLE.to_csv(VECTOR_OUT/'all_50_decisions.csv',index=False)
# A single contact sheet includes every committed predicted-clean preview.
contact=Image.new('RGB',(5*192,11*218),'white')
for k,frame in enumerate(VECTOR_FRAMES):
    x=(k%5)*192;y=(k//5)*218
    contact.paste(frame.resize((192,192)),(x,y+26))
    ImageDraw.Draw(contact).text((x+6,y+7),'Original' if k==0 else f'Decision {k:02d}',fill='black')
contact.save(VECTOR_OUT/'all_50_progression.png')
animated=[frame.resize((384,384)) for frame in VECTOR_FRAMES]+[VECTOR_FINAL.resize((384,384))]
animated[0].save(VECTOR_OUT/'complete_progression.gif',save_all=True,append_images=animated[1:],
    duration=[1200]+[240]*50+[1800],loop=0)
fig,axes=plt.subplots(2,1,figsize=(11,6),gridspec_kw={'height_ratios':[2,1]})
weights=np.array([r['committed_weights'] for r in VECTOR_RECORDS])
im=axes[0].imshow(weights.T,aspect='auto',cmap='coolwarm',vmin=-1.5,vmax=1.5,extent=[.5,50.5,7.5,-.5])
axes[0].set_yticks(range(8),[s['id'] for s in VECTOR_SLOTS]);axes[0].set_ylabel('Attention relationship')
axes[0].set_title('All jointly committed attention biases');fig.colorbar(im,ax=axes[0],label='Added logit bias')
axes[1].plot(VECTOR_TABLE.decision,VECTOR_TABLE.joint_noise_prediction_change)
axes[1].set_xlabel('Jev decision');axes[1].set_ylabel('Mean absolute change\nin noise prediction')
axes[1].set_title('Effect of each new joint edit at the same latent; this is not a quality score')
plt.tight_layout();fig.savefig(VECTOR_OUT/'attention_edit_history.png',dpi=150);plt.show()
vector_summary={'completed_decisions':50,'jev_judgments_per_call':16,'editable_coordinates':8,
    'decisions_with_multiple_coordinate_edits':int((VECTOR_TABLE.edited_coordinates>=2).sum()),
    'decisions_with_any_edit':int((VECTOR_TABLE.edited_coordinates>0).sum()),
    'total_coordinate_updates':int(VECTOR_TABLE.edited_coordinates.sum()),
    'final_bias_vector':weights[-1].tolist(),'measured_history':VECTOR_STATS,
    'model':VECTOR_RECORDS[0]['jev']['model'],
    'note':'Fixed original image and one forward denoising trajectory. All 50 decisions committed; no best-frame selection. Probes are saved, not hidden. This demonstrates control, not independently verified anatomical improvement.'}
(VECTOR_OUT/'summary.json').write_text(json.dumps(vector_summary,indent=2))
readme='''# Jev: 50 simultaneous-attention-edit decisions
The source is the original seed-123 Jev image from the earlier experiment, not its later refinements.
A fixed 100-step DDIM schedule starts at index 50 from the encoded source with saved noise seed 1235050.
One API request per denoising step contains 16 probability questions, producing eight bounded logit updates applied together.
The current state and two orthogonal joint perturbation probes feed the next decision. Probe rollouts are not separate comparison methods.
Feature clusters define fixed token groups; there are no manually specified hand or finger boxes. Only two named layers, heads 0 and 1, and eight fixed relationships are editable.
The observation model is Qwen2-VL-2B-Instruct. Its descriptions can be wrong, incomplete, or truncated; they are not ground truth. Jev sees text and measured internal graphs, not raw images.
Progression PNG/GIF includes all 50 committed predicted-clean previews. These are intermediate predictions at changing noise levels. 050_final_decoded.png is the fully decoded endpoint.
Every API state, question template, returned probability, edit, group mask, probe preview and committed latent checkpoint is preserved. No manual choice of a best round was used.
The bias trace and noise-prediction effect show whether internal controls were exercised; neither measures image quality or establishes benefit from Jev.
'''
(VECTOR_OUT/'README.md').write_text(readme)
vector_zip=VECTOR_OUT.with_suffix('.zip')
with zipfile.ZipFile(vector_zip,'w',zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(VECTOR_OUT.iterdir()):
        if path.is_file():bundle.write(path,arcname=VECTOR_OUT.name+'/'+path.name)
vector_base='/files'+str(VECTOR_OUT.resolve())
print(json.dumps(vector_summary,indent=2))
display(Markdown('### Complete 50-decision progression — no frames omitted'))
display(DisplayImage(filename=str(VECTOR_OUT/'all_50_progression.png'),width=720))
display(Markdown('### Final decoded endpoint — decision 50, not a selected best image'))
display(VECTOR_FINAL)
display(HTML(f'<a href="{vector_base}/complete_progression.gif" target="_blank">Watch all 50 decisions</a> | <a href="{vector_base}/all_50_progression.png" target="_blank">Every frame</a> | <a href="{vector_base}/attention_edit_history.png" target="_blank">Attention edit history</a> | <a href="/files{vector_zip.resolve()}" download>All evidence and checkpoints</a>'))
print('Artifacts:',VECTOR_OUT)



<a id="architecture"></a>

## 6 · Actual architecture audit

Inspection of the loaded SD1.5 pipeline, attention modules, spatial resolutions, and scheduler. This is a read-only audit; broader interventions discussed afterwards have not been implemented here.

In [ ]:
# 27. Audit the actual loaded architecture and current control surface (no generation)
from collections import Counter
import diffusers, transformers, inspect
arch_modules=dict(pipe.unet.named_modules())
arch_attention=[]
for name,module in arch_modules.items():
    if name.endswith(('.attn1','.attn2')) and hasattr(module,'to_q'):
        arch_attention.append({'module':name,'kind':'spatial self' if name.endswith('attn1') else 'text cross',
            'heads':module.heads,'query_channels':module.to_q.in_features,'context_channels':module.to_k.in_features,
            'head_width':module.to_q.out_features//module.heads,'processor':type(module.processor).__name__})
arch_config={'model':str(pipe.config.get('_name_or_path','unknown')),'pipeline':type(pipe).__name__,
    'diffusers':diffusers.__version__,'transformers':transformers.__version__,
    'unet':{k:pipe.unet.config.get(k) for k in ['in_channels','out_channels','sample_size','block_out_channels','down_block_types','up_block_types','layers_per_block','cross_attention_dim','attention_head_dim']},
    'vae':{'class':type(pipe.vae).__name__,'latent_channels':pipe.vae.config.latent_channels,'spatial_scale_factor':pipe.vae_scale_factor,'amplitude_scaling':pipe.vae.config.scaling_factor},
    'text':{'class':type(pipe.text_encoder).__name__,'hidden_size':pipe.text_encoder.config.hidden_size,'max_positions':pipe.text_encoder.config.max_position_embeddings,'token_limit':pipe.tokenizer.model_max_length},
    'scheduler':{'class':type(pipe.scheduler).__name__,'prediction_type':pipe.scheduler.config.prediction_type,'training_timesteps':pipe.scheduler.config.num_train_timesteps,'step_signature':str(inspect.signature(pipe.scheduler.step))},
    'counts':dict(Counter(x['kind'] for x in arch_attention)),
    'resnet_blocks':sum(type(m).__name__=='ResnetBlock2D' for m in arch_modules.values()),
    'freeu_method_present':hasattr(pipe.unet,'enable_freeu'),
    'controlnet_present':getattr(pipe,'controlnet',None) is not None,
    'last_run_editable_relationships':VECTOR_SLOTS}
print(json.dumps(arch_config,indent=2,default=str))
display(pd.DataFrame(arch_attention))
# One existing-state forward pass records real spatial resolutions. No diffusion step, API call or weight change.
arch_shapes={};arch_handles=[]
def arch_shape_hook(name):
    def hook(module,args):arch_shapes[name]=list(args[0].shape)
    return hook
for row in arch_attention:
    if row['kind']=='spatial self':arch_handles.append(arch_modules[row['module']].register_forward_pre_hook(arch_shape_hook(row['module'])))
try:
    with torch.inference_mode():
        _arch_prediction=pipe.unet(VECTOR_INITIAL.repeat(2,1,1,1),VECTOR_SCHEDULER.timesteps[50],encoder_hidden_states=embeddings).sample
finally:
    for handle in arch_handles:handle.remove()
print('Measured self-attention inputs (batch, spatial tokens, channels):')
print(json.dumps(arch_shapes,indent=2))
arch_config['measured_attention_shapes']=arch_shapes
(VECTOR_OUT/'architecture_audit.json').write_text(json.dumps(arch_config,indent=2,default=str))


## Continue the research

Add the next experiment below. Keep the original seed 123 anchor and retain every attempted intervention, its settings, and its chronological output.

In [ ]:
# 28. Inspect reusable sampler/controller interfaces before the new run
import nbformat, inspect
from pathlib import Path
_nb_source = nbformat.read('/workspace/crazy_exp/Jev_Attention_Downsampling_Hands.ipynb', as_version=4)
for n in (22,23,24,25):
    c = next(c for c in _nb_source.cells if c.source.startswith(f'# {n}.'))
    print(c.source)


In [ ]:
# 29. Agentic generation protocol: exact prompts and hard executor limits
AGENT_CONTROLLER = """Direct an unfinished image-generation process toward the original request. Keep successful visual content while investigating unresolved structure or detail. The state describes executable operations, current model measurements, candidate consequences and accumulated outcomes. Choose the next operation; for an edit, choose coordinated parameter changes considering their combined effects. Inspection, ordinary advance, local refinement and revisiting a saved state are legitimate actions. There is no reward for editing frequently. Computational addresses are not anatomical labels. Use the footprint descriptions and measured interventions to relate them to visible changes. A larger activation change or attention weight is not itself an improvement. Judge coherent shape, connections, proportions and integration, not a count of fingertips. All visual reports are uncertain measurements from a separate small vision model, not ground truth."""
AGENT_STATE_PROMPT = """Read the original request and source observation, the current full-view observation, active working-view observation, noise position, editable head measurements and token footprints. Review the latest consequences and the accumulated operation history. Prefer evidence from comparable noise levels, resolutions and parent states. Treat unmeasured controls as hypotheses. The candidate table distinguishes completed probes from untested actions. Identify the next useful operation through the operation choice and the reason category; select a predicted effect and a check that could disconfirm it. Parameter answers form one simultaneous vector, not independently established causal effects."""
AGENT_LOCAL_PROMPT = """You are inside a temporary detail task. Its soft edit region was derived from model features, not a finger box. Keep the full parent image in view. The crop is a higher spatial sampling density working view, not a different reference image. Continue, change controls, inspect, or return the candidate for full-image integration review. Reject a candidate that improves local appearance but disrupts connections, boundaries, identity or the surrounding image. The executor retains the exact parent checkpoint and guarantees a return; local decisions still count toward the same global budget."""
AGENT_REVIEW_PROMPT = """Compare the recorded before and after observations under their stated conditions. Distinguish visible improvement, regression and unresolved ambiguity. Check structure and pose as well as local texture, boundaries, sleeve and background preservation. Do not infer causation for one coordinate from a joint edit. Do the observations support retaining this change, and what visible discrepancy remains? A successful local crop must also integrate with the full image."""
AGENT_OBSERVER = """Describe visible evidence in this image in at most 100 words. Trace digit-like structures into the palm, noting ambiguous connections, relative proportions, joint bends, overlapping surfaces and small boundaries. Describe visible artifacts and uncertainty without assuming that five tips means correct anatomy. Include changes in pose, skin, sleeve and background when apparent. Use image-relative locations. Do not claim to know what an internal model component represents."""
AGENT_LIMITS = dict(decisions=100, probes_per_decision=2, local_decisions=10, nesting=1,
                    unet_calls=2400, schedule_steps=100, start_index=50,
                    finalization_reserve=220, noise_seed=123100100)
AGENT_OUT = OUT / ('jev123_agent100_' + datetime.now().strftime('%H%M%S'))
AGENT_OUT.mkdir(parents=True,exist_ok=False)
AGENT_SOURCE=RESULTS[(123,'jev')]['image'].copy()
AGENT_SOURCE.save(AGENT_OUT/'000_original.png')
for name in ['AGENT_CONTROLLER','AGENT_STATE_PROMPT','AGENT_LOCAL_PROMPT','AGENT_REVIEW_PROMPT','AGENT_OBSERVER']:
    print(name, globals()[name], '\n')
print('Executor limits:',AGENT_LIMITS)
print('Typed API: one choice for operation plus simultaneous bounded coordinate choices. Visual reports use local Qwen2-VL; Jev receives text/numeric context, not raw images. Reason/effect/check are selected categories, not free-form explanations.')
print('Artifacts:',AGENT_OUT)


In [ ]:
# 30. Multiscale routing, temperature and head-output contribution controls
import copy, math, json, time
import torch.nn.functional as F
from PIL import ImageFilter, ImageDraw
from diffusers import DDIMScheduler
A_SAVED_PROCESSORS=dict(pipe.unet.attn_processors)
A_SCHED=DDIMScheduler.from_config(pipe.scheduler.config)
A_SCHED.set_timesteps(100,device=pipe.device)
A_TARGETS=[
 'mid_block.attentions.0.transformer_blocks.0.attn1.processor',
 'down_blocks.2.attentions.1.transformer_blocks.0.attn1.processor',
 'up_blocks.2.attentions.1.transformer_blocks.0.attn1.processor',
 'up_blocks.3.attentions.1.transformer_blocks.0.attn1.processor']
A_PARAMS={};A_GROUPS={};A_EDGES={};A_GRAPH={};A_CAPTURE=True;A_CALLS=0

def a_defaults():
    return {f'L{li}H{h}':dict(bias=0.,gate=1.,temperature=1.) for li in range(4) for h in (0,1)}

class AgentAttention:
    def __init__(self,name,li): self.name=name;self.li=li
    def __call__(self,attn,hidden_states,encoder_hidden_states=None,attention_mask=None,temb=None,*args,**kwargs):
        assert encoder_hidden_states is None and hidden_states.ndim==3 and attention_mask is None
        assert attn.group_norm is None and attn.spatial_norm is None
        residual=hidden_states;b,n,c=hidden_states.shape;heads=attn.heads
        q,k,v=(f(hidden_states) for f in (attn.to_q,attn.to_k,attn.to_v))
        d=k.shape[-1]//heads
        q=q.view(b,n,heads,d).transpose(1,2);k=k.view(b,n,heads,d).transpose(1,2);v=v.view(b,n,heads,d).transpose(1,2)
        if attn.norm_q is not None:q=attn.norm_q(q)
        if attn.norm_k is not None:k=attn.norm_k(k)
        if self.name not in A_GROUPS: A_GROUPS[self.name]=cluster_tokens(hidden_states[-1])
        lab=A_GROUPS[self.name];assert len(lab)==n
        out=F.scaled_dot_product_attention(q,k,v,dropout_p=0.,is_causal=False)
        for h in (0,1):
            key=f'L{self.li}H{h}';p=A_PARAMS[key]
            qi=torch.arange(0,n,max(1,n//128),device=q.device)
            if key not in A_EDGES or A_CAPTURE:
                logits=(q[-1,h,qi].float()@k[-1,h].float().T)/(d**.5)
                if key not in A_EDGES:
                    mass=logits.softmax(-1)
                    gm=torch.stack([torch.stack([mass[lab[qi]==a][:,lab==g].sum(-1).mean() if (lab[qi]==a).any() else mass[:,lab==g].sum(-1).mean() for g in range(6)]) for a in range(6)])
                    src=int(gm.var(-1).argmax());dst=int(gm[src].masked_fill(torch.arange(6,device=q.device)==src,-1).argmax())
                    A_EDGES[key]=(src,dst)
            src,dst=A_EDGES[key];bias=None
            if abs(p['bias'])>1e-8:
                bias=((lab[:,None]==src)&(lab[None,:]==dst)).to(q.dtype)*p['bias']
            if bias is not None or abs(p['temperature']-1)>1e-8:
                out[-1,h]=F.scaled_dot_product_attention(q[-1,h:h+1]/p['temperature'],k[-1,h:h+1],v[-1,h:h+1],attn_mask=bias,dropout_p=0.)[0]
            out[-1,h]*=p['gate']
            if A_CAPTURE:
                logits=logits/p['temperature']
                if bias is not None:logits+=bias[qi].float()
                mass=logits.softmax(-1)
                gm=torch.stack([torch.stack([mass[lab[qi]==a][:,lab==g].sum(-1).mean() if (lab[qi]==a).any() else mass[:,lab==g].sum(-1).mean() for g in range(6)]) for a in range(6)])
                s=int(n**.5)
                rms=out[-1,h].float().square().mean(-1).sqrt().reshape(1,1,s,s)
                A_GRAPH[key]={'layer':self.name,'head':h,'spatial_shape':[s,s],'source_group':src,'destination_group':dst,
                  'group_mass_6x6':gm.cpu().numpy().round(4).tolist(),'sampled_query_count':len(qi),
                  'entropy':float(-(mass*mass.clamp_min(1e-12).log()).sum(-1).mean()),
                  'gated_output_rms':float(rms.mean()),'contribution_map_4x4':F.adaptive_avg_pool2d(rms,4)[0,0].cpu().numpy().round(4).tolist()}
        out=out.transpose(1,2).reshape(b,n,heads*d).to(q.dtype)
        out=attn.to_out[1](attn.to_out[0](out))
        if attn.residual_connection:out+=residual
        return out/attn.rescale_output_factor

def a_install():
    pipe.unet.set_attn_processor({n:AgentAttention(n,A_TARGETS.index(n)) if n in A_TARGETS else getattr(p,'base',p) for n,p in A_SAVED_PROCESSORS.items()})

@torch.inference_mode()
def a_predict(z,index,params,capture=True):
    global A_PARAMS,A_CAPTURE,A_CALLS
    assert 0<=index<100
    if A_CALLS>=AGENT_LIMITS['unet_calls']:raise RuntimeError('UNet call cap reached; checkpoint retained')
    A_PARAMS=copy.deepcopy(params);A_CAPTURE=capture;A_CALLS+=1
    pred=pipe.unet(z.repeat(2,1,1,1),A_SCHED.timesteps[index],encoder_hidden_states=embeddings).sample
    u,c=pred.chunk(2);return u+CFG*(c-u)

@torch.inference_mode()
def a_step(z,index,params,capture=True):
    eps=a_predict(z,index,params,capture)
    r=A_SCHED.step(eps,A_SCHED.timesteps[index],z,eta=0.)
    assert torch.isfinite(r.prev_sample).all()
    return r.prev_sample,r.pred_original_sample

@torch.inference_mode()
def a_encode(im):
    arr=np.asarray(im.convert('RGB'),dtype=np.float32)/127.5-1
    ten=torch.from_numpy(arr).permute(2,0,1)[None].to(device=pipe.device,dtype=pipe.vae.dtype)
    return pipe.vae.encode(ten).latent_dist.mode()*pipe.vae.config.scaling_factor

A_SOURCE_LATENT=a_encode(AGENT_SOURCE)
_a_gen=torch.Generator(device=pipe.device).manual_seed(AGENT_LIMITS['noise_seed'])
A_NOISE=torch.randn(A_SOURCE_LATENT.shape,device=pipe.device,dtype=A_SOURCE_LATENT.dtype,generator=_a_gen)
A_INITIAL=A_SCHED.add_noise(A_SOURCE_LATENT,A_NOISE,A_SCHED.timesteps[50:51])
A_PARAMS=a_defaults();POLICY.update(action='full',boxes=[])
with torch.inference_mode():
    raw=pipe.unet(A_INITIAL.repeat(2,1,1,1),A_SCHED.timesteps[50],encoder_hidden_states=embeddings).sample
    u,c=raw.chunk(2);_a_baseline=u+CFG*(c-u)
a_install()
try:
    _a_zero=a_predict(A_INITIAL,50,a_defaults())
    A_IDENTITY=float((_a_zero-_a_baseline).abs().max());assert A_IDENTITY<1e-4,A_IDENTITY
    A_NEUTRAL_GRAPH=copy.deepcopy(A_GRAPH)
    _test=a_defaults();_test['L1H0']['gate']=1.15
    _a_gate=a_predict(A_INITIAL,50,_test)
    A_GATE_EFFECT=float((_a_gate-_a_zero).abs().mean());assert A_GATE_EFFECT>1e-7
    _test=a_defaults();_test['L2H1']['bias']=.3;_test['L2H1']['temperature']=.9
    _a_route=a_predict(A_INITIAL,50,_test)
    A_ROUTE_EFFECT=float((_a_route-_a_zero).abs().mean());assert A_ROUTE_EFFECT>1e-7
finally:
    pipe.unet.set_attn_processor(dict(A_SAVED_PROCESSORS))
A_ROOT_GROUPS={k:v.clone() for k,v in A_GROUPS.items()};A_ROOT_EDGES=copy.deepcopy(A_EDGES)
print('Identity max error:',A_IDENTITY,'Head gate effect:',A_GATE_EFFECT,'Routing/temperature effect:',A_ROUTE_EFFECT)
print('Measured spatial grids:',{k:v['spatial_shape'] for k,v in A_NEUTRAL_GRAPH.items()})
print('Conditional self-attention only. Original text prompt and CFG remain fixed. Neutral gate=1, temperature=1, bias=0.')


In [ ]:
# 31. Model-derived edit regions, local tasks, valid-noise integration and restoration

def a_dump(name,obj):
    blob=json.dumps(obj,indent=2,default=lambda x:float(x) if isinstance(x,np.generic) else str(x))
    assert _api_key not in blob
    (AGENT_OUT/name).write_text(blob)

@torch.inference_mode()
def a_observe(images,question=AGENT_OBSERVER):
    return vector_observe(images,question,max_tokens=170)

def a_bind(s):
    global A_GROUPS,A_EDGES
    A_GROUPS=s['groups'];A_EDGES=s['edges']

def a_snapshot(s):
    return {'z':s['z'].clone(),'i':s['i'],'params':copy.deepcopy(s['params']),
            'groups':{k:v.clone() for k,v in s['groups'].items()},'edges':copy.deepcopy(s['edges']),
            'image':s['image'].copy(),'clean':s['clean'].clone(),'graph':copy.deepcopy(s.get('graph',{})),
            'obs':s.get('obs',''),'task':s['task'],'age':s.get('age',0),'footprints':copy.deepcopy(s.get('footprints',{}))}

@torch.inference_mode()
def a_advance(s,steps=2):
    a_bind(s)
    for j in range(s['i'],min(s['i']+steps,100)):
        s['z'],s['clean']=a_step(s['z'],j,s['params'],capture=True);s['i']=j+1
    s['graph']=copy.deepcopy(A_GRAPH)
    s['image']=decode(s['z'] if s['i']==100 else s['clean'])[0]
    return s

def a_masks(s):
    lab=s['groups'][A_TARGETS[1]].detach().cpu().numpy();side=int(len(lab)**.5);lab=lab.reshape(side,side)
    masks=[]
    for g in range(6):
        binary=Image.fromarray((lab==g).astype('uint8')*255).resize(s['image'].size,Image.Resampling.BILINEAR)
        masks.append(binary.filter(ImageFilter.GaussianBlur(max(s['image'].size)/100)))
    return masks

def a_region(mask):
    w=np.asarray(mask,dtype=float)/255;ys,xs=np.where(w>.25)
    if not len(xs):raise RuntimeError('Empty model-derived region')
    # Extent comes only from the feature mask; no manually supplied hand/finger coordinates.
    pad=32;x0=max(0,int(xs.min())-pad);y0=max(0,int(ys.min())-pad)
    x1=min(mask.width,int(xs.max())+pad+1);y1=min(mask.height,int(ys.max())+pad+1)
    return (x0,y0,x1,y1)

def a_ground(s,label):
    masks=a_masks(s);views=[];facts={}
    for g,m in enumerate(masks):
        ar=np.asarray(m,dtype=float)/255;im=np.asarray(s['image'],dtype=float)
        views.append(Image.fromarray(np.uint8(np.clip(im*(.15+.85*ar[:,:,None]),0,255))))
        occ=np.asarray(m.resize((4,4),Image.Resampling.BOX),dtype=float)/255
        facts[str(g)]={'soft_occupancy_4x4':occ.round(3).tolist(),'mask_mean':float(ar.mean()),'crop_extent':list(a_region(m))}
        m.save(AGENT_OUT/f'{label}_region{g}_mask.png')
    captions=a_observe(views,'Describe only the bright model-selected locations and their image-relative position in at most 45 words. Dimmed areas are context. The selection may be disconnected and has no assumed anatomical meaning.')
    for g,caption in enumerate(captions):facts[str(g)]['unverified_footprint_observation']=caption
    s['footprints']=facts
    atlas=frame_strip(views,[f'Feature group {g}' for g in range(6)],width=128)
    atlas.save(AGENT_OUT/f'{label}_region_atlas.png')
    return atlas

@torch.inference_mode()
def a_open_local(parent,group,resolution,restart,decision):
    mask=a_masks(parent)[group];box=a_region(mask)
    crop=parent['image'].crop(box).resize((resolution,resolution),Image.Resampling.LANCZOS)
    clean=a_encode(crop)
    gen=torch.Generator(device=pipe.device).manual_seed(123100100+decision)
    noise=torch.randn(clean.shape,device=clean.device,dtype=clean.dtype,generator=gen)
    z=A_SCHED.add_noise(clean,noise,A_SCHED.timesteps[restart:restart+1])
    s={'z':z,'i':restart,'clean':clean,'image':crop,'params':copy.deepcopy(parent['params']),
       'groups':{},'edges':{},'task':'local','age':0,'obs':'','graph':{},'footprints':{}}
    a_bind(s);a_predict(z,restart,s['params']);s['graph']=copy.deepcopy(A_GRAPH)
    ctx={'parent':a_snapshot(parent),'mask':mask,'box':box,'group':group,'resolution':resolution,'opened_at':decision}
    return s,ctx

@torch.inference_mode()
def a_integrate(local,ctx):
    parent=a_snapshot(ctx['parent']);box=ctx['box'];mask=ctx['mask']
    layer=parent['image'].copy();layer.paste(local['image'].resize((box[2]-box[0],box[3]-box[1]),Image.Resampling.LANCZOS),box[:2])
    candidate=Image.composite(layer,parent['image'],mask)
    candidate_clean=a_encode(candidate)
    lm=F.interpolate(torch.tensor(np.asarray(mask).copy(),device=parent['z'].device,dtype=torch.float32)[None,None]/255,
                     size=parent['z'].shape[-2:],mode='bilinear',align_corners=False).to(parent['z'].dtype)
    # At the parent's SAME next timestep, preserve its residual noise and blend only the candidate clean component.
    if parent['i']<100:
        alpha=A_SCHED.alphas_cumprod[A_SCHED.timesteps[parent['i']]].to(parent['z'])
        noise=(parent['z']-alpha.sqrt()*parent['clean'])/(1-alpha).sqrt().clamp_min(1e-6)
        candidate_z=alpha.sqrt()*candidate_clean+(1-alpha).sqrt()*noise
        parent['z']=parent['z']+lm*(candidate_z-parent['z'])
    else:parent['z']=parent['z']+lm*(candidate_clean-parent['z'])
    parent['clean']=parent['clean']+lm*(candidate_clean-parent['clean'])
    parent['image']=decode(parent['clean'])[0]
    if parent['i']<100:
        a_bind(parent);a_predict(parent['z'],parent['i'],parent['params']);parent['graph']=copy.deepcopy(A_GRAPH)
    return parent

@torch.inference_mode()
def a_reopen(s,restart,decision):
    s=a_snapshot(s);clean=a_encode(s['image']);gen=torch.Generator(device=pipe.device).manual_seed(123200200+decision)
    noise=torch.randn(clean.shape,device=clean.device,dtype=clean.dtype,generator=gen)
    s.update(z=A_SCHED.add_noise(clean,noise,A_SCHED.timesteps[restart:restart+1]),clean=clean,i=restart)
    a_bind(s);a_predict(s['z'],s['i'],s['params']);s['graph']=copy.deepcopy(A_GRAPH)
    return s

A_ROOT={'z':A_INITIAL.clone(),'i':50,'clean':A_SOURCE_LATENT.clone(),'image':AGENT_SOURCE.copy(),
        'params':a_defaults(),'groups':{k:v.clone() for k,v in A_ROOT_GROUPS.items()},'edges':copy.deepcopy(A_ROOT_EDGES),
        'task':'parent','age':0,'graph':copy.deepcopy(A_NEUTRAL_GRAPH),'obs':'','footprints':{}}
A_ROOT['obs']=a_observe([AGENT_SOURCE])[0]
display(a_ground(A_ROOT,'000_source'))
# Exercise local entry, one advance, same-noise integration, and exact parent restoration before API calls.
a_install()
try:
    _local,_ctx=a_open_local(A_ROOT,0,512,80,0)
    _local=a_advance(_local,1);_integrated=a_integrate(_local,_ctx)
    assert _integrated['z'].shape==A_ROOT['z'].shape and _integrated['i']==A_ROOT['i']
    _restored=a_snapshot(_ctx['parent']);assert torch.equal(_restored['z'],A_ROOT['z'])
    _mask=F.interpolate(torch.tensor(np.asarray(_ctx['mask']).copy(),device=A_ROOT['z'].device)[None,None].float(),size=(64,64),mode='bilinear',align_corners=False)
    _delta=(_integrated['z']-A_ROOT['z']).abs()
    _outside=(_mask==0).expand_as(_delta)
    assert not _outside.any() or float(_delta[_outside].max())==0.
    print('Local entry/advance/integration passed; untouched latent locations preserved exactly; restoration is bit-exact.')
finally:pipe.unet.set_attn_processor(dict(A_SAVED_PROCESSORS));a_bind(A_ROOT)
print('Local crops are explicit feature-mask selections. They can cover broad/disconnected regions; no semantic segmentation is claimed.')


In [ ]:
# 32. Typed Jev choices, joint parameter vectors, and evidence packets
A_CAPABILITIES={
 'advance':'Two ordinary DDIM updates at the active noise level; existing controls retained.',
 'edit':'Apply all chosen head bias/gate/temperature deltas together, then advance two DDIM updates.',
 'probe':'Keep the active state unchanged; evaluate two alternative joint vectors for two-step lookahead; retain both candidates.',
 'commit_probe':'Commit the selected valid probe from this exact active state, including its two denoising steps.',
 'open_local':'Save parent; select one of six feature-derived soft masks, crop its measured extent, and restart a 512 or 768 working view at a valid noise level.',
 'return_local':'Create a same-noise parent integration candidate. The next decision must accept or reject its full-image effect.',
 'abandon_local':'Restore the exact parent checkpoint without integration.',
 'restore':'Restore the previous or original complete parent checkpoint, including parameters, feature groups and scheduler index.',
 'reopen':'Re-noise the active predicted clean image with a recorded seeded noise tensor at a chosen valid scheduler index.',
 'inspect':'Obtain a paired before/current full-view observation without changing the latent.',
 'accept_integration':'Accept the pending full-image integration and resume its parent trajectory.',
 'reject_integration':'Discard the pending integration and restore the exact parent.'}
A_BOUNDS={'bias':[-1.5,1.5],'gate':[.5,1.5],'temperature':[.7,1.4]}
A_DELTAS={'bias':.2,'gate':.1,'temperature':.1}
A_REASON={'structure':'Visible spatial connections/proportions remain unresolved.',
 'detail':'A visible local boundary or surface detail needs investigation.',
 'preservation':'A change threatens already satisfactory surrounding content.',
 'trajectory':'The current noise trajectory remains a plausible way forward.',
 'uncertainty':'Missing or conflicting evidence makes another observation useful.',
 'measured_effect':'A recorded comparable intervention supports this choice.'}
A_EFFECT={'geometry':'Connections or proportions change coherently.','detail':'Local boundary or surface detail becomes clearer.',
 'preserve':'Surrounding content remains stable.','diagnose':'The observation distinguishes competing actions.','none':'No justified visible improvement is predicted yet.'}
A_CHECK={'full_context':'A local gain damages full-image connections or composition.','unchanged':'The targeted visible discrepancy persists.',
 'artifact':'New duplicated, fused, sharp or texture artifacts appear.','drift':'Pose, skin appearance, sleeve or background drifts.',
 'inconclusive':'The next observation remains too ambiguous to support the hypothesis.'}

def a_choice(instruction,options):return {'type':'choice','instructions':instruction,'criteria':options}

def a_questions(allowed,local=False,review=False):
    base=AGENT_CONTROLLER+' '+AGENT_STATE_PROMPT+(' '+AGENT_LOCAL_PROMPT if local else '')+(' '+AGENT_REVIEW_PROMPT if review else '')
    questions={'operation':a_choice(base+' Which available operation should execute next?',{k:A_CAPABILITIES[k] for k in allowed}),
      'reason':a_choice(base+' Which evidence category best supports the operation? This is a brief categorical justification, not a causal proof.',A_REASON),
      'expected_effect':a_choice(base+' What observable consequence should be checked?',A_EFFECT),
      'countercheck':a_choice(base+' Which observation would most directly count against this choice?',A_CHECK),
      'region':a_choice(base+' If opening local refinement, which supplied feature-mask region deserves investigation? Use the region footprint observations; labels have no intrinsic anatomy.',{str(g):f'Model feature group {g}; use its measured footprint and current coverage.' for g in range(6)}),
      'resolution':a_choice(base+' Choose the local working resolution if opening refinement.',{'512':'512-square crop pass; lower compute.','768':'768-square crop pass; denser spatial sampling, greater compute and possible SD1.5 scale artifacts.'}),
      'restart':a_choice(base+' Choose a valid noise entry if reopening or starting a local task.',{'50':'Schedule index 50: more noise and structural freedom.','65':'Index 65: intermediate noise and freedom.','80':'Index 80: lower noise, less structural freedom.'}),
      'probe_id':a_choice(base+' If committing a probe, select its stored candidate. Both are compared against the same untouched parent.',{'plus':'The stored plus joint perturbation.','minus':'The stored minus joint perturbation.'}),
      'restore_id':a_choice(base+' If restoring a parent state, choose the saved checkpoint.',{'previous':'Previous committed parent checkpoint.','original':'Original source-derived parent state at index 50.'})}
    for key in a_defaults():
        for kind in A_BOUNDS:
            questions[key+'_'+kind]=a_choice(base+f' For {key}, choose a {kind} delta as part of the simultaneous vector. Use current values, bounds, measurements and comparable probe evidence. No individual causal effect is established by a joint probe.',
                {'down':f'Decrease {kind} by {A_DELTAS[kind]}, bounded to {A_BOUNDS[kind]}.','hold':'Keep this value unchanged.','up':f'Increase {kind} by {A_DELTAS[kind]}, bounded to {A_BOUNDS[kind]}.'})
    return questions

def a_api(state,questions,decision):
    a_dump(f'{decision:03d}_request.json',{'model':'jev-latest','state':state,'questions':questions})
    for attempt in range(5):
        response=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_api_key},
           json={'model':'jev-latest','state':state,'questions':questions},timeout=90)
        if response.status_code not in (429,529,502,503,504):break
        time.sleep(min(2**attempt,8))
    if response.status_code!=200:raise RuntimeError('TypeSafe HTTP '+str(response.status_code)+'; response body withheld')
    data=response.json()
    for key,q in questions.items():
        ans=data['answers'][key];assert ans['choice'] in q['criteria'],key
        assert all(np.isfinite(v) and 0<=v<=1 for v in ans['probabilities'].values()),key
    a_dump(f'{decision:03d}_response.json',data)
    return {key:value['choice'] for key,value in data['answers'].items()},data

def a_params_after(params,choices):
    updated=copy.deepcopy(params)
    for key in updated:
        for kind,(lo,hi) in A_BOUNDS.items():
            delta={'down':-1,'hold':0,'up':1}[choices[key+'_'+kind]]*A_DELTAS[kind]
            updated[key][kind]=round(float(np.clip(params[key][kind]+delta,lo,hi)),4)
    return updated

def a_head_context(s):
    result=copy.deepcopy(s.get('graph',{}))
    for key,g in result.items():
        lab=s['groups'][g['layer']];n=int(len(lab)**.5)
        for role in ('source','destination'):
            mask=(lab==g[role+'_group']).float().reshape(1,1,n,n)
            g[role+'_occupancy_4x4']=F.adaptive_avg_pool2d(mask,4)[0,0].cpu().numpy().round(3).tolist()
        g['current_parameters']=s['params'][key]
    return result

def a_allowed(s,ctx,pending,probes,decision):
    if pending is not None:return ['accept_integration','reject_integration']
    if ctx is not None and (s['age']>=AGENT_LIMITS['local_decisions'] or decision>=99):return ['return_local','abandon_local']
    ops=['probe','reopen','inspect']
    if s['i']<100:ops=['advance','edit']+ops
    if probes:ops+=['commit_probe']
    if ctx is None:
        ops+=['restore']
        if decision<98 and A_CALLS<AGENT_LIMITS['unet_calls']-AGENT_LIMITS['finalization_reserve']-150:ops+=['open_local']
    else:ops+=['return_local','abandon_local']
    if A_CALLS>=AGENT_LIMITS['unet_calls']-AGENT_LIMITS['finalization_reserve']:
        return ['return_local','abandon_local'] if ctx is not None else ['inspect']
    return ops

A_MANIFEST={'limits':AGENT_LIMITS,'source':'RESULTS[(123,jev)] original; never previous refinement endpoint',
  'controller':AGENT_CONTROLLER,'state_prompt':AGENT_STATE_PROMPT,'local_prompt':AGENT_LOCAL_PROMPT,'review_prompt':AGENT_REVIEW_PROMPT,
  'observer_prompt':AGENT_OBSERVER,'capabilities':A_CAPABILITIES,'bounds':A_BOUNDS,'coordinate_steps':A_DELTAS,
  'targets':A_TARGETS,'heads':[0,1],'fixed_CFG':float(CFG),'API':'typed choice; raw probabilities retained; no free-form reasoning output',
  'visibility':'Jev receives numeric internal measurements and unverified local-Qwen visual descriptions. Raw images/maps are retained for human inspection.',
  'control_scope':'Self-attention biases, temperatures, gates; task/mask/resolution/restart/trajectory choices. Text, CFG, FreeU and model weights are not modified.',
  'finalization':'After decision 100, reject any unreviewed integration, restore any unfinished local parent, finish remaining parent DDIM steps with last controls; record every completion frame separately.',
  'validation':{'identity_max_error':A_IDENTITY,'gate_noise_effect':A_GATE_EFFECT,'routing_temperature_effect':A_ROUTE_EFFECT,'local_restore':'bit-exact'}}
a_dump('manifest.json',A_MANIFEST)
print('Example exact executable questions:')
print(json.dumps(a_questions(['advance','edit','probe','open_local','inspect'])['operation'],indent=2))
print(json.dumps(a_questions(['advance','edit'])['L1H0_gate'],indent=2))
print('Each action call returns',len(a_questions(['advance'])),'typed answers, including 24 simultaneous parameter choices. Irrelevant parameter choices are logged but not applied.')


In [ ]:
# 33. The 100-decision state machine: all branches, returns, evidence and frames retained
A_RECORDS=[];A_FRAMES=[AGENT_SOURCE.copy()];A_COMPLETION_FRAMES=[];A_OPERATION_COUNTS={}
A_STATE=None

def a_pixel_delta(before,after,mask=None):
    b=np.asarray(before.resize((512,512)),dtype=float)/255;a=np.asarray(after.resize((512,512)),dtype=float)/255
    delta=np.abs(a-b).mean(-1);result={'mean_abs_rgb_change':float(delta.mean())}
    if mask is not None:
        m=np.asarray(mask.resize((512,512)),dtype=float)/255
        result['mask_weighted_change']=float((delta*m).sum()/max(m.sum(),1))
        result['outside_weighted_change']=float((delta*(1-m)).sum()/max((1-m).sum(),1))
    return result

def a_checkpoint(s):
    return {**{k:s[k] for k in ['i','params','edges','task','age','obs','graph','footprints']},
            'z':s['z'].cpu(),'clean':s['clean'].cpu(),'groups':{k:v.cpu() for k,v in s['groups'].items()}}

def a_status(decision,s,status='running',**extra):
    a_dump('status.json',{'status':status,'decisions_completed':decision,'budget':100,'task':s['task'],
       'schedule_index':s['i'],'unet_calls':A_CALLS,'operation_counts':A_OPERATION_COUNTS,**extra})

def a_export_progress():
    cols=8;width=128;rows=math.ceil(len(A_FRAMES)/cols)
    sheet=Image.new('RGB',(cols*width,rows*(width+25)),'white')
    for start in range(0,len(A_FRAMES),cols):
        strip=frame_strip(A_FRAMES[start:start+cols],[('Original' if j==0 else f'Decision {j}') for j in range(start,min(start+cols,len(A_FRAMES)))],width)
        sheet.paste(strip,(0,(start//cols)*(width+25)))
    sheet.save(AGENT_OUT/'all_decisions_progression.png')
    pd.DataFrame([{'decision':r['decision'],'operation':r['operation'],'task':r['after_task'],
        'schedule_index':r['after_index'],'reason':r['choices']['reason'],'expected_effect':r['choices']['expected_effect'],
        'countercheck':r['choices']['countercheck'],'changed_coordinates':r['changed_coordinates']} for r in A_RECORDS]).to_csv(AGENT_OUT/'decisions.csv',index=False)

@torch.inference_mode()
def a_probes(s,proposed,decision):
    # Baseline and both alternatives start from precisely the same latent/noise index.
    baseline=a_advance(a_snapshot(s),2)
    direction={key:{kind:proposed[key][kind]-s['params'][key][kind] for kind in A_BOUNDS} for key in proposed}
    if sum(abs(v) for d in direction.values() for v in d.values())<1e-6:
        for j,key in enumerate(direction):
            for k,kind in enumerate(A_BOUNDS):direction[key][kind]=A_DELTAS[kind]*(1 if ((j*3+k)&(decision%32)).bit_count()%2==0 else -1)
    branches={};images=[baseline['image']]
    baseline['image'].save(AGENT_OUT/f'{decision:03d}_probe_baseline.png')
    for name,sign in [('plus',1),('minus',-1)]:
        branch=a_snapshot(s)
        for key in branch['params']:
            for kind,(lo,hi) in A_BOUNDS.items():branch['params'][key][kind]=float(np.clip(s['params'][key][kind]+sign*direction[key][kind],lo,hi))
        branch=a_advance(branch,2);branches[name]=branch;images.append(branch['image'])
        branch['image'].save(AGENT_OUT/f'{decision:03d}_probe_{name}.png')
    obs=a_observe(images);evidence={'baseline':{'observation':obs[0],'index':baseline['i'],'params':baseline['params']},'candidates':{}}
    for (name,branch),ob in zip(branches.items(),obs[1:]):
        branch['obs']=ob
        evidence['candidates'][name]={'params':branch['params'],'observation':ob,'delta_vs_baseline':a_pixel_delta(baseline['image'],branch['image']),
            'measured_heads':branch['graph'],'index':branch['i']}
    evidence['valid_start_index']=s['i'];evidence['task']=s['task'];evidence['created_at']=decision
    a_dump(f'{decision:03d}_probe_evidence.json',evidence)
    a_bind(s)
    return branches,evidence

@torch.inference_mode()
def run_agent100():
    global A_STATE,A_CALLS
    A_CALLS=0;s=a_snapshot(A_ROOT);ctx=None;pending=None;probes={};probe_evidence={};previous_parent=a_snapshot(s)
    recent=[];decision=0;last_pair_review='No intervention yet.';original_observation=s['obs'];started=time.perf_counter()
    live=display(AGENT_SOURCE.resize((384,384)),display_id=True)
    a_install()
    try:
        for decision in range(1,101):
            a_bind(s)
            allowed=a_allowed(s,ctx,pending,probes,decision)
            if s['i']>=100 and 'probe' in allowed:allowed.remove('probe')
            full=ctx['parent']['image'] if ctx is not None else s['image']
            parent_info=None if ctx is None else {'opened_at':ctx['opened_at'],'local_decisions':s['age'],'feature_group':ctx['group'],
              'crop_extent':list(ctx['box']),'working_resolution':ctx['resolution'],'parent_index':ctx['parent']['i'],'parent_observation':ctx['parent']['obs']}
            state={'original_request':'Studio photograph of a single human hand, open palm facing camera, all five fingers spread apart and fully visible, wrist entering from bottom, centered, plain dark gray background, realistic skin, sharp focus',
              'source_observation_unverified':original_observation,'active_observation_unverified':s['obs'],
              'full_view_observation_unverified':ctx['parent']['obs'] if ctx else s['obs'],
              'last_paired_review_unverified':last_pair_review,'active_task':s['task'],'noise':{'index':s['i'],'timestep':int(A_SCHED.timesteps[s['i']]) if s['i']<100 else None,'preview':'predicted clean estimate' if s['i']<100 else 'decoded endpoint'},
              'resolution':list(s['image'].size),'heads':a_head_context(s),'local_region_footprints':s['footprints'],
              'local_task':parent_info,'available_operations':{k:A_CAPABILITIES[k] for k in allowed},
              'joint_probe_evidence':probe_evidence,'recent_outcomes':recent[-5:],
              'accumulated_operations':copy.deepcopy(A_OPERATION_COUNTS),
              'integration_review':None if pending is None else pending['evidence'],
              'control_bounds':A_BOUNDS,'uncertainty':'Visual captions may hallucinate. Feature memberships are fixed within a task and can lose alignment after geometry changes. Sampled graph measurements are not causal explanations. Jev does not receive raw pixels.'}
            questions=a_questions(allowed,local=ctx is not None,review=pending is not None)
            choices,answer=a_api(state,questions,decision);op=choices['operation']
            before=a_snapshot(s);before_full=full.copy();proposed=a_params_after(s['params'],choices)
            changed=0;event={};new_local=False
            if ctx is None and pending is None and op not in ('restore','inspect','probe'):
                previous_parent=a_snapshot(s)
            if op=='advance':s=a_advance(s,2);probes={};probe_evidence={}
            elif op=='edit':
                changed=sum(abs(proposed[k][v]-s['params'][k][v])>1e-6 for k in proposed for v in A_BOUNDS)
                s['params']=proposed;s=a_advance(s,2);probes={};probe_evidence={}
            elif op=='probe':
                probes,probe_evidence=a_probes(s,proposed,decision);event={'probe_evidence_file':f'{decision:03d}_probe_evidence.json'}
            elif op=='commit_probe':
                s=a_snapshot(probes[choices['probe_id']]);event={'committed_probe':choices['probe_id'],'probe_created_at':probe_evidence['created_at']}
                changed=sum(abs(s['params'][k][v]-before['params'][k][v])>1e-6 for k in s['params'] for v in A_BOUNDS)
                probes={};probe_evidence={}
            elif op=='open_local':
                s,ctx=a_open_local(s,int(choices['region']),int(choices['resolution']),int(choices['restart']),decision)
                atlas=a_ground(s,f'{decision:03d}_local');display(atlas)
                event={'feature_group':ctx['group'],'crop_extent':list(ctx['box']),'resolution':ctx['resolution'],'restart':s['i']}
                new_local=True;probes={};probe_evidence={}
            elif op=='return_local':
                # Return a completed local decode, not an unfinished noisy crop; record every completion step.
                for j in range(s['i'],100):
                    s=a_advance(s,1);s['image'].save(AGENT_OUT/f'{decision:03d}_local_completion_{j+1:03d}.png')
                candidate=a_integrate(s,ctx);candidate['image'].save(AGENT_OUT/f'{decision:03d}_integration_candidate.png')
                review_pair=frame_strip([ctx['parent']['image'],candidate['image']],['Before parent','Candidate integration'],width=384)
                review_pair.save(AGENT_OUT/f'{decision:03d}_integration_review.png')
                comparison=a_observe([review_pair],'Compare LEFT before and RIGHT candidate. '+AGENT_REVIEW_PROMPT)[0]
                pending={'candidate':candidate,'evidence':{'comparison_unverified':comparison,
                      'changes':a_pixel_delta(ctx['parent']['image'],candidate['image'],ctx['mask']),
                      'feature_group':ctx['group'],'parent_index':ctx['parent']['i']}}
                display(review_pair);probes={};probe_evidence={};event=pending['evidence']
            elif op=='abandon_local':s=a_snapshot(ctx['parent']);ctx=None;probes={};probe_evidence={}
            elif op in ('accept_integration','reject_integration'):
                event={'integration':copy.deepcopy(pending['evidence'])}
                s=a_snapshot(pending['candidate'] if op=='accept_integration' else ctx['parent'])
                ctx=None;pending=None;probes={};probe_evidence={}
            elif op=='restore':
                s=a_snapshot(previous_parent if choices['restore_id']=='previous' else A_ROOT)
                event={'restored':choices['restore_id']};probes={};probe_evidence={}
            elif op=='reopen':
                s=a_reopen(s,int(choices['restart']),decision);event={'restart':s['i'],'noise_seed':123200200+decision};probes={};probe_evidence={}
            elif op=='inspect':
                pair=frame_strip([AGENT_SOURCE,full],['Original','Current full view'],width=384)
                last_pair_review=a_observe([pair],'Compare the LEFT original and RIGHT current view. '+AGENT_REVIEW_PROMPT)[0]
                event={'inspection_unverified':last_pair_review}
            else:raise AssertionError(op)
            if ctx is not None:s['age']=decision-ctx['opened_at']
            s['obs']=a_observe([s['image']])[0]
            after_full=ctx['parent']['image'] if ctx is not None else s['image']
            if decision%5==0 or op in ('accept_integration','reject_integration'):
                pair=frame_strip([before_full,after_full],['Before action','After action'],width=384)
                pair.save(AGENT_OUT/f'{decision:03d}_full_review.png')
                last_pair_review=a_observe([pair],'Compare LEFT before action and RIGHT after action. '+AGENT_REVIEW_PROMPT)[0]
            consequences={'operation':op,'before_task':before['task'],'after_task':s['task'],'before_index':before['i'],'after_index':s['i'],
               'before_observation':before['obs'],'after_observation':s['obs'],'full_view_delta':a_pixel_delta(before_full,after_full),
               'paired_review_unverified':last_pair_review,'event':event,'params_before':before['params'],'params_after':s['params']}
            recent.append(consequences);A_OPERATION_COUNTS[op]=A_OPERATION_COUNTS.get(op,0)+1
            record={'decision':decision,'operation':op,'choices':choices,'changed_coordinates':changed,
              'after_task':s['task'],'after_index':s['i'],'consequences':consequences,'elapsed_seconds':time.perf_counter()-started,'unet_calls':A_CALLS}
            A_RECORDS.append(record);A_FRAMES.append(after_full.copy());A_STATE=s
            a_dump(f'{decision:03d}_decision.json',record)
            after_full.save(AGENT_OUT/f'{decision:03d}_full_committed.png');s['image'].save(AGENT_OUT/f'{decision:03d}_active_preview.png')
            ckpt={'active':a_checkpoint(s),'decision':decision,'parent':None if ctx is None else a_checkpoint(ctx['parent']),
                  'local_context':None if ctx is None else {k:ctx[k] for k in ('box','group','resolution','opened_at')},
                  'pending_integration':None if pending is None else a_checkpoint(pending['candidate']),
                  'probe_candidates':{k:a_checkpoint(v) for k,v in probes.items()},'probe_evidence':probe_evidence}
            if ctx is not None:ckpt['soft_mask']=np.asarray(ctx['mask'])
            torch.save(ckpt,AGENT_OUT/f'{decision:03d}_checkpoint.pt')
            a_status(decision,s)
            if ctx is not None:live.update(frame_strip([after_full,s['image']],['Parent (uncommitted local)','Local working view'],width=320))
            else:live.update(after_full.resize((384,384)))
            print(f'Decision {decision:03d}/100 | {op} | task={s["task"]} index={s["i"]} | joint edits={changed} | UNet calls={A_CALLS}',flush=True)
            if decision%5==0:
                display(frame_strip(A_FRAMES[-5:],[f'Decision {j}' for j in range(decision-4,decision+1)],width=144));a_export_progress()
        # Deterministic completion is explicitly separated from Jev's 100 decisions.
        if ctx is not None:
            a_dump('unfinished_local_return.json',{'reason':'Decision budget exhausted; unreviewed local work is not automatically accepted.','pending_integration':pending is not None})
            s=a_snapshot(ctx['parent']);ctx=None;pending=None
        for j in range(s['i'],100):
            s=a_advance(s,1);A_COMPLETION_FRAMES.append(s['image'].copy())
            s['image'].save(AGENT_OUT/f'completion_{j+1:03d}.png')
        A_STATE=s;s['image'].save(AGENT_OUT/'final_decoded.png');a_export_progress()
        movie=[im.resize((384,384)) for im in A_FRAMES+A_COMPLETION_FRAMES]
        movie[0].save(AGENT_OUT/'complete_progression.gif',save_all=True,append_images=movie[1:],duration=450,loop=0)
        a_status(100,s,'complete',completion_frames=len(A_COMPLETION_FRAMES),elapsed_seconds=time.perf_counter()-started)
        a_dump('summary.json',{'decisions':len(A_RECORDS),'operations':A_OPERATION_COUNTS,'completion_frames':len(A_COMPLETION_FRAMES),
            'elapsed_seconds':time.perf_counter()-started,'unet_calls':A_CALLS,'outcome':'Chronological endpoint, not a selected best image; anatomy improvement requires visual assessment.'})
        live.update(s['image'].resize((384,384)))
        print('COMPLETE: 100 Jev decisions. Endpoint and every attempted branch are retained in',AGENT_OUT,flush=True)
        return s['image']
    except Exception as exc:
        A_STATE=s;a_export_progress()
        a_status(len(A_RECORDS),s,'error',error_type=type(exc).__name__,failed_decision=decision)
        torch.save({'active':a_checkpoint(s),'parent':None if ctx is None else a_checkpoint(ctx['parent'])},AGENT_OUT/'failure_checkpoint.pt')
        raise
    finally:
        pipe.unet.set_attn_processor(dict(A_SAVED_PROCESSORS));POLICY.update(action='full',boxes=[])

print('State machine ready. Local tasks cannot nest. A pending integration only permits accept/reject. Budget exhaustion restores unfinished local work to its saved parent and then completes the parent schedule.')
print('Outputs: status.json; exact per-decision request/response; all probes; all checkpoints; full/active frames; chronology CSV, contact sheet and GIF.')


In [ ]:
# 34. Final preflight: high-resolution path and mandatory return rules
assert a_allowed(A_ROOT,None,{'candidate':None},{},1)==['accept_integration','reject_integration']
_test_state=a_snapshot(A_ROOT);_test_state.update(task='local',age=10)
assert a_allowed(_test_state,{'parent':A_ROOT},None,{},20)==['return_local','abandon_local']
a_install()
try:
    _hi,_hi_ctx=a_open_local(A_ROOT,0,768,80,999)
    _hi=a_advance(_hi,1)
    assert _hi['image'].size==(768,768) and _hi['z'].shape[-2:]==(96,96)
    _hi_parent=a_integrate(_hi,_hi_ctx)
    assert _hi_parent['z'].shape==A_ROOT['z'].shape and torch.isfinite(_hi_parent['z']).all()
    print('768-pixel local generation, integration, and forced-return rules: passed.')
finally:
    pipe.unet.set_attn_processor(dict(A_SAVED_PROCESSORS));a_bind(A_ROOT)
print('Ready to start 100 decisions from original seed 123. No Jev API calls made by preflight.')


## 7 · Seed 123: agentic generation — 100 decisions

The exact prompts and implementation are in cells 29–34 above. The live run below starts from the original seed-123 Jev image.

**Available controls:** multiscale self-attention bias, head-output gate and temperature; feature-mask local refinement at 512/768 pixels; advance, probe, commit, reopen, restore, and explicit local integration review.

**Evidence:** every request, response, probe, committed/full-view image, local working image and checkpoint is retained. Local previews are labeled separately from committed parent images. The endpoint is not selected as a best attempt.

**Visibility limitation:** Jev receives measured internals and unverified Qwen2-VL descriptions, not raw pixels. Typed reason/effect/check selections are recorded; they are not free-form explanations.

The executor allows 100 decisions, at most 10 decisions per local task and one nesting level. A separate completion phase finishes the parent diffusion trajectory without pretending those steps were Jev choices.

In [ ]:
# 35. Start the fixed 100-decision Jev run (do not rerun this cell)
assert not A_RECORDS, 'This run already has decisions. Preserve its record; do not restart in the same output directory.'
from IPython.display import HTML
_a_url='/files/workspace/crazy_exp/'+str(AGENT_OUT.resolve().relative_to(Path('/workspace/crazy_exp')))
display(HTML(f'<p><b>Live run artifacts:</b> <a href="{_a_url}/status.json" target="_blank">Status</a> | <a href="{_a_url}/manifest.json" target="_blank">Exact protocol</a> | <a href="{_a_url}/all_decisions_progression.png" target="_blank">Full progression (updates every 5 decisions)</a> | <a href="{_a_url}/decisions.csv" target="_blank">Decision table</a></p>'))
print('Starting original seed 123; artifacts:',AGENT_OUT,flush=True)
AGENT_FINAL=run_agent100()
display(AGENT_FINAL)



In [ ]:
# 37. W&B SDK and Jev's explicit tracking-organization choice (outside generation budget)
%pip -q install wandb
_tracking_questions={'organization':{'type':'choice','instructions':'Choose a W&B organization for this experiment. One interrupted but resumable SD1.5/Jev seed-123 experiment has a fixed total of 100 decisions. The user wants a separate run with every historical and future decision, including failures. Choose the project/run grouping that makes continuity and failure history easiest to audit. This is a tracking decision, not one of the 100 generation decisions.','criteria':{'dedicated_project':'Create jev-diffusion-control, with one run seed123-agent100; backfill and resume the same experiment in that run.','existing_project':'Place a new seed123-agent100 run in an existing suitable diffusion project if one is available.','split_runs':'Create separate runs for each local refinement or resumed segment, connected by metadata.'}}}
_tracking_state={'known_existing_suitable_project':False,'experiment':'SD1.5 seed123, controller choices over attention and generation','completed_decisions':17,'interruption':'max_tokens_exceeded on decision18','requirements':['one chronological record','all attempted branches','visible failures','no best-attempt filter']}
_tracking_response=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_api_key},json={'model':'jev-latest','state':_tracking_state,'questions':_tracking_questions},timeout=90)
assert _tracking_response.status_code==200, _tracking_response.status_code
TRACKING_CHOICE=_tracking_response.json()
a_dump('tracking_organization_choice.json',{'state':_tracking_state,'questions':_tracking_questions,'response':TRACKING_CHOICE})
print('Jev tracking choice:',TRACKING_CHOICE['answers']['organization'])


In [ ]:
# 38. Compact Jev context, preserve rejected request, and validate decision-18 transport
import shutil
if not (AGENT_OUT/'018_request_rejected_original.json').exists():
    shutil.copy2(AGENT_OUT/'018_request.json',AGENT_OUT/'018_request_rejected_original.json')

def a_round_tree(x):
    if isinstance(x,float):return round(x,4)
    if isinstance(x,dict):return {k:a_round_tree(v) for k,v in x.items()}
    if isinstance(x,(list,tuple)):return [a_round_tree(v) for v in x]
    return x

def a_compact_state(state):
    s=copy.deepcopy(state)
    history=[]
    for r in s.get('recent_outcomes',[]):
        history.append({k:r.get(k) for k in ('operation','before_task','after_task','before_index','after_index','after_observation','full_view_delta')})
        history[-1]['joint_parameter_changes']={k:{v:round(r['params_after'][k][v]-r['params_before'][k][v],3) for v in A_BOUNDS if abs(r['params_after'][k][v]-r['params_before'][k][v])>1e-6} for k in r.get('params_after',{})}
        history[-1]['joint_parameter_changes']={k:v for k,v in history[-1]['joint_parameter_changes'].items() if v}
    s['recent_outcomes']=history[-4:]
    probe=s.get('joint_probe_evidence',{})
    if probe:
        base=probe['baseline'];compact={'baseline_observation':base['observation'],'baseline_index':base['index'],
            'valid_start_index':probe['valid_start_index'],'created_at':probe['created_at'],'task':probe['task'],'candidates':{}}
        for name,c in probe['candidates'].items():
            heads={}
            for key,g in c['measured_heads'].items():
                src=g['source_group'];dst=g['destination_group']
                heads[key]={'edited_edge_mass':g['group_mass_6x6'][src][dst],'entropy':g['entropy'],'gated_output_rms':g['gated_output_rms']}
            compact['candidates'][name]={'observation':c['observation'],'delta_vs_baseline':c['delta_vs_baseline'],
                'joint_parameter_delta':{k:{v:round(c['params'][k][v]-base['params'][k][v],3) for v in A_BOUNDS} for k in c['params']},'measured_heads':heads}
        s['joint_probe_evidence']=compact
    # Full current graph stays visible; repetitive spatial arrays are quantized, not assigned semantic labels.
    s['controller_instruction']=AGENT_CONTROLLER
    s['state_instruction']=AGENT_STATE_PROMPT
    if s.get('active_task')=='local':s['local_instruction']=AGENT_LOCAL_PROMPT
    if s.get('integration_review'):s['review_instruction']=AGENT_REVIEW_PROMPT
    s['archive_note']='Full graph arrays, long histories, requests, responses and images are retained in the run archive. This transport view removes repeated historical parameter vectors and reports probe graph effects at each editable edge.'
    return a_round_tree(s)

def a_compact_questions(questions):
    cleaned=copy.deepcopy(questions)
    for q in cleaned.values():
        text=q['instructions']
        for p in (AGENT_CONTROLLER,AGENT_STATE_PROMPT,AGENT_LOCAL_PROMPT,AGENT_REVIEW_PROMPT):text=text.replace(p,'')
        q['instructions']='Apply the controller/state instructions in the supplied state. '+text.strip()
    return cleaned

A_CACHED_RESPONSES={}
def a_api(state,questions,decision):
    compact=a_compact_state(state);qs=a_compact_questions(questions)
    payload={'model':'jev-latest','state':compact,'questions':qs}
    a_dump(f'{decision:03d}_full_context.json',state)
    a_dump(f'{decision:03d}_request.json',payload)
    if decision in A_CACHED_RESPONSES:data=A_CACHED_RESPONSES.pop(decision)
    else:
        for attempt in range(5):
            response=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_api_key},json=payload,timeout=90)
            if response.status_code not in (429,529,502,503,504):break
            time.sleep(min(2**attempt,8))
        if response.status_code!=200:
            try:detail=response.json().get('detail',{})
            except Exception:detail={'error_type':'non_json_response'}
            safe={'http_status':response.status_code,'detail':detail,'state_bytes':len(json.dumps(compact))}
            a_dump(f'{decision:03d}_api_error.json',safe)
            raise RuntimeError('TypeSafe HTTP '+str(response.status_code)+'; safe diagnostic saved')
        data=response.json()
    for key,q in qs.items():
        ans=data['answers'][key];assert ans['choice'] in q['criteria'],key
        assert all(np.isfinite(v) and 0<=v<=1 for v in ans['probabilities'].values()),key
    a_dump(f'{decision:03d}_response.json',data)
    return {key:value['choice'] for key,value in data['answers'].items()},data

_resume_choices,_resume_response=a_api(_failed_request['state'],_failed_request['questions'],18)
A_CACHED_RESPONSES[18]=_resume_response
_compact_payload=json.loads((AGENT_OUT/'018_request.json').read_text())
print('Decision18 accepted and cached for execution:',_resume_choices['operation'])
print('State bytes:',len(json.dumps(_failed_request['state'])),'->',len(json.dumps(_compact_payload['state'])))
print('Total payload bytes:',len(json.dumps(_failed_request)),'->',len(json.dumps(_compact_payload)))
a_dump('recovery_note.json',{'failure_after_decision':17,'cause':'max_tokens_exceeded','transport_change':'deduplicate instructions/history and reduce probe graphs to measured editable-edge effects; full context retained','cached_decision18':True,'no_restart_or_discarded_decisions':True})


In [ ]:
# 39. Restore exact decision-17 state and add a non-blocking monitoring hook
import ast
_resume17=torch.load(AGENT_OUT/'017_checkpoint.pt',map_location='cpu',weights_only=False)
def a_restore_checkpoint(cp,image):
    s=copy.deepcopy(cp)
    s['z']=cp['z'].to(device=pipe.device,dtype=A_INITIAL.dtype)
    s['clean']=cp['clean'].to(device=pipe.device,dtype=A_INITIAL.dtype)
    s['groups']={k:v.to(pipe.device) for k,v in cp['groups'].items()}
    s['image']=image.copy()
    return s
_A_RESUME_STATE=a_restore_checkpoint(_resume17['active'],Image.open(AGENT_OUT/'017_active_preview.png').convert('RGB'))
_A_RESUME_PARENT=a_restore_checkpoint(_resume17['parent'],Image.open(AGENT_OUT/'017_full_committed.png').convert('RGB'))
_A_RESUME_CTX={**_resume17['local_context'],'parent':_A_RESUME_PARENT,'mask':Image.fromarray(_resume17['soft_mask'])}
_A_RESUME_PROBES={k:a_restore_checkpoint(cp,Image.open(AGENT_OUT/f'017_probe_{k}.png').convert('RGB')) for k,cp in _resume17['probe_candidates'].items()}
assert len(A_RECORDS)==17 and len(A_FRAMES)==18 and set(_A_RESUME_PROBES)=={'plus','minus'}
assert torch.equal(_A_RESUME_STATE['z'],A_STATE['z'])
A_MONITOR=None
A_MONITOR_ERRORS=[]
def a_monitor_emit(record):
    if A_MONITOR is not None:
        try:A_MONITOR(record)
        except Exception as exc:
            A_MONITOR_ERRORS.append({'decision':record['decision'],'error_type':type(exc).__name__})
            a_dump('monitoring_errors.json',A_MONITOR_ERRORS)
            print('Monitoring update deferred; generation checkpoint remains saved.',flush=True)

# Recover the existing executor body without rerunning setup or any completed decision.
_executor_cell=next(src for src in reversed(In) if src.startswith('# 33. The 100-decision state machine'))
_executor_tree=ast.parse(_executor_cell)
_executor_node=next(node for node in _executor_tree.body if isinstance(node,ast.FunctionDef) and node.name=='run_agent100')
_resume_source=ast.get_source_segment(_executor_cell,_executor_node).replace('def run_agent100():','def resume_agent100():',1)
_begin=_resume_source.index('    A_CALLS=0;')
_end=_resume_source.index('    live=display',_begin)
_resume_initialization='''    A_CALLS=A_RECORDS[-1]['unet_calls'];s=a_snapshot(_A_RESUME_STATE);ctx=copy.deepcopy(_A_RESUME_CTX);pending=None
    probes={k:a_snapshot(v) for k,v in _A_RESUME_PROBES.items()};probe_evidence=copy.deepcopy(_resume17['probe_evidence']);previous_parent=a_snapshot(_A_RESUME_PARENT)
    recent=[copy.deepcopy(r['consequences']) for r in A_RECORDS[-5:]];decision=17
    last_pair_review=A_RECORDS[-1]['consequences']['paired_review_unverified'];original_observation=A_ROOT['obs']
    started=time.perf_counter()-A_RECORDS[-1]['elapsed_seconds']
'''
_resume_source=_resume_source[:_begin]+_resume_initialization+_resume_source[_end:]
_resume_source=_resume_source.replace('range(1,101)','range(18,101)',1)
_resume_source=_resume_source.replace("if ctx is not None and not new_local:s['age']+=1","if ctx is not None:s['age']=decision-ctx['opened_at']")
_resume_source=_resume_source.replace('            a_status(decision,s)','            a_status(decision,s)\n            a_monitor_emit(record)')
exec(compile('@torch.inference_mode()\n'+_resume_source,'<resume from decision17>','exec'))
(AGENT_OUT/'resume_executor.py').write_text('@torch.inference_mode()\n'+_resume_source)
print('Exact latent restored; local parent/mask/probes retained. Resume begins with cached Jev decision18:',_resume_choices['operation'])
print('Completed decisions will not be replayed. Monitoring failures cannot terminate generation.')


In [ ]:
# 40. Authenticate the separate W&B monitoring process through a masked input
import getpass, os
os.environ['WANDB_API_KEY']=getpass.getpass('W&B tracking key (hidden): ')
print('Credential loaded in memory. W&B will run in a separate process so its SDK does not interfere with the diffusion kernel.')


In [ ]:
# 41. Independent W&B watcher: backfill every decision, then follow the run live
import subprocess, sys, textwrap
WANDB_RUN_ID='seed123-agent100-014328'
_monitor_code = r'''
import os, sys, json, time, traceback, zipfile
from pathlib import Path
import wandb
p=Path(sys.argv[1]);rid=sys.argv[2]
def read(name,default=None):
    try:return json.loads((p/name).read_text())
    except (FileNotFoundError,json.JSONDecodeError):return default

def status(**kw):
    (p/'wandb_monitor_status.json').write_text(json.dumps(kw,indent=2))
run=None
try:
    manifest=read('manifest.json',{})
    run=wandb.init(entity='peepaclan',project='jev-diffusion-control',id=rid,resume='allow',
       name='seed123-agent100',job_type='agentic-diffusion',group='original-seed123',
       tags=['sd15','jev','100-decisions','all-attempts','resumed-after-context-limit'],
       config=manifest,dir=str(p),save_code=False,
       settings=wandb.Settings(console='off',disable_git=True,init_timeout=90),
       notes='One chronological experiment. Decisions 1–17 backfilled. TypeSafe context overflow at 18 preserved; recovered from the exact saved latent. Full and local images are separate. No best-image selection.')
    run.define_metric('decision')
    for pattern in ['progress/*','schedule/*','cost/*','edits/*','changes/*','head/*','operation/*','choice/*','images/*']:
        run.define_metric(pattern,step_metric='decision')
    cols=['decision','operation','task','schedule_index','joint_edits','reason_category','expected_effect','countercheck','full_committed','active_view','unverified_observation']
    table=wandb.Table(columns=cols,log_mode='MUTABLE')
    run.log({'decision':0,'images/original':wandb.Image(str(p/'000_original.png'),caption='Original seed123 anchor; not a preferred endpoint'),'progress/completed':0})
    run.summary['tracking_organization_choice']=read('tracking_organization_choice.json',{}).get('response',{})
    run.summary['previous_failure']='After decision17: TypeSafe max_tokens_exceeded. Full rejected request preserved.'
    run.summary['anatomy_correctness']='Not established; visual observer is fallible.'
    run.summary['checkpoint_location']='Remote run directory; tensor checkpoints not uploaded by this image/metrics watcher.'
    (p/'wandb_run_url.txt').write_text(run.url)
    status(status='running',url=run.url,last_logged=0)
    logged=0;last_update=time.time();last_heartbeat=0
    while True:
        d=logged+1
        record=read(f'{d:03d}_decision.json')
        ready=record is not None and (p/f'{d:03d}_full_committed.png').exists() and (p/f'{d:03d}_active_preview.png').exists() and (p/f'{d:03d}_checkpoint.pt').exists()
        if ready:
            c=record['consequences'];answer=read(f'{d:03d}_response.json',{}).get('answers',{})
            full=wandb.Image(str(p/f'{d:03d}_full_committed.png'),caption=f'Decision {d}: committed parent | {record["operation"]}')
            active=wandb.Image(str(p/f'{d:03d}_active_preview.png'),caption=f'Decision {d}: {record["after_task"]} working view; index {record["after_index"]}')
            metrics={'decision':d,'progress/completed':d,'schedule/index':record['after_index'],
                'cost/unet_calls':record['unet_calls'],'cost/generation_elapsed_seconds':record['elapsed_seconds'],
                'edits/coordinates':record['changed_coordinates'],'progress/local_task':int(record['after_task']=='local'),
                'changes/full_mean_abs_rgb':c['full_view_delta']['mean_abs_rgb_change'],
                'images/full_committed':full,'images/active_view':active}
            for head,params in c['params_after'].items():
                for kind,value in params.items():metrics[f'head/{head}_{kind}']=value
            for op,prob in answer.get('operation',{}).get('probabilities',{}).items():metrics['choice/'+op]=prob
            for op in manifest.get('capabilities',{}):metrics['operation/'+op]=int(record['operation']==op)
            for suffix in ['probe_baseline','probe_plus','probe_minus','integration_review','integration_candidate','full_review','local_region_atlas']:
                im=p/f'{d:03d}_{suffix}.png'
                if im.exists():metrics['images/'+suffix]=wandb.Image(str(im),caption=f'Decision {d}: {suffix}; retained irrespective of acceptance')
            table.add_data(d,record['operation'],record['after_task'],record['after_index'],record['changed_coordinates'],
               record['choices']['reason'],record['choices']['expected_effect'],record['choices']['countercheck'],full,active,c['after_observation'])
            if d%5==0 or d==17:metrics['decision_history']=table
            run.log(metrics)
            run.summary.update({'last_decision':d,'last_operation':record['operation'],'active_task':record['after_task'],
                  'last_reason_category':record['choices']['reason'],'last_observation_unverified':c['after_observation']})
            logged=d;last_update=time.time();status(status='running',url=run.url,last_logged=logged)
            continue
        current=read('status.json',{})
        if time.time()-last_heartbeat>10:
            run.summary.update({'generation_status':current.get('status','unknown'),'seconds_since_new_decision':int(time.time()-last_update),
                                'generation_error':current.get('error_type','none')})
            last_heartbeat=time.time();status(status='watching',url=run.url,last_logged=logged,generation_status=current.get('status'))
        if current.get('status')=='complete' and logged>=current.get('decisions_completed',100):
            run.log({'decision':logged,'decision_history':table,'images/final_endpoint':wandb.Image(str(p/'final_decoded.png'),caption='Chronological final endpoint after explicitly separate completion phase')})
            bundle=p/'wandb_evidence.zip'
            with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as archive:
                for f in sorted(p.iterdir()):
                    if f.is_file() and f.suffix in ('.json','.png','.gif','.csv','.py') and f.name!='wandb_monitor_status.json':archive.write(f,f.name)
            artifact=wandb.Artifact('seed123-agent100-complete-evidence',type='experiment-evidence',description='All attempted images and probes, exact prompts/responses, chronology and recovery evidence. Tensor checkpoints remain on Runpod.')
            artifact.add_file(str(bundle));run.log_artifact(artifact)
            run.summary.update({'generation_status':'complete','result_selection':'chronological endpoint','completion_frames':current.get('completion_frames',0)})
            run.finish();status(status='complete',url=run.url,last_logged=logged);break
        if time.time()-last_update>3600:
            run.summary['watcher_exit']='No new decision for one hour; remote evidence remains saved.'
            run.finish(exit_code=1);status(status='inactive',url=run.url,last_logged=logged);break
        time.sleep(2)
except Exception as exc:
    status(status='error',error_type=type(exc).__name__,message=str(exc).replace(os.environ.get('WANDB_API_KEY','not-a-key'),'[redacted]'))
    traceback.print_exc()
    if run is not None:run.finish(exit_code=1)
    raise
'''
_monitor_path=AGENT_OUT/'wandb_monitor.py'
_monitor_path.write_text(textwrap.dedent(_monitor_code))
_monitor_log=open(AGENT_OUT/'wandb_monitor.log','a')
assert 'WANDB_API_KEY' in os.environ
WANDB_MONITOR_PROCESS=subprocess.Popen([sys.executable,'-u',str(_monitor_path.resolve()),str(AGENT_OUT.resolve()),WANDB_RUN_ID],
     stdout=_monitor_log,stderr=subprocess.STDOUT,cwd=str(Path.cwd()),env=os.environ.copy(),start_new_session=True)
print('Independent watcher PID:',WANDB_MONITOR_PROCESS.pid)
print('Dashboard will be recorded in:',AGENT_OUT/'wandb_run_url.txt')


In [ ]:
# 42. Verify W&B connection before resuming the generation loop
_status_path=AGENT_OUT/'wandb_monitor_status.json'
print('Watcher process exit:',WANDB_MONITOR_PROCESS.poll())
print(_status_path.read_text() if _status_path.exists() else 'W&B is initializing; the watcher log is retained locally.')
if (AGENT_OUT/'wandb_run_url.txt').exists():
    WANDB_URL=(AGENT_OUT/'wandb_run_url.txt').read_text().strip()
    display(HTML(f'<h3>Live W&B dashboard</h3><p><a href="{WANDB_URL}" target="_blank">Open seed123-agent100 in W&B</a></p><p>Images, every decision, joint controls, operation probabilities and interruption history. Use the step slider on image panels to inspect the progression.</p>'))


In [ ]:
# 43. Resume decisions 18–100; W&B follows saved files independently
assert len(A_RECORDS)==17 and 18 in A_CACHED_RESPONSES
assert WANDB_MONITOR_PROCESS.poll() is None
print('Resuming exact saved state at decision18. W&B:',WANDB_URL,flush=True)
a_status(17,_A_RESUME_STATE,'resuming',recovery='Context transport compacted; cached decision18 applied once.')
AGENT_FINAL=resume_agent100()
display(AGENT_FINAL)
print('Chronological endpoint complete. W&B watcher uploads the full evidence bundle automatically.')


In [ ]:
# 44. Adaptive feature-group experiment: environment and reusable implementation audit
import ast, copy, json, time, hashlib, threading, subprocess, sys, os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from scipy import ndimage
assert json.loads((AGENT_OUT/'status.json').read_text())['status']=='complete'
assert os.environ.get('WANDB_API_KEY') and _api_key
V_OUT=OUT/('jev123_adaptive_'+time.strftime('%Y%m%d_%H%M%S'))
V_OUT.mkdir(parents=True,exist_ok=False)
V_CONFIG=dict(seed=123,action_budget=100,analysis_request_budget=600,unet_budget=2400,
              completion_reserve=160,local_depth=1,local_decision_limit=18,
              scales=[8,16,32,64],heads=list(range(8)),advance_steps=2,probe_steps=[2,6],
              resolutions=[512,768],margins=[0.1,0.25,0.5],model='jev-1.13.0')
(V_OUT/'config.json').write_text(json.dumps(V_CONFIG,indent=2))
print('New run:',V_OUT.resolve())
print('W&B CLI:',subprocess.run([sys.executable,'-m','wandb','--version'],capture_output=True,text=True).stdout.strip())
_wanted={'a_predict','a_step','a_encode','a_integrate','a_reopen','vector_observe','decode'}
_seen={}
for _src in In[:-1]:
 try: _tree=ast.parse(_src)
 except SyntaxError: continue
 for _n in _tree.body:
  if isinstance(_n,(ast.FunctionDef,ast.ClassDef)) and (_n.name in _wanted or _n.name=='AgentAttentionProcessor'):
   _seen[_n.name]=ast.get_source_segment(_src,_n)
print('\n\n'.join(_seen.values()))


In [ ]:
# 45. Verbatim focused Jev questions, observer prompt, and bounded API transport
import requests
V_PROMPTS={
 'representation':'Which representation of group is better supported for the next localized experiment: its current membership, its connected components, or its supplied feature subdivision? Use measured geometry and prior interventions; a component ID has no anatomical meaning.',
 'region':'Which listed region merits the next focused experiment, given the spatial records, current visual observations, and measured intervention history? Select unresolved if none is distinguishable.',
 'neighbor':'Which supplied neighboring component, if any, needs to remain in the contextual view to assess changes to this region?',
 'head':'Which listed head has the most useful measured relationship to the selected region for a new intervention probe? Large magnitude alone does not establish usefulness.',
 'edge':'Which listed destination group is the most informative routing connection to probe from the selected source region? Use the measured attention mass and intervention history.',
 'parameter':'Which listed bounded change to this parameter is worth testing as part of the candidate joint edit? This is an untested proposal, not an established improvement. Hold is allowed.',
 'action':'Which listed operation should be executed next, given its measured consequences, unresolved observations, and the current trajectory state? The objective is a plausible hand with coherent connections, proportions and detail while preserving the original request. Group IDs and internal statistics are not anatomy judgments.',
 'branch':'Which matched branch should be retained after the supplied observations at two and six denoising steps? Prefer a changed branch only when the evidence supports a useful effect; unresolved is allowed.',
 'local':'Do the supplied local and contextual observations support improved connections, proportions, or detail relative to the matched starting image?',
 'regression':'Do the supplied observations support a new defect in neighboring structure or a material unintended change outside the edited region?',
 'persistence':'Do the later observations support retaining the earlier assessment of the intervention?',
 'integration':'Should the integrated candidate be retained, restored to its parent, or examined more closely, given the separate local, contextual and full-image review evidence?'
}
V_OBSERVER='''Examine only the supplied labeled image views. Describe visible connections, proportions, occlusion, boundaries and surface detail. For each finding give its view and approximate location, concrete evidence, and whether clearly visible or ambiguous. For paired views, describe discernible differences before expressing a preference; report no discernible difference when appropriate. Do not infer internal controls or their intent. Finger count and sharpness alone do not establish a plausible hand. Include relevant findings beyond these categories. Describe hidden details as not assessable.'''
V_LOCK=threading.Lock();V_API_COUNTS={'analysis':0,'action':0};V_API_CACHE={};V_USAGE=[]
def v_dump(name,value):
 text=json.dumps(value,indent=2,default=lambda x: x.item() if hasattr(x,'item') else str(x))
 assert _api_key not in text and os.environ['WANDB_API_KEY'] not in text
 path=V_OUT/name;tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(text);tmp.replace(path)
def v_choice(instructions,criteria):return {'type':'choice','instructions':instructions,'criteria':criteria}
def v_call(tag,state,questions,kind='analysis'):
 payload={'model':V_CONFIG['model'],'state':state,'questions':questions}
 raw=json.dumps(payload,sort_keys=True);key=hashlib.sha256(raw.encode()).hexdigest()
 with V_LOCK:
  if key in V_API_CACHE:return copy.deepcopy(V_API_CACHE[key])
  cap=V_CONFIG['analysis_request_budget'] if kind=='analysis' else 100
  if V_API_COUNTS[kind]>=cap:raise RuntimeError('Jev request budget exhausted: '+kind)
  V_API_COUNTS[kind]+=1;serial=sum(V_API_COUNTS.values())
 v_dump(f'api_{serial:04}_{tag}_request.json',payload)
 for attempt in range(4):
  r=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_api_key},json=payload,timeout=90)
  if r.status_code==200:break
  if r.status_code not in (429,500,502,503,529):raise RuntimeError(f'Jev HTTP {r.status_code}; request saved at {tag}')
  time.sleep(2**attempt)
 if r.status_code!=200:raise RuntimeError('Jev retries exhausted')
 result=r.json();answers=result['answers']
 for qid,q in questions.items():
  if q['type']=='choice':assert answers[qid]['choice'] in q['criteria']
 v_dump(f'api_{serial:04}_{tag}_response.json',result)
 with V_LOCK:V_API_CACHE[key]=answers;V_USAGE.append(result.get('usage',{}))
 return answers
v_dump('prompts.json',{'jev':V_PROMPTS,'visual_observer':V_OBSERVER})
for name,prompt in V_PROMPTS.items():print(name+': '+prompt+'\n')
print('QWEN:',V_OBSERVER)


In [ ]:
# 46. Instrumented attention at four scales: all heads, arbitrary region routing, bounded joint edits
import torch.nn.functional as F
V_TARGETS=list(A_TARGETS);V_BASE_PROCESSORS=dict(A_SAVED_PROCESSORS)
V_ACTIVE=None;V_CAPTURE=False;V_CALLS=0
class AdaptiveAttention:
 def __init__(self,name,layer):self.name=name;self.layer=layer
 def __call__(self,attn,hidden_states,encoder_hidden_states=None,attention_mask=None,temb=None,*args,**kwargs):
  residual=hidden_states;ndim=hidden_states.ndim
  if attn.spatial_norm is not None:hidden_states=attn.spatial_norm(hidden_states,temb)
  if ndim==4:
   b,ch,hh,ww=hidden_states.shape;hidden_states=hidden_states.view(b,ch,hh*ww).transpose(1,2)
  b,n,ch=hidden_states.shape
  if attn.group_norm is not None:hidden_states=attn.group_norm(hidden_states.transpose(1,2)).transpose(1,2)
  q=attn.to_q(hidden_states);k=attn.to_k(hidden_states);v=attn.to_v(hidden_states)
  h=attn.heads;d=q.shape[-1]//h
  q=q.view(b,n,h,d).transpose(1,2);k=k.view(b,n,h,d).transpose(1,2);v=v.view(b,n,h,d).transpose(1,2)
  if getattr(attn,'norm_q',None) is not None:q=attn.norm_q(q)
  if getattr(attn,'norm_k',None) is not None:k=attn.norm_k(k)
  out=F.scaled_dot_product_attention(q,k,v,dropout_p=0.)
  state=V_ACTIVE;lh,lw=state['z'].shape[-2:];gh=round((n*lh/lw)**.5);gw=n//gh
  assert gh*gw==n
  for hd,p in state['params'].get(self.name,{}).items():
   hd=int(hd);bias=None
   if p['bias']:
    qm=F.interpolate(p['source'][None,None].float().to(q.device),size=(gh,gw),mode='nearest').flatten().to(q.dtype)
    km=F.interpolate(p['destination'][None,None].float().to(q.device),size=(gh,gw),mode='nearest').flatten().to(q.dtype)
    bias=p['bias']*qm[:,None]*km[None,:]
   changed=F.scaled_dot_product_attention(q[-1:,hd:hd+1]/p['temperature'],k[-1:,hd:hd+1],v[-1:,hd:hd+1],attn_mask=bias,dropout_p=0.)
   out[-1:,hd:hd+1]=changed*p['gate']
  if V_CAPTURE:
   labels=state['groups'].get(self.name)
   if labels is None or labels.numel()!=n:
    labels=cluster_tokens(hidden_states[-1],k=6).detach().cpu();state['groups'][self.name]=labels
   qi=torch.linspace(0,n-1,min(n,128),device=q.device).long()
   logits=(q[-1,:,qi].float()@k[-1].float().transpose(-1,-2))*(d**-.5)
   for hd,p in state['params'].get(self.name,{}).items():
    hd=int(hd);logits[hd]/=p['temperature']
    if p['bias']:
     sm=F.interpolate(p['source'][None,None].float().to(q.device),size=(gh,gw),mode='nearest').flatten()
     dm=F.interpolate(p['destination'][None,None].float().to(q.device),size=(gh,gw),mode='nearest').flatten()
     logits[hd]+=p['bias']*sm[qi,None]*dm[None,:]
   probs=logits.softmax(-1);lab=labels.to(q.device);mass=torch.zeros(h,6,6,device=q.device)
   for src in range(6):
    rows=lab[qi]==src
    if rows.any():
     for dst in range(6):mass[:,src,dst]=probs[:,rows][:,:,lab==dst].sum(-1).mean(-1)
   state['captures'][self.name]={'shape':[gh,gw],'features':hidden_states[-1].detach().cpu().half(),
    'query_indices':qi.cpu(),'attention_rows':probs.detach().cpu().half(),
    'group_mass':mass.cpu().tolist(),'entropy':(-(probs*probs.clamp_min(1e-8).log()).sum(-1).mean(-1)).cpu().tolist(),
    'head_rms':out[-1].float().square().mean((1,2)).sqrt().cpu().tolist()}
  out=out.transpose(1,2).reshape(b,n,h*d).to(q.dtype);out=attn.to_out[0](out);out=attn.to_out[1](out)
  if ndim==4:out=out.transpose(-1,-2).reshape(b,ch,hh,ww)
  if attn.residual_connection:out=out+residual
  return out/attn.rescale_output_factor

def v_install():
 pipe.unet.set_attn_processor({name:AdaptiveAttention(name,V_TARGETS.index(name)) if name in V_TARGETS else proc for name,proc in V_BASE_PROCESSORS.items()})
def v_predict(state,capture=True):
 global V_ACTIVE,V_CAPTURE,V_CALLS
 assert state['i']<100 and V_CALLS<V_CONFIG['unet_budget']
 V_ACTIVE=state;V_CAPTURE=capture;V_CALLS+=1
 pred=pipe.unet(state['z'].repeat(2,1,1,1),A_SCHED.timesteps[state['i']],encoder_hidden_states=embeddings).sample
 u,c=pred.chunk(2);return u+CFG*(c-u)
def v_clone(s):
 r=copy.copy(s);r['z']=s['z'].clone();r['clean']=s['clean'].clone();r['params']=copy.deepcopy(s['params']);r['groups']=dict(s['groups']);r['captures']=dict(s['captures']);r['image']=s['image'].copy()
 return r
@torch.inference_mode()
def v_advance(s,steps=2):
 s=v_clone(s)
 for _ in range(min(steps,100-s['i'])):
  eps=v_predict(s,True);r=A_SCHED.step(eps,A_SCHED.timesteps[s['i']],s['z'],eta=0.)
  s['z']=r.prev_sample;s['clean']=r.pred_original_sample;s['i']+=1
 s['image']=decode(s['clean'])[0];s['revision']+=1
 assert torch.isfinite(s['z']).all()
 return s
print('Four modules, all eight heads available. Conditional branch edits; original processors retained for restoration.')


In [ ]:
# 47. Adaptive group catalogues, connected masks, subdivisions, neighbors, and real zoom geometry
V_REGION_CACHE={};V_VISION_CACHE={};V_LEDGER=[];V_RECORDS=[];V_FRAMES=[];V_EXECUTION_WARNINGS=[]
def v_catalogue(s,layer):
 name=V_TARGETS[layer];cap=s['captures'][name];gh,gw=cap['shape'];labels=s['groups'][name].numpy().reshape(gh,gw)
 feats=F.normalize(cap['features'].float(),dim=-1).numpy().reshape(gh,gw,-1)
 regions={};components=[]
 def add(rid,mask,kind,parent):
  ys,xs=np.where(mask)
  if len(xs)<2:return
  bbox=[int(xs.min()),int(ys.min()),int(xs.max()+1),int(ys.max()+1)]
  _,nc=ndimage.label(mask);vectors=feats[mask];center=vectors.mean(0)
  grid=F.adaptive_avg_pool2d(torch.from_numpy(mask.astype('float32'))[None,None],(4,4))[0,0].tolist()
  record={'id':rid,'kind':kind,'parent_group':int(parent),'tokens':int(mask.sum()),'fraction':round(float(mask.mean()),4),
   'components':int(nc),'bbox_normalized':[round(bbox[0]/gw,4),round(bbox[1]/gh,4),round(bbox[2]/gw,4),round(bbox[3]/gh,4)],
   'feature_dispersion':round(float(1-np.linalg.norm(center)),4),'occupancy4x4':np.round(grid,2).tolist()}
  regions[rid]={'mask':torch.from_numpy(mask.copy()),'record':record,'layer':layer}
 for g in range(6):
  raw=labels==g;add(f'L{layer}G{g}',raw,'feature_group',g)
  cc,nc=ndimage.label(raw);sizes=np.bincount(cc.ravel());ids=sorted(range(1,nc+1),key=lambda j:(-sizes[j],j))
  for j in ids[:4]:
   mask=cc==j;rid=f'L{layer}G{g}C{j}';add(rid,mask,'connected_component',g)
   if rid not in regions:continue
   components.append(rid)
   if mask.sum()>=8:
    vec=feats[mask];axis=vec[np.argmin(vec@vec[0])]-vec[0];projection=vec@axis
    order=np.argsort(projection,kind='stable');part=np.zeros(len(vec),dtype=bool);part[order[:len(vec)//2]]=True
    for k,take in enumerate([part,~part]):
     sm=np.zeros_like(mask);sm[mask]=take;add(rid+f'S{k}',sm,'feature_subdivision',g)
 # Spatially adjacent alternatives; masks and original groups are both retained.
 for rid in components:
  base=regions[rid];boundary=ndimage.binary_dilation(base['mask'].numpy())
  near=[other for other in components if other!=rid and np.logical_and(boundary,regions[other]['mask'].numpy()).any()]
  base['record']['neighbors']=near
  for other in near[:1]:
   mask=np.logical_or(base['mask'],regions[other]['mask']).numpy() if isinstance(np.logical_or(base['mask'],regions[other]['mask']),torch.Tensor) else np.logical_or(base['mask'].numpy(),regions[other]['mask'].numpy())
   add(rid+'M'+other,mask,'neighbor_union',base['record']['parent_group'])
 return regions

def v_crop_spec(image,mask,margin,long_side):
 w,h=image.size;ys,xs=np.where(mask.numpy())
 if not len(xs):raise ValueError('Empty mask')
 gh,gw=mask.shape;x0=xs.min()*w/gw;y0=ys.min()*h/gh;x1=(xs.max()+1)*w/gw;y1=(ys.max()+1)*h/gh
 pad=max(8,margin*max(x1-x0,y1-y0));box=(max(0,int(x0-pad)),max(0,int(y0-pad)),min(w,int(np.ceil(x1+pad))),min(h,int(np.ceil(y1+pad))))
 cw,ch=box[2]-box[0],box[3]-box[1];scale=long_side/max(cw,ch)
 # Isotropic resize, followed by edge padding to UNet-compatible multiples of 64.
 rw,rh=max(64,round(cw*scale)),max(64,round(ch*scale));pw=int(np.ceil(rw/64)*64);ph=int(np.ceil(rh/64)*64)
 return {'box':list(map(int,box)),'resized':[rw,rh],'working':[pw,ph],'magnification':round(scale,3),'margin':margin,'long_side':long_side}
def v_softmask(image,mask):
 return Image.fromarray(mask.numpy().astype('uint8')*255).resize(image.size,Image.Resampling.BILINEAR).filter(ImageFilter.GaussianBlur(max(image.size)/200))
def v_region_views(s,region,prefix):
 mask=v_softmask(s['image'],region['mask']);dim=s['image'].point(lambda x:int(x*.25));highlight=Image.composite(s['image'],dim,mask)
 spec=v_crop_spec(s['image'],region['mask'],.25,512);crop=s['image'].crop(spec['box'])
 highlight.save(V_OUT/(prefix+'_footprint.png'));crop.save(V_OUT/(prefix+'_context.png'));mask.save(V_OUT/(prefix+'_mask.png'))
 return highlight,crop,spec
@torch.inference_mode()
def v_vision(images,labels):
 strip=frame_strip(images,labels,width=384);key=hashlib.sha256(strip.tobytes()+V_OBSERVER.encode()).hexdigest()
 if key not in V_VISION_CACHE:V_VISION_CACHE[key]=vector_observe([strip],V_OBSERVER,max_tokens=260)[0]
 return {'report':V_VISION_CACHE[key],'view_labels':labels,'image_hash':key,'source':'Qwen2-VL-2B; uncalibrated visual estimate'}
def v_delta(before,after,mask=None):
 b=np.asarray(before).astype('float32')/255;a=np.asarray(after.resize(before.size)).astype('float32')/255;d=np.abs(a-b).mean(-1)
 out={'mean_abs_rgb':float(d.mean()),'p95_abs_rgb':float(np.quantile(d,.95))}
 if mask is not None:
  m=np.asarray(mask.resize(before.size)).astype('float32')/255
  out.update(inside=float((d*m).sum()/max(m.sum(),1)),outside=float((d*(1-m)).sum()/max((1-m).sum(),1)))
 return out
print('Catalogues retain feature groups, connected components, binary feature subdivisions and neighbor unions.')
print('Zoom is isotropic, padded to multiples of64, and reports actual crop magnification.')


In [ ]:
# 48. Checkpoints, masked local trajectories, matched probes and full-image integration
@torch.inference_mode()
def v_refresh(s,recluster=False):
 old=dict(s['groups'])
 if recluster:s['groups']={}
 if s['i']<100:v_predict(s,True)
 correspondence={}
 if recluster:
  for name,labels in s['groups'].items():
   if name not in old or old[name].shape!=labels.shape:continue
   overlaps=[]
   for g in range(6):
    a=old[name]==g;scores=[float(((a)&(labels==j)).sum()/((a)|(labels==j)).sum().clamp_min(1)) for j in range(6)]
    j=int(np.argmax(scores));overlaps.append({'old_group':g,'new_group':j,'iou':round(scores[j],3),'ambiguous':scores[j]<.5})
   correspondence[name]=overlaps
 s['correspondence']=correspondence
 return s

# Preserve context in local denoising at the appropriate noise level, after every step.
@torch.inference_mode()
def v_advance(s,steps=2):
 s=v_clone(s)
 for _ in range(min(steps,100-s['i'])):
  eps=v_predict(s,True);r=A_SCHED.step(eps,A_SCHED.timesteps[s['i']],s['z'],eta=0.)
  z=r.prev_sample;clean=r.pred_original_sample;s['i']+=1
  if 'latent_mask' in s:
   m=s['latent_mask'];reference=s['reference_clean']
   rz=A_SCHED.add_noise(reference,s['reference_noise'],A_SCHED.timesteps[s['i']:s['i']+1]) if s['i']<100 else reference
   z=m*z+(1-m)*rz;clean=m*clean+(1-m)*reference
  s['z']=z;s['clean']=clean
 s['image']=decode(s['clean'])[0];s['revision']+=1
 assert torch.isfinite(s['z']).all()
 return v_refresh(s)

@torch.inference_mode()
def v_open_local(parent,region,margin,resolution,restart,decision):
 spec=v_crop_spec(parent['image'],region['mask'],margin,resolution);mask=v_softmask(parent['image'],region['mask'])
 crop=parent['image'].crop(spec['box']).resize(spec['resized'],Image.Resampling.LANCZOS)
 arr=np.asarray(crop);pw,ph=spec['working'];rw,rh=spec['resized']
 work=Image.fromarray(np.pad(arr,((0,ph-rh),(0,pw-rw),(0,0)),mode='edge'))
 clean=a_encode(work);gen=torch.Generator(device=pipe.device).manual_seed(123300000+decision)
 noise=torch.randn(clean.shape,device=clean.device,dtype=clean.dtype,generator=gen)
 cm=mask.crop(spec['box']).resize(spec['resized'],Image.Resampling.BILINEAR)
 ma=np.pad(np.asarray(cm),((0,ph-rh),(0,pw-rw)),mode='constant')
 lm=F.interpolate(torch.from_numpy(ma.copy()).float()[None,None].to(clean.device)/255,size=clean.shape[-2:],mode='bilinear',align_corners=False).to(clean.dtype)
 s={'z':A_SCHED.add_noise(clean,noise,A_SCHED.timesteps[restart:restart+1]),'clean':clean,'image':work,
    'i':restart,'groups':{},'captures':{},'params':{},'revision':parent['revision']+1,'task':f'local_{decision:03}',
    'latent_mask':lm,'reference_clean':clean,'reference_noise':noise}
 ctx={'parent':v_clone(parent),'mask':mask,'spec':spec,'region':region['record'],'opened_at':decision}
 return v_refresh(s),ctx
@torch.inference_mode()
def v_integrate(local,ctx):
 parent=v_clone(ctx['parent']);spec=ctx['spec'];box=spec['box'];rw,rh=spec['resized']
 patch=local['image'].crop((0,0,rw,rh)).resize((box[2]-box[0],box[3]-box[1]),Image.Resampling.LANCZOS)
 canvas=parent['image'].copy();canvas.paste(patch,box[:2]);candidate_clean=a_encode(canvas)
 lm=F.interpolate(torch.from_numpy(np.asarray(ctx['mask']).copy()).float()[None,None].to(parent['z'].device)/255,size=parent['z'].shape[-2:],mode='bilinear',align_corners=False).to(parent['z'].dtype)
 old_clean=parent['clean'];new_clean=old_clean+lm*(candidate_clean-old_clean)
 if parent['i']<100:
  alpha=A_SCHED.alphas_cumprod[A_SCHED.timesteps[parent['i']]].to(parent['z'])
  parent['z']=parent['z']+alpha.sqrt()*(new_clean-old_clean)
 else:parent['z']=new_clean.clone()
 parent['clean']=new_clean;parent['image']=decode(new_clean)[0];parent['revision']+=1
 return v_refresh(parent)
@torch.inference_mode()
def v_reopen(s,index,decision):
 s=v_clone(s);clean=a_encode(s['image']);gen=torch.Generator(device=pipe.device).manual_seed(123400000+decision)
 noise=torch.randn(clean.shape,device=clean.device,dtype=clean.dtype,generator=gen)
 s.update(z=A_SCHED.add_noise(clean,noise,A_SCHED.timesteps[index:index+1]),clean=clean,i=index,revision=s['revision']+1)
 if 'latent_mask' in s:s.update(reference_clean=clean,reference_noise=noise)
 return v_refresh(s,recluster=True)
def v_checkpoint(s,name):
 obj={k:v for k,v in s.items() if k not in ['image','captures']}
 def cpu(x):
  if isinstance(x,torch.Tensor):return x.detach().cpu()
  if isinstance(x,dict):return {k:cpu(v) for k,v in x.items()}
  return x
 torch.save(cpu(obj),V_OUT/(name+'_checkpoint.pt'))
 torch.save(cpu(s['captures']),V_OUT/(name+'_measured_tensors.pt'))
 s['image'].save(V_OUT/(name+'_active.png'))
print('Local edits preserve noisy context after each step; integration applies one latent mask at the saved parent noise level.')


In [ ]:
# 49. Preflight: identity, group diversity, true magnification, masked integration, observer reliability
@torch.inference_mode()
def v_preflight():
 source=RESULTS[(123,'jev')]['image'].copy();clean=a_encode(source)
 gen=torch.Generator(device=pipe.device).manual_seed(123100100);noise=torch.randn(clean.shape,device=clean.device,dtype=clean.dtype,generator=gen)
 root={'z':A_SCHED.add_noise(clean,noise,A_SCHED.timesteps[50:51]),'clean':clean,'image':source,'i':50,
       'groups':{},'captures':{},'params':{},'revision':0,'task':'parent'}
 pipe.unet.set_attn_processor(dict(V_BASE_PROCESSORS))
 raw=pipe.unet(root['z'].repeat(2,1,1,1),A_SCHED.timesteps[50],encoder_hidden_states=embeddings).sample
 u,c=raw.chunk(2);reference=u+CFG*(c-u)
 v_install();observed=v_predict(root);err=float((reference-observed).abs().max())
 assert err<.02,('neutral attention mismatch',err)
 cats={l:v_catalogue(root,l) for l in range(4)}
 chosen=min([r for r in cats[2].values() if r['record']['kind']=='connected_component' and .02<r['record']['fraction']<.3],key=lambda r:abs(r['record']['fraction']-.08))
 local,ctx=v_open_local(root,chosen,.25,512,80,0)
 assert ctx['spec']['magnification']>1
 moved=v_advance(local,2);integrated=v_integrate(moved,ctx)
 mask=F.interpolate(torch.from_numpy(np.asarray(ctx['mask']).copy()).float()[None,None].to(root['z'].device)/255,size=root['z'].shape[-2:],mode='bilinear',align_corners=False)
 outside=(mask==0).expand_as(root['z']);outside_error=float((integrated['z']-root['z'])[outside].abs().max()) if outside.any() else None
 assert outside_error==0.0
 changed=v_clone(root);name=V_TARGETS[2];destination=cats[2][next(k for k in cats[2] if k.startswith('L2G') and cats[2][k]['record']['kind']=='feature_group')]
 changed['params']={name:{'7':{'bias':.2,'temperature':.9,'gate':1.1,'source':chosen['mask'],'destination':destination['mask']}}}
 effect=float((v_predict(changed)-observed).abs().mean());assert effect>0
 # 768 path with non-square crop: confirms shape compatibility, no image selection involved.
 wide,wctx=v_open_local(root,chosen,.1,768,80,0);v_predict(wide,False)
 metrics={'neutral_max_abs':err,'joint_edit_mean_abs_effect':effect,'outside_mask_latent_max_abs':outside_error,
          'region_counts':{l:len(cat) for l,cat in cats.items()},'zoom_512':ctx['spec'],'zoom_768':wctx['spec']}
 v_dump('preflight.json',metrics);source.save(V_OUT/'000_original.png')
 print(json.dumps(metrics,indent=2))
 atlas=[];labels=[]
 for l,cat in cats.items():
  alternatives=sorted([r for r in cat.values() if r['record']['kind']=='connected_component'],key=lambda r:abs(r['record']['fraction']-.1))[:3]
  for r in alternatives:
   view,crop,spec=v_region_views(root,r,'preflight_'+r['record']['id']);atlas.append(view);labels.append(r['record']['id']+f" {spec['magnification']}x")
 pic=frame_strip(atlas,labels,width=160);pic.save(V_OUT/'preflight_region_atlas.png');display(pic)
 sanity=v_vision([source,source],['A: identical source','B: identical source']);v_dump('observer_identical_check.json',sanity);print('Observer identical-image check:',sanity['report'])
 v_refresh(root);return root
V_ROOT=v_preflight()
V_CALLS=0
print('Preflight passed. Run budget starts at zero; preflight images are labeled separately.')


In [ ]:
# 50. Parallel numerical maps + independent visual observation; focused detail retrieval
V_OBSERVER+=' Keep the entire response below 150 words. Prioritize concrete differences and ambiguity over a checklist.'
v_dump('prompts.json',{'jev':V_PROMPTS,'visual_observer':V_OBSERVER})
# Correct the initial sanity check's suggestive panel labels; this check is blind.
V_BLIND_CHECK=v_vision([V_ROOT['image'],V_ROOT['image']],['A','B'])
v_dump('observer_blind_identical_check.json',V_BLIND_CHECK);print('Blind A/B check:',V_BLIND_CHECK['report'])
V_POOL=ThreadPoolExecutor(max_workers=5);V_MAP_CACHE={};V_INSPECTED=set()
def v_compact_params(s):
 return {n:{h:{k:v for k,v in p.items() if k not in ['source','destination']} for h,p in hs.items()} for n,hs in s['params'].items()}
def v_history(region=None):
 matches=[r for r in V_LEDGER if region is not None and r.get('region_id')==region]
 ids={r['decision'] for r in matches[-4:]};recent=matches[-4:]+[r for r in V_LEDGER[-6:] if r['decision'] not in ids]
 return recent

def v_map_layer(s,layer,cat,decision):
 cap=s['captures'][V_TARGETS[layer]]
 state={'trajectory':{'task':s['task'],'noise_index':s['i'],'revision':s['revision']},
   'definitions':{'feature_dispersion':'1 minus norm of mean unit feature; not a quality score','occupancy4x4':'mean mask membership per cell','fraction':'fraction of layer tokens','entropy':'natural-log attention entropy of up to128 sampled queries','head_rms':'RMS of head output after gating; units only comparable within same module'},
   'regions':{rid:r['record'] for rid,r in cat.items()},'heads':{str(h):{'entropy':round(cap['entropy'][h],4),'rms':round(cap['head_rms'][h],4),'group_mass':np.round(cap['group_mass'][h],3).tolist()} for h in range(8)},
   'recent_interventions':v_history(),'visual_estimate':s.get('observation',{}),'correspondence':s.get('correspondence',{}).get(V_TARGETS[layer],[]),
   'coverage':'All6 similarity groups; largest4 connected components per group, their feature subdivisions and adjacent unions. Smaller omitted pieces are not assumed irrelevant.'}
 qs={'region':v_choice(V_PROMPTS['region'],{**{rid:f"Region {rid}; details in regions[{rid}]." for rid in cat},'unresolved':'No region is distinguished by evidence.'})}
 for g in range(6):
  qs[f'representation_{g}']=v_choice(V_PROMPTS['representation']+f' Here group is regions[L{layer}G{g}]; its alternatives share parent_group={g}.',{'current':'Retain the similarity group.','connected_components':'Inspect its spatially separate pieces.','feature_subdivision':'Inspect variation within a component.','unresolved':'Evidence does not distinguish representations.'})
 return v_call(f'{decision:03}_mapL{layer}',state,qs)

def v_maps(s,decision):
 key=(s['task'],s['revision'],len(V_LEDGER))
 cats={l:v_catalogue(s,l) for l in range(4)}
 visual=V_POOL.submit(v_vision,[s['image']],['Current active view'])
 jobs={l:V_POOL.submit(v_map_layer,s,l,cats[l],decision) for l in range(4)}
 answers={l:f.result() for l,f in jobs.items()};s['observation']=visual.result()
 shortlist={}
 for l,ans in answers.items():
  probs=ans['region']['probabilities'];ranked=sorted(cats[l],key=lambda rid:probs.get(rid,0),reverse=True)
  # Return actual source records, including runner-up, rather than a generated summary.
  for rid in ranked[:2]:
   r=cats[l][rid];r['map_probability']=probs.get(rid,0);r['map_unresolved_probability']=probs.get('unresolved',0)
   r['representation_preference']=ans[f"representation_{r['record']['parent_group']}"]['choice'];shortlist[rid]=r
 v_dump(f'{decision:03}_map_evidence.json',{'answers':answers,'selected_records':{rid:{**r['record'],'map_probability':r['map_probability'],'representation_preference':r['representation_preference']} for rid,r in shortlist.items()},'visual':s['observation']})
 return cats,shortlist,answers

def v_proposal(s,region,decision):
 # Spatial masks map to all scales; all8 heads can be selected at each scale.
 state={'region':region['record'],'history':v_history(region['record']['id']),'heads':{},'current_parameters':v_compact_params(s),'correspondence':s.get('correspondence',{})}
 qs={}
 for l,name in enumerate(V_TARGETS):
  cap=s['captures'][name];gh,gw=cap['shape'];rm=F.interpolate(region['mask'][None,None].float(),(gh,gw),mode='nearest').flatten().bool()
  rows=rm[cap['query_indices']];probs=cap['attention_rows'].float();labels=s['groups'][name]
  measurements={}
  for h in range(8):
   mass=[float(probs[h,rows][:,labels==g].sum(-1).mean()) if rows.any() else None for g in range(6)]
   measurements[str(h)]={'region_query_samples':int(rows.sum()),'destination_group_mass':mass,'entropy':cap['entropy'][h],'rms':cap['head_rms'][h]}
  state['heads'][f'L{l}']=measurements
  qs[f'head{l}']=v_choice(V_PROMPTS['head']+f' Inspect heads.L{l}.',{str(h):f'Head{h}' for h in range(8)})
  qs[f'dest{l}']=v_choice(V_PROMPTS['edge']+f' Inspect all heads.L{l}; the source is the spatial projection of region.',{str(g):f'Feature group{g}' for g in range(6)})
  for kind,step in [('bias',.2),('temperature',.1),('gate',.1)]:
   qs[f'{kind}{l}']=v_choice(V_PROMPTS['parameter']+f' Parameter {kind} at layer{l}; selected head is resolved by code after this independent proposal pass. Changes will be tested jointly.',{'down':f'Decrease by{step}','hold':'No change','up':f'Increase by{step}'})
 answers=v_call(f'{decision:03}_proposal',state,qs);proposal=v_clone(s);edits=[]
 for l,name in enumerate(V_TARGETS):
  hd=answers[f'head{l}']['choice'];dg=int(answers[f'dest{l}']['choice']);cap=s['captures'][name]
  old=s['params'].get(name,{}).get(hd,{'bias':0.,'temperature':1.,'gate':1.})
  p={k:old[k] for k in ['bias','temperature','gate']};p['source']=region['mask'].clone();p['destination']=(s['groups'][name]==dg).reshape(cap['shape'])
  for kind,step,bounds in [('bias',.2,(-1.5,1.5)),('temperature',.1,(.7,1.4)),('gate',.1,(.5,1.5))]:
   sign={'down':-1,'hold':0,'up':1}[answers[f'{kind}{l}']['choice']];p[kind]=round(float(np.clip(p[kind]+sign*step,*bounds)),4)
  proposal['params'].setdefault(name,{})[hd]=p;edits.append({'layer':l,'head':int(hd),'destination_group':dg,**{k:p[k] for k in ['bias','temperature','gate']}})
 return proposal,edits,answers
print('Four scale maps run concurrently with Qwen. Parameter proposals are explicitly untested joint vectors.')


In [ ]:
# 51. Matched intervention branches and comprehensive reviews (source evidence retained)
V_REVIEW_CHOICES={'supported':'The supplied observations support the stated condition.','contradicted':'The supplied observations contradict the stated condition.','unresolved':'The observations are ambiguous or disagree.','not_assessed':'Required observations are absent.'}
def v_review(before,after,mask,decision,tag):
 if mask is None:mask=Image.new('L',before.size,255)
 bbox=mask.point(lambda x:255 if x>64 else 0).getbbox() or (0,0,*before.size)
 x0,y0,x1,y1=bbox;pad=max(16,int(max(x1-x0,y1-y0)*.3));context=(max(0,x0-pad),max(0,y0-pad),min(before.width,x1+pad),min(before.height,y1+pad))
 local=v_vision([before.crop(bbox),after.crop(bbox)],['A','B'])
 surrounding=v_vision([before.crop(context),after.crop(context)],['A','B'])
 full=v_vision([before,after],['A','B'])
 pair=frame_strip([before,after],['A saved state','B candidate'],width=384);pair.save(V_OUT/f'{decision:03}_{tag}_review.png')
 evidence={'A':'saved starting view','B':'candidate view','local_structure':local,'context':surrounding,'full_image':full,
           'pixel_changes':v_delta(before,after,mask),'persistence':'not assessed by this review','observer_reliability':'Qwen2B estimates; numeric change is not anatomical correctness'}
 qs={'local':v_choice(V_PROMPTS['local'],V_REVIEW_CHOICES),'regression':v_choice(V_PROMPTS['regression'],V_REVIEW_CHOICES)}
 answer=v_call(f'{decision:03}_{tag}_review',{'review':evidence},qs)
 v_dump(f'{decision:03}_{tag}_review.json',{'evidence':evidence,'judgments':answer})
 return {'evidence':evidence,'judgments':answer}

@torch.inference_mode()
def v_probe(s,region,decision):
 proposal,edits,answers=v_proposal(s,region,decision)
 inverse=v_clone(s)
 for name,hs in proposal['params'].items():
  for hd,p in hs.items():
   old=s['params'].get(name,{}).get(hd,{'bias':0.,'temperature':1.,'gate':1.})
   r=copy.deepcopy(p)
   for key,bounds in [('bias',(-1.5,1.5)),('temperature',(.7,1.4)),('gate',(.5,1.5))]:r[key]=round(float(np.clip(2*old[key]-p[key],*bounds)),4)
   inverse['params'].setdefault(name,{})[hd]=r
 endpoints={};early={}
 for label,start in [('unchanged_controls',s),('proposal',proposal),('opposite',inverse)]:
  short=v_advance(start,2);long=v_advance(short,4);early[label]=short;endpoints[label]=long
  short['image'].save(V_OUT/f'{decision:03}_probe_{label}_2.png');long['image'].save(V_OUT/f'{decision:03}_probe_{label}_6.png')
  v_checkpoint(long,f'{decision:03}_probe_{label}')
 reports={}
 for label in ['proposal','opposite']:
  reports[label]={'two_steps':v_vision([early['unchanged_controls']['image'],early[label]['image']],['A','B']),
                  'six_steps':v_vision([endpoints['unchanged_controls']['image'],endpoints[label]['image']],['A','B']),
                  'delta2':v_delta(early['unchanged_controls']['image'],early[label]['image']),
                  'delta6':v_delta(endpoints['unchanged_controls']['image'],endpoints[label]['image'])}
 state={'paired_views':'A=unchanged controls; B=named branch. All start from same latent, noise, timestep.','region':region['record'],'joint_proposal':edits,'observations':reports}
 qs={'branch':v_choice(V_PROMPTS['branch'],{'unchanged_controls':'Advance along the unedited trajectory.','proposal':'Retain the proposed joint edit and its six-step endpoint.','opposite':'Retain the opposite joint edit and its six-step endpoint.','unresolved':'Evidence does not distinguish branches; retain unedited continuation.'})}
 for label in reports:qs['persistence_'+label]=v_choice(V_PROMPTS['persistence']+f' Inspect observations.{label}.',V_REVIEW_CHOICES)
 selected=v_call(f'{decision:03}_probe_review',state,qs);pick=selected['branch']['choice']
 actual='unchanged_controls' if pick=='unresolved' else pick
 v_dump(f'{decision:03}_probe_evidence.json',{'state':state,'answers':selected,'retained':actual})
 result=endpoints[actual];result['observation']=reports.get(actual,{}).get('six_steps',s.get('observation',{}))
 return result,{'branch':actual,'requested_branch':pick,'joint_edit':edits,'persistence':{k:v['choice'] for k,v in selected.items() if k.startswith('persistence')},'visual_reports':{k:v['six_steps']['report'] for k,v in reports.items()}}
print('Each probe retains all3 branches at2 and6 steps. Local, contextual and full-image review evidence stays separate.')


In [ ]:
# 52. Executable workflow: adaptive grouping, zoom choices, short local steps, and completion obligations
V_MAP_REUSE={};V_CURRENT=None;V_PARENT_CONTEXT=None;V_PENDING=None;V_START_TIME=None

def v_state_key(s):return (s['task'],s['revision'],s['i'])
def v_get_maps(s,d):
 key=v_state_key(s)
 if key not in V_MAP_REUSE:
  cats,short,answers=v_maps(s,d)
  # Independent group-representation and region answers are composed in code.
  for l,ans in answers.items():
   ranked=sorted(cats[l],key=lambda rid:ans['region']['probabilities'].get(rid,0),reverse=True)
   first=ranked[0];g=cats[l][first]['record']['parent_group'];pref=ans[f'representation_{g}']['choice']
   kind={'connected_components':'connected_component','feature_subdivision':'feature_subdivision','current':'feature_group'}.get(pref)
   alternatives=[rid for rid in ranked if cats[l][rid]['record']['parent_group']==g and cats[l][rid]['record']['kind']==kind]
   if alternatives:
    rid=alternatives[0];r=cats[l][rid];r['map_probability']=ans['region']['probabilities'].get(rid,0);r['representation_preference']=pref;short[rid]=r
  V_MAP_REUSE[key]=(cats,short,answers)
 return V_MAP_REUSE[key]

def v_local_settings(s,region,d):
 criteria={};specs={}
 for resolution in [512,768]:
  for margin in [.1,.25,.5]:
   spec=v_crop_spec(s['image'],region['mask'],margin,resolution)
   for restart in [50,65,80]:
    key=f'r{resolution}_m{int(margin*100)}_n{restart}';specs[key]=(margin,resolution,restart,spec)
    criteria[key]=f"Crop {spec['box']}; work{spec['working']}; magnification{spec['magnification']}x; context margin{margin}; restart index{restart} (50 more noise,80 less noise)."
 q=v_choice('Which listed crop, working resolution and noise entry provides a useful local refinement test for this region, considering context, magnification and previous attempts?',criteria)
 result=v_call(f'{d:03}_zoom',{'region':region['record'],'visual':s.get('observation',{}),'history':v_history(region['record']['id'])},{'setting':q})
 return specs[result['setting']['choice']]

def v_status(d,s,status,**extra):
 v_dump('status.json',{'status':status,'decisions_completed':d,'task':s['task'],'schedule_index':s['i'],'unet_calls':V_CALLS,'jev_requests':dict(V_API_COUNTS),'elapsed_seconds':round(time.time()-V_START_TIME,2) if V_START_TIME else 0,**extra})

@torch.inference_mode()
def run_adaptive():
 global V_CURRENT,V_PARENT_CONTEXT,V_PENDING,V_START_TIME
 V_START_TIME=time.time();s=v_clone(V_ROOT);ctx=None;pending=None;previous=v_clone(s);v_install();V_FRAMES[:]=[s['image'].copy()]
 try:
  for d in range(1,101):
   forced_completion=d>=81 or V_CALLS>=V_CONFIG['unet_budget']-V_CONFIG['completion_reserve']
   details={};short={};cats={};review=None
   if pending is not None:
    review=pending['review'];options={'accept_integration':'Retain candidate at saved parent noise level; schedule persistence review.','reject_integration':'Restore exact parent checkpoint.'}
    state={'phase':'full-image integration','review':review,'original_request':PROMPT if 'PROMPT' in globals() else 'Single realistic open hand on gray background','history':v_history()}
   elif forced_completion:
    options={'return_local':'Return the current predicted-clean local image for parent integration review.'} if ctx else {'advance':'Advance a short segment to complete the parent schedule; no new branches.'}
    state={'phase':'completion reserve','task':s['task'],'index':s['i'],'remaining_action_slots':101-d}
   else:
    cats,short,mapped=v_get_maps(s,d)
    options={}
    if s['i']<100:options['advance']='Advance two denoising steps with current controls.'
    if s['i']<=93 and V_CALLS<2200:
     for rid,r in short.items():options['probe:'+rid]=f"Test a joint multiscale attention intervention using {rid} as source region; unchanged/proposal/opposite branches observed at2 and6 steps."
    for rid,r in short.items():
     ik=(v_state_key(s),rid)
     if ik not in V_INSPECTED:options['inspect:'+rid]=f'Inspect highlighted footprint and contextual crop of {rid}; no generation change.'
     if ctx is None:
      spec=v_crop_spec(s['image'],r['mask'],.25,512)
      if spec['magnification']>1.05 and r['record']['fraction']<.65:
       options['open:'+rid]=f"Open local refinement of {rid}. Example512 crop{spec['box']} gives{spec['magnification']}x; choose margin,resolution,noise entry next. Parent state saved."
    if ctx:
     options['return_local']='Integrate current predicted-clean local result into saved parent; mandatory review follows.'
     options['abandon_local']='Restore saved parent without integrating local changes.'
     if d-ctx['opened_at']>=V_CONFIG['local_decision_limit']:options={k:v for k,v in options.items() if k in ['return_local','abandon_local']}
    else:
     options['reopen65']='Reopen the current image at schedule index65 with fixed recorded noise seed.'
     if d>1:options['restore']='Restore the previous committed state.'
    # Prevent identical no-op inspections; new fine masks are already explicit catalogue alternatives.
    state={'task':s['task'],'noise_index':s['i'],'revision':s['revision'],'original_request':'Studio photograph of a single human hand, open palm facing camera, all five fingers spread apart and fully visible, wrist entering from bottom, centered, plain dark gray background, realistic skin, sharp focus',
     'regions':{rid:{**r['record'],'map_probability':r.get('map_probability'),'representation_preference':r.get('representation_preference')} for rid,r in short.items()},
     'visual_observation':s.get('observation',{}),'recent_interventions':v_history(),'current_parameters':v_compact_params(s),
     'local_obligation':{'opened_at':ctx['opened_at'],'parent_index':ctx['parent']['i'],'crop':ctx['spec']} if ctx else None,
     'last_focused_inspection':s.get('inspection'), 'operation_counts':dict(__import__('collections').Counter(r['operation'] for r in V_RECORDS))}
   ans=v_call(f'{d:03}_action',state,{'operation':v_choice(V_PROMPTS['action'],options)},kind='action')
   action=ans['operation']['choice'];operation,_,rid=action.partition(':');before=v_clone(s);full_before=(ctx['parent']['image'] if ctx else s['image']).copy()
   if operation=='advance':
    steps=max(2,int(np.ceil((100-s['i'])/max(1,101-d)))) if forced_completion else 2
    s=v_advance(s,steps);details={'denoising_steps':steps}
   elif operation=='probe':
    s,details=v_probe(s,short[rid],d);previous=before
   elif operation=='inspect':
    r=short[rid];highlight,crop,spec=v_region_views(s,r,f'{d:03}_inspection')
    obs=v_vision([s['image'],highlight,crop],['Full active image','Selected footprint','Contextual crop'])
    s['inspection']={'region':r['record'],'crop':spec,'observation':obs};details=s['inspection'];V_INSPECTED.add((v_state_key(s),rid))
    # Invalidate only the map selection so newly observed spatial evidence reaches it.
    s['observation']=obs;V_MAP_REUSE.pop(v_state_key(s),None)
   elif operation=='open':
    margin,resolution,restart,spec=v_local_settings(s,short[rid],d)
    s,ctx=v_open_local(s,short[rid],margin,resolution,restart,d)
    details={'region':ctx['region'],'crop':spec,'restart':restart}
    ctx['parent']['image'].save(V_OUT/f'{d:03}_local_parent.png');ctx['mask'].save(V_OUT/f'{d:03}_local_mask.png')
   elif operation=='return_local':
    candidate=v_integrate(s,ctx);review=v_review(ctx['parent']['image'],candidate['image'],ctx['mask'],d,'integration')
    pending={'candidate':candidate,'parent':ctx['parent'],'mask':ctx['mask'],'review':review,'region':ctx['region']}
    details={'local_noise_index':s['i'],'parent_index':ctx['parent']['i'],'region':ctx['region'],'review':review}
    s=candidate;ctx=None
   elif operation=='abandon_local':s=v_clone(ctx['parent']);details={'abandoned_local':before['task']};ctx=None
   elif operation=='accept_integration':
    s=pending['candidate'];previous=pending['parent'];s['persistence_check']={'image':s['image'].copy(),'parent_image':previous['image'].copy(),'mask':pending['mask'],'accepted_at':d}
    details={'review':pending['review'],'accepted':True};pending=None
   elif operation=='reject_integration':s=v_clone(pending['parent']);details={'review':pending['review'],'accepted':False};pending=None
   elif operation=='reopen65':previous=before;s=v_reopen(s,65,d)
   elif operation=='restore':s=v_clone(previous);details={'restored_revision':s['revision']}
   if operation in ['advance','probe'] and s['i']<100:
    # Refresh group memberships only between committed states, never inside a matched probe.
    s=v_refresh(s,recluster=True)
   if operation in ['advance','probe'] and 'persistence_check' in s:
    p=s.pop('persistence_check');pr=v_review(p['image'],s['image'],p['mask'],d,'persistence')
    details['persistence_review']=pr
   if d%10==0 and pending is None:
    parent_image=ctx['parent']['image'] if ctx else s['image']
    details['periodic_full_review']=v_review(V_ROOT['image'],parent_image,None,d,'periodic')
   full=ctx['parent']['image'] if ctx else (pending['parent']['image'] if pending else s['image'])
   full.save(V_OUT/f'{d:03}_full.png');s['image'].save(V_OUT/f'{d:03}_active.png')
   if pending:pending['candidate']['image'].save(V_OUT/f'{d:03}_integration_candidate.png')
   v_checkpoint(s,f'{d:03}')
   record={'decision':d,'operation':operation,'action':action,'region_id':rid or details.get('region',{}).get('id'),
           'task':s['task'],'noise_index':s['i'],'unet_calls':V_CALLS,'jev_requests':dict(V_API_COUNTS),'action_answer':ans,
           'details':details,'parameters':v_compact_params(s),'full_image_delta':v_delta(full_before,full),
           'elapsed_seconds':time.time()-V_START_TIME}
   v_dump(f'{d:03}_decision.json',record);V_RECORDS.append(record)
   V_LEDGER.append({'decision':d,'operation':operation,'region_id':record['region_id'],'task':s['task'],'noise_index':s['i'],
    'result':{k:v for k,v in details.items() if k in ['branch','requested_branch','joint_edit','persistence','accepted','crop','restart','local_noise_index']},
    'pixel_change':record['full_image_delta'],'visual_observation':s.get('observation',{}).get('report','')[:900]})
   V_FRAMES.append(full.copy());V_CURRENT=s;V_PARENT_CONTEXT=ctx;V_PENDING=pending;v_status(d,s,'running')
   print(f"Decision {d:03}/100 | {operation} | {s['task']} | index{s['i']} | UNet{V_CALLS} | Jev{sum(V_API_COUNTS.values())}",flush=True)
   if operation in ['open','return_local','accept_integration','probe'] or d%5==0:
    display(frame_strip([full,s['image']],['Committed parent',f'Active: {operation}'],width=320))
   if d%10==0:
    frame_strip(V_FRAMES,[str(i) for i in range(len(V_FRAMES))],width=128).save(V_OUT/'all_decisions.png')
  # Deterministic completion is explicitly separated from Jev decisions.
  if pending is not None:s=v_clone(pending['parent']);V_EXECUTION_WARNINGS.append('Unresolved final integration restored to parent.')
  if ctx is not None:s=v_clone(ctx['parent']);V_EXECUTION_WARNINGS.append('Unresolved local task restored to parent at finalization.')
  completion=0
  while s['i']<100:
   s=v_advance(s,2);completion+=1;s['image'].save(V_OUT/f'completion_{completion:03}.png');V_FRAMES.append(s['image'].copy())
  final_review=v_review(V_ROOT['image'],s['image'],None,100,'final')
  s['image'].save(V_OUT/'final.png');V_FRAMES[0].save(V_OUT/'progression.gif',save_all=True,append_images=V_FRAMES[1:],duration=450,loop=0)
  v_dump('summary.json',{'decisions':len(V_RECORDS),'operations':dict(__import__('collections').Counter(r['operation'] for r in V_RECORDS)),
       'jev_requests':V_API_COUNTS,'unet_calls':V_CALLS,'automatic_completion_segments':completion,'warnings':V_EXECUTION_WARNINGS,'final_review':final_review,
       'quality_claim':'No anatomical improvement established solely by observer or internal measurements.'})
  V_CURRENT=s;v_status(100,s,'complete');return s['image']
 except Exception as exc:
  V_CURRENT=s;V_PARENT_CONTEXT=ctx;V_PENDING=pending;v_checkpoint(s,'failure')
  v_status(len(V_RECORDS),s,'error',error_type=type(exc).__name__,message=str(exc)[:500]);raise
 finally:pipe.unet.set_attn_processor(dict(V_BASE_PROCESSORS))
print('Workflow ready:100 decisions,600 analysis requests, short denoising segments, mandatory integration review.')


In [ ]:
# 53. Final protocol checks, source archive, and offline W&B export/CLI sync program
import shutil
print('Free disk GiB:',round(shutil.disk_usage(V_OUT).free/2**30,1))
print('Blind observer check:',V_BLIND_CHECK['report'])
# Remove a harmless numpy/torch coercion warning in the catalogue code before the run.
_catalogue_src=next(src for src in reversed(In[:-1]) if src.startswith('# 47.'))
_tree=ast.parse(_catalogue_src)
_func=next(n for n in _tree.body if isinstance(n,ast.FunctionDef) and n.name=='v_catalogue')
_clean_source=ast.get_source_segment(_catalogue_src,_func)
_old="mask=np.logical_or(base['mask'],regions[other]['mask']).numpy() if isinstance(np.logical_or(base['mask'],regions[other]['mask']),torch.Tensor) else np.logical_or(base['mask'].numpy(),regions[other]['mask'].numpy())"
_clean_source=_clean_source.replace(_old,"mask=np.logical_or(base['mask'].numpy(),regions[other]['mask'].numpy())")
exec(compile(_clean_source,'adaptive_catalogue.py','exec'),globals())
V_ROOT['observation']=v_vision([V_ROOT['image']],['Current active image'])
# API smoke check uses a literal typed condition; separately counted and archived.
smoke=v_call('preflight_api',{'neutral_attention_max_abs':0.0},{'identity':v_choice('Does the supplied neutral_attention_max_abs equal zero?',{'yes':'The supplied value is zero.','no':'The supplied value differs from zero.'})})
assert smoke['identity']['choice']=='yes'
V_API_COUNTS={'analysis':0,'action':0};V_API_CACHE={};V_USAGE=[]
for src in In:
 if any(src.startswith(f'# {n}.') for n in range(44,54)):
  number=src.split('.')[0].replace('# ','');(V_OUT/f'cell_{number}.py').write_text(src)
v_dump('manifest.json',{'config':V_CONFIG,'prompts':V_PROMPTS,'visual_prompt':V_OBSERVER,'source_image':'RESULTS[(123,jev)] original image; unchanged prompt and model weights',
 'targets':V_TARGETS,'bounds':{'bias':[-1.5,1.5],'temperature':[.7,1.4],'gate':[.5,1.5]},
 'group_options':'6 feature groups per scale, largest4 connected components each, two feature subdivisions, adjacent unions; all8heads',
 'completion_policy':'Final20 decisions resolve local work and complete parent trajectory; any automatic remainder explicitly logged',
 'preflight_excluded_from_budget':True,'evidence':'All decisions, all probe branches, sampled attention tensors, checkpoints, masks and reviews retained',
 'limitations':'Qwen2B remains fallible; no calibrated anatomical quality metric; numerical attention data does not identify anatomy.'})
V_WANDB_ID='seed123-adaptive-'+V_OUT.name.split('_')[-1]
V_WANDB_URL='https://wandb.ai/peepaclan/jev-diffusion-control/runs/'+V_WANDB_ID
(V_OUT/'wandb_run_url.txt').write_text(V_WANDB_URL)
V_EXPORT_SCRIPT=r'''
import os,sys,json,pathlib,zipfile,subprocess
import wandb
p=pathlib.Path(sys.argv[1]).resolve();rid=sys.argv[2]
config=json.loads((p/'config.json').read_text());summary=json.loads((p/'summary.json').read_text())
run=wandb.init(entity='peepaclan',project='jev-diffusion-control',id=rid,name=rid,group='original-seed123',job_type='adaptive-groups',mode='offline',dir=str(p),config=config,save_code=False,settings=wandb.Settings(console='off',disable_git=True))
run.define_metric('decision');run.define_metric('*',step_metric='decision')
rows=[]
run.log({'decision':0,'images/full':wandb.Image(str(p/'000_original.png')),'images/active':wandb.Image(str(p/'000_original.png'))})
for path in sorted(p.glob('[0-9][0-9][0-9]_decision.json')):
 r=json.loads(path.read_text());d=r['decision'];op=r['operation']
 data={'decision':d,'trajectory/noise_index':r['noise_index'],'compute/unet_calls':r['unet_calls'],'compute/jev_requests':sum(r['jev_requests'].values()),'progress/decisions':d,'change/mean_abs_rgb':r['full_image_delta']['mean_abs_rgb'],
       'images/full':wandb.Image(str(p/f'{d:03}_full.png')),'images/active':wandb.Image(str(p/f'{d:03}_active.png')),'operation/'+op:1,'elapsed_seconds':r['elapsed_seconds']}
 for pattern in [f'{d:03}_probe_*_2.png',f'{d:03}_probe_*_6.png',f'{d:03}_*review.png',f'{d:03}_inspection_footprint.png']:
  for media in sorted(p.glob(pattern)):data['evidence/'+media.stem[4:]]=wandb.Image(str(media))
 crop=r['details'].get('crop')
 if crop:data['refinement/magnification']=crop['magnification'];data['refinement/long_side']=crop['long_side']
 for li,(name,heads) in enumerate(r['parameters'].items()):
  for h,pars in heads.items():
   for k,v in pars.items():data[f'controls/{name}/head{h}/{k}']=v
 run.log(data)
 rows.append([d,op,r.get('region_id') or '',r['task'],r['noise_index'],json.dumps(r['details']),wandb.Image(str(p/f'{d:03}_full.png')),wandb.Image(str(p/f'{d:03}_active.png'))])
run.log({'decision':100,'images/final':wandb.Image(str(p/'final.png')),'decision_history':wandb.Table(columns=['decision','operation','region','task','noise_index','evidence','full_image','active_image'],data=rows)})
run.summary.update({k:v for k,v in summary.items() if k!='final_review'})
run.summary['final_review']=summary['final_review']
archive=p/'reviewable_evidence.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in p.iterdir():
  if f.is_file() and f.suffix in ['.json','.png','.gif','.py'] and not f.name.startswith('wandb_export'):z.write(f,f.name)
artifact=wandb.Artifact(rid+'-evidence',type='experiment');artifact.add_file(str(archive));run.log_artifact(artifact)
offline=pathlib.Path(run.dir).parent;run.finish()
(p/'wandb_offline_path.txt').write_text(str(offline))
cmd=[sys.executable,'-m','wandb','sync','--entity','peepaclan','--project','jev-diffusion-control',str(offline)]
print('Uploading with W&B CLI: python -m wandb sync <this run>',flush=True)
result=subprocess.run(cmd,check=False)
(p/'wandb_sync_status.json').write_text(json.dumps({'exit_code':result.returncode,'url':'https://wandb.ai/peepaclan/jev-diffusion-control/runs/'+rid}))
sys.exit(result.returncode)
'''
(V_OUT/'wandb_export.py').write_text(V_EXPORT_SCRIPT)
print('API smoke check passed. Protocol/source archived. Separate W&B destination:',V_WANDB_URL)


In [ ]:
# 54. Record failed observer sanity test; numerical identity overrides invented differences
V_OBSERVER_WARNING='Qwen2-VL-2B failed a blind identical-image check by inventing different finger counts. Its reports are uncertain suggestions, not reliable anatomy measurements. This failure is retained in observer_blind_identical_check.json.'
_v_vision_original=v_vision
@torch.inference_mode()
def v_vision(images,labels):
 hashes=[hashlib.sha256(im.convert('RGB').tobytes()+str(im.size).encode()).hexdigest() for im in images]
 if len(images)>1 and len(set(hashes))==1:
  return {'report':'The supplied views contain exactly identical RGB pixels and dimensions. No visual change occurred.','view_labels':labels,'source':'Exact pixel identity computed in code','observer_warning':V_OBSERVER_WARNING,'image_hash':hashes[0]}
 result=dict(_v_vision_original(images,labels));result['observer_warning']=V_OBSERVER_WARNING
 return result
V_ROOT['observation']=v_vision([V_ROOT['image']],['Current active image'])
_manifest=json.loads((V_OUT/'manifest.json').read_text());_manifest['observer_sanity']='FAILED blind identical pair';_manifest['observer_warning']=V_OBSERVER_WARNING
v_dump('manifest.json',_manifest)
(V_OUT/'cell_54.py').write_text(In[-1])
print(V_OBSERVER_WARNING)
print('Run starts next; Qwen reports carry this warning, exact identities are computed, and all images remain reviewable.')


In [ ]:
# 55. Start the adaptive100 experiment from the original seed123 image
assert not V_RECORDS and V_CALLS==0
print('Starting adaptive100. Output:',V_OUT.resolve(),flush=True)
print('W&B CLI upload follows completion:',V_WANDB_URL,flush=True)
v_status(0,V_ROOT,'starting')
V_FINAL=run_adaptive()
display(V_FINAL)
print('All100 decisions complete. Preparing the separate W&B run via CLI sync.',flush=True)
V_EXPORT_LOG=open(V_OUT/'wandb_export.log','w')
V_EXPORT_PROCESS=subprocess.Popen([sys.executable,str(V_OUT/'wandb_export.py'),str(V_OUT.resolve()),V_WANDB_ID],stdout=V_EXPORT_LOG,stderr=subprocess.STDOUT,start_new_session=True,env=os.environ.copy())
print('W&B exporter PID:',V_EXPORT_PROCESS.pid)


In [ ]:
# 56. Logged executor correction and exact-state resume (earlier decisions retained)
print('Retained decisions:',len(V_RECORDS),'requests:',V_API_COUNTS,'current:',V_CURRENT['task'],V_CURRENT['i'])
V_CONFIG['action_request_budget']=102 # permits the interrupted HTTP attempt; decision budget remains100
_api_src=next(src for src in In if src.startswith('# 45.'))
_api_node=next(n for n in ast.parse(_api_src).body if isinstance(n,ast.FunctionDef) and n.name=='v_call')
_api_code=ast.get_source_segment(_api_src,_api_node).replace("else 100","else V_CONFIG.get('action_request_budget',100)")
exec(compile(_api_code,'adaptive_api_resume.py','exec'),globals())
_run_src=next(src for src in In if src.startswith('# 52.'))
_node=next(n for n in ast.parse(_run_src).body if isinstance(n,ast.FunctionDef) and n.name=='run_adaptive')
_resume_src=ast.get_source_segment(_run_src,_node)
_resume_src=_resume_src.replace('def run_adaptive():','def resume_adaptive():')
_resume_src=_resume_src.replace("V_START_TIME=time.time();s=v_clone(V_ROOT);ctx=None;pending=None;previous=v_clone(s);v_install();V_FRAMES[:]=[s['image'].copy()]", "s=v_clone(V_CURRENT);ctx=V_PARENT_CONTEXT;pending=V_PENDING;previous=v_clone(ctx['parent'] if ctx else V_ROOT);v_install()")
_resume_src=_resume_src.replace('for d in range(1,101):','for d in range(len(V_RECORDS)+1,101):')
_resume_src=_resume_src.replace("s,ctx=v_open_local(s,short[rid],margin,resolution,restart,d)","s,ctx=v_open_local(s,short[rid],margin,resolution,restart,d);s=v_advance(s,2)")
_resume_src=_resume_src.replace("'restart':restart}","'restart':restart,'initial_denoising_steps':2}")
_resume_src=_resume_src.replace("Parent state saved.\"","Parent state saved; this operation also performs its first two denoising steps.\"")
_resume_src=_resume_src.replace("if d>1:options['restore']", "if d>1 and (s['z'].shape!=previous['z'].shape or not torch.equal(s['z'],previous['z'])):options['restore']")
exec(compile('@torch.inference_mode()\n'+_resume_src,'adaptive_resume.py','exec'),globals())
(V_OUT/'adaptive_resume.py').write_text('@torch.inference_mode()\n'+_resume_src)
correction={'after_decision':len(V_RECORDS),'change':'Opening a refinement includes two denoising steps; omit exact no-op restore. Existing decisions preserved; resume same saved latent/task.','interrupted_request_retained':True,'max_action_requests':102,'action_decisions':100}
v_dump('executor_correction.json',correction);V_EXECUTION_WARNINGS.append(correction['change'])
_manifest=json.loads((V_OUT/'manifest.json').read_text());_manifest['executor_correction']=correction;v_dump('manifest.json',_manifest)
print('Resume ready from decision',len(V_RECORDS)+1)


In [ ]:
# 57. Resume the same100-decision history, then upload through W&B CLI
V_FINAL=resume_adaptive()
display(V_FINAL)
print('All100 decisions completed. Starting W&B offline export + CLI sync.',flush=True)
V_EXPORT_LOG=open(V_OUT/'wandb_export.log','w')
V_EXPORT_PROCESS=subprocess.Popen([sys.executable,str(V_OUT/'wandb_export.py'),str(V_OUT.resolve()),V_WANDB_ID],stdout=V_EXPORT_LOG,stderr=subprocess.STDOUT,start_new_session=True,env=os.environ.copy())
print('W&B exporter PID:',V_EXPORT_PROCESS.pid,'Destination:',V_WANDB_URL)


In [ ]:
# 58. Complete chronology, exact outcome counts, and W&B CLI upload verification
from PIL import ImageDraw
summary=json.loads((V_OUT/'summary.json').read_text())
assert len(V_RECORDS)==100 and [r['decision'] for r in V_RECORDS]==list(range(1,101))
assert all((V_OUT/f'{i:03}_decision.json').exists() and (V_OUT/f'{i:03}_checkpoint.pt').exists() and (V_OUT/f'{i:03}_active.png').exists() and (V_OUT/f'{i:03}_full.png').exists() for i in range(1,101))
opens=[r for r in V_RECORDS if r['operation']=='open']
unique_crops={tuple(r['details']['crop']['box']) for r in opens}
print('Completed100/100. Operation counts:',summary['operations'])
print('Analysis/action requests:',summary['jev_requests'],'UNet calls:',summary['unet_calls'])
print('Distinct local crop extents:',len(unique_crops),'Local opens:',len(opens))
print('No retained local integration and no attention probe were selected. Endpoint changes come from completing the original parent trajectory.')
print('Observer reliability: FAILED blind identical-image check. Executor correction after decision17 is retained in manifest and history.')
def v_grid(kind):
 grid=Image.new('RGB',(10*160,10*185),'white');draw=ImageDraw.Draw(grid)
 for r in V_RECORDS:
  d=r['decision'];x=((d-1)%10)*160;y=((d-1)//10)*185
  im=Image.open(V_OUT/f'{d:03}_{kind}.png').convert('RGB');im.thumbnail((156,156))
  grid.paste(im,(x+(160-im.width)//2,y));draw.text((x+3,y+157),f"{d:03} {r['operation']}",fill='black')
 grid.save(V_OUT/f'all100_{kind}_grid.png');return grid
active_grid=v_grid('active');full_grid=v_grid('full')
display(HTML('<h3>All100 active views — every attempt retained</h3>'));display(active_grid)
display(HTML('<h3>Chronological endpoint</h3>'));display(V_FINAL)
for src in In:
 if any(src.startswith(f'# {n}.') for n in range(44,59)):
  number=src.split('.')[0].replace('# ','');(V_OUT/f'cell_{number}.py').write_text(src)
print('W&B exporter exit code:',V_EXPORT_PROCESS.poll())
sync=V_OUT/'wandb_sync_status.json'
print(sync.read_text() if sync.exists() else 'CLI export/sync is still in progress; see wandb_export.log.')
display(HTML(f'<p><a href="{V_WANDB_URL}" target="_blank">Separate W&B run: adaptive groups, seed123</a></p>'))


In [ ]:
# 59. W&B CLI sync status (safe to rerun; does not rerun diffusion)
print('Exporter exit:',V_EXPORT_PROCESS.poll())
_sync=V_OUT/'wandb_sync_status.json'
if _sync.exists():print(_sync.read_text())
else:
 _log=(V_OUT/'wandb_export.log').read_text()[-2500:]
 print(_log.replace(_api_key,'[redacted]').replace(os.environ['WANDB_API_KEY'],'[redacted]'))
print('Run:',V_WANDB_URL)


In [ ]:
# 60. Preregistered Jev diagnostic suite: evidence, ID/order and representation sensitivity
from decimal import Decimal
from concurrent.futures import ThreadPoolExecutor,as_completed
import itertools,random,statistics,collections
S_OUT=OUT/('jev_sensitivity_'+time.strftime('%Y%m%d_%H%M%S'));S_OUT.mkdir(parents=True,exist_ok=False)
S_MODEL='jev-1.13.0';S_SEED=20260922
S_RULE='A candidate is eligible only if target_error_after_2_steps and target_error_after_6_steps are BOTH strictly less than target_error_before, and outside_mean_abs_change is at most0.02 (equality is allowed). Among eligible candidates choose the UNIQUE smallest target_error_after_6_steps. If neither is eligible choose none. If eligible candidates tie for smallest final error choose tie. Lower target_error is better; this is a synthetic diagnostic quantity, not a measured hand-quality metric.'
S_QUESTION='Apply selection_rule to candidates. Which candidate is eligible and has the unique smallest final target error? Choose none when no candidate is eligible; choose tie when eligible candidates tie for smallest final error. Computed comparisons, when present, were calculated in code and represent the same rule; do not infer missing anatomy or image quality.'
S_ELIGIBILITY='Does candidate at candidates[{cid}] satisfy ALL eligibility conditions in selection_rule? Eligibility does not require winning against the other candidate. If computed comparisons are supplied, eligibility requires early_improved, late_improved and outside_within_limit all true.'
def s_record(before,early,late,drift):
 return dict(zip(['target_error_before','target_error_after_2_steps','target_error_after_6_steps','outside_mean_abs_change'],map(str,[before,early,late,drift])))
S_FAMILIES={
 'clear_gap':(s_record('.8','.5','.4','.01'),s_record('.8','.7','.65','.01')),
 'close_gap':(s_record('.8','.72','.7001','.01'),s_record('.8','.73','.7002','.01')),
 'drift_veto':(s_record('.8','.3','.25','.03'),s_record('.8','.7','.65','.01')),
 'persistence_veto':(s_record('.8','.3','.85','.01'),s_record('.8','.7','.65','.01')),
 'threshold_boundary':(s_record('.8','.3','.25','.020001'),s_record('.8','.7','.65','.02')),
 'tiny_scale':(s_record('.0000008','.0000005','.0000004','.01'),s_record('.0000008','.0000007','.0000006','.01')),
 'none_to_one':(s_record('.8','.5','.4','.03'),s_record('.8','.7','.65','.04')),
 'tie_to_one':(s_record('.8','.5','.4','.01'),s_record('.8','.6','.4','.01'))}
S_CASES=[]
for family,(a,b) in S_FAMILIES.items():
 base={'A':a,'B':b};changed=copy.deepcopy(base)
 if family=='none_to_one':changed['A']['outside_mean_abs_change']='.01'
 elif family=='tie_to_one':changed['B']['target_error_after_6_steps']='.3999'
 else:changed={'A':copy.deepcopy(b),'B':copy.deepcopy(a)}
 for evidence,records in enumerate([base,changed]):S_CASES.append({'family':family,'evidence':evidence,'records':records})
def s_truth(records):
 eligible={cid:(Decimal(r['target_error_after_2_steps'])<Decimal(r['target_error_before']) and Decimal(r['target_error_after_6_steps'])<Decimal(r['target_error_before']) and Decimal(r['outside_mean_abs_change'])<=Decimal('.02')) for cid,r in records.items()}
 yes=[cid for cid in records if eligible[cid]]
 if not yes:return 'none',eligible
 best=min(Decimal(records[cid]['target_error_after_6_steps']) for cid in yes);winners=[cid for cid in yes if Decimal(records[cid]['target_error_after_6_steps'])==best]
 return (winners[0] if len(winners)==1 else 'tie'),eligible
S_ID_SETS={'original':{'A':'L1G3C9','B':'L3G1C2'},'renamed':{'A':'r_907','B':'r_214'}}
S_REPRESENTATIONS=['raw','computed','both']
S_VARIANTS=list(itertools.product(S_ID_SETS,['AB','BA']))
def s_build(case,representation,id_set,order):
 records=case['records'];mapping=S_ID_SETS[id_set];candidates={}
 for canonical in order:
  r=records[canonical];numeric={k:float(v) for k,v in r.items()};_,eligible=s_truth(records)
  final=Decimal(r['target_error_after_6_steps']);others=[x for x in records if x!=canonical]
  computed={'early_improved':Decimal(r['target_error_after_2_steps'])<Decimal(r['target_error_before']),
   'late_improved':final<Decimal(r['target_error_before']),'outside_within_limit':Decimal(r['outside_mean_abs_change'])<=Decimal('.02'),
   'final_error_rank':1+sum(Decimal(records[o]['target_error_after_6_steps'])<final for o in others),
   'final_error_tied':any(Decimal(records[o]['target_error_after_6_steps'])==final for o in others)}
  candidates[mapping[canonical]]=numeric if representation=='raw' else computed if representation=='computed' else {**numeric,**computed}
 state={'selection_rule':S_RULE,'representation':representation,'computed_definitions':'final_error_rank=1 means lowest final error among all candidates; equal errors share rank. Filter eligibility before comparing rank. Flags are exact code-computed comparisons.','candidates':candidates}
 criteria={mapping[c]:f'Candidate {mapping[c]} in candidates: select only if eligible and uniquely best under selection_rule.' for c in order}
 criteria.update({'none':'Neither candidate satisfies all eligibility conditions.','tie':'Eligible candidates tie for the smallest final error.'})
 qs={'selection':{'type':'choice','instructions':S_QUESTION,'criteria':criteria}}
 for c in order:qs['eligible_'+mapping[c]]={'type':'noul','instructions':S_ELIGIBILITY.format(cid=mapping[c])}
 truth,eligible=s_truth(records)
 return {'model':S_MODEL,'state':state,'questions':qs},mapping,truth,eligible
S_JOBS=[]
for case,rep,(ids,order),repeat in itertools.product(S_CASES,S_REPRESENTATIONS,S_VARIANTS,range(2)):
 payload,mapping,truth,eligible=s_build(case,rep,ids,order)
 S_JOBS.append({'family':case['family'],'evidence':case['evidence'],'representation':rep,'id_variant':ids,'order':order,'repeat':repeat,'truth':truth,'eligible_truth':eligible,'mapping':mapping,'payload':payload})
assert len(S_JOBS)==384
for case in S_CASES[::2]:
 pair=next(c for c in S_CASES if c['family']==case['family'] and c['evidence']==1)
 assert s_truth(case['records'])[0]!=s_truth(pair['records'])[0]
S_PROTOCOL={'model':S_MODEL,'seed':S_SEED,'synthetic_requests':384,'questions_per_request':3,'families':list(S_FAMILIES),'representations':S_REPRESENTATIONS,'variants':S_VARIANTS,'repeats':2,
 'primary_metrics':['Oracle selection accuracy','Correct evidence-induced selection changes','Canonical choice invariance under renaming and order reversal','Choice agreement across equivalent representations','Exact-payload repeatability','Total variation distance of canonical probability distributions'],
 'limits':'Synthetic policy-following diagnostic; not a test of reading full tensors, anatomy, causal discovery, or tokenizer internals. Repeated variants are correlated; no population confidence claims.',
 'selection_rule':S_RULE,'selection_prompt':S_QUESTION,'eligibility_prompt':S_ELIGIBILITY}
(S_OUT/'protocol.json').write_text(json.dumps(S_PROTOCOL,indent=2))
(S_OUT/'oracle_cases.json').write_text(json.dumps(S_CASES,indent=2))
(S_OUT/'planned_jobs.json').write_text(json.dumps(S_JOBS,indent=2))
print('Diagnostic directory:',S_OUT.resolve());print(json.dumps(S_PROTOCOL,indent=2))
print('Example actual request (gold answers and test metadata are NOT sent):');print(json.dumps(S_JOBS[0]['payload'],indent=2))


In [ ]:
# 61. Real-context replay controls and uncached concurrent API runner
import re
# Four saved contexts: two action decisions and two layer1 region judgments.
_replay_paths=[]
for decision in [19,60]:
 found=sorted(V_OUT.glob(f'api_*_{decision:03}_action_request.json'));assert len(found)==1;_replay_paths.append(found[0])
_maps=sorted(V_OUT.glob('api_*_mapL1_request.json'));assert len(_maps)>=2
_replay_paths.extend([_maps[0],_maps[len(_maps)//2]])
S_REPLAY_JOBS=[]
for path in _replay_paths:
 original=json.loads(path.read_text());primary='operation' if 'operation' in original['questions'] else 'region'
 # Every occurrence is replaced, including option descriptions and nested references.
 identifiers=sorted(set(re.findall(r'L[0-3]G\d+(?:C\d+)?(?:S[01])?(?:ML[0-3]G\d+(?:C\d+)?)?',json.dumps(original))),key=lambda x:(-len(x),x))
 rename={old:f'unit_{700+j}' for j,old in enumerate(identifiers)}
 def transform(obj,mapping):
  if isinstance(obj,str):
   if not mapping:return obj
   return re.sub('|'.join(re.escape(k) for k in sorted(mapping,key=len,reverse=True)),lambda m:mapping[m.group()],obj)
  if isinstance(obj,list):return [transform(v,mapping) for v in obj]
  if isinstance(obj,dict):return {transform(k,mapping):transform(v,mapping) for k,v in obj.items()}
  return obj
 for ids,order,repeat in itertools.product(['original','renamed'],['AB','BA'],range(2)):
  mapping=rename if ids=='renamed' else {};payload=transform(copy.deepcopy(original),mapping)
  if order=='BA':
   for q in payload['questions'].values():
    if q['type']=='choice':q['criteria']=dict(reversed(list(q['criteria'].items())))
   if isinstance(payload['state'].get('regions'),dict):payload['state']['regions']=dict(reversed(list(payload['state']['regions'].items())))
  inverse_options={transform(k,mapping):k for k in original['questions'][primary]['criteria']}
  S_REPLAY_JOBS.append({'source':path.name,'primary':primary,'id_variant':ids,'order':order,'repeat':repeat,'inverse_options':inverse_options,'payload':payload})
(S_OUT/'real_replay_plan.json').write_text(json.dumps(S_REPLAY_JOBS,indent=2))
S_HTTP_LOCK=threading.Lock();S_HTTP_ATTEMPTS=0

def s_request(job,number,suite):
 global S_HTTP_ATTEMPTS
 payload=job['payload'];stem=f'{suite}_{number:04}'
 (S_OUT/(stem+'_request.json')).write_text(json.dumps(payload,indent=2))
 started=time.time();result=None
 for attempt in range(4):
  with S_HTTP_LOCK:S_HTTP_ATTEMPTS+=1
  try:
   response=requests.post('https://api.typesafe.ai/v1/systemone',headers={'Authorization':'Bearer '+_api_key},json=payload,timeout=90)
  except requests.RequestException as exc:
   if attempt==3:return {**{k:v for k,v in job.items() if k!='payload'},'error':type(exc).__name__,'number':number}
   time.sleep(2**attempt);continue
  if response.status_code==200:result=response.json();break
  if response.status_code not in [429,500,502,503,529]:
   return {**{k:v for k,v in job.items() if k!='payload'},'error':'HTTP '+str(response.status_code),'number':number}
  time.sleep(2**attempt)
 if result is None:return {**{k:v for k,v in job.items() if k!='payload'},'error':'retries_exhausted','number':number}
 (S_OUT/(stem+'_response.json')).write_text(json.dumps(result,indent=2))
 primary='selection' if suite=='synthetic' else job['primary'];ans=result['answers'][primary]
 assert ans['choice'] in payload['questions'][primary]['criteria']
 probs=ans['probabilities'];assert abs(sum(probs.values())-1)<.05
 if suite=='synthetic':
  inverse={v:k for k,v in job['mapping'].items()};inverse.update({'none':'none','tie':'tie'})
  canonical=inverse[ans['choice']];cp={inverse[k]:v for k,v in probs.items()}
  eligibility={c:result['answers']['eligible_'+cid]['noul'] for c,cid in job['mapping'].items()}
 else:
  inverse=job['inverse_options'];canonical=inverse[ans['choice']];cp={inverse[k]:v for k,v in probs.items()};eligibility={}
 record={**{k:v for k,v in job.items() if k!='payload'},'number':number,'choice':canonical,'probabilities':cp,'eligibility':eligibility,
   'model_returned':result.get('model'),'usage':result.get('usage',{}),'latency_seconds':time.time()-started,
   'payload_hash':hashlib.sha256(json.dumps(payload,separators=(',',':')).encode()).hexdigest()}
 if suite=='synthetic':record['correct']=canonical==job['truth']
 (S_OUT/(stem+'_scored.json')).write_text(json.dumps(record,indent=2));return record

def s_run_suite(jobs,suite):
 schedule=list(enumerate(jobs));random.Random(S_SEED+(suite=='real')).shuffle(schedule);out=[]
 with ThreadPoolExecutor(max_workers=8) as pool:
  futures={pool.submit(s_request,job,i,suite):i for i,job in schedule}
  for fut in as_completed(futures):
   record=fut.result();out.append(record)
   if len(out)%32==0 or len(out)==len(jobs):
    errors=sum('error' in r for r in out);acc=sum(r.get('correct',False) for r in out)/max(1,len(out)-errors) if suite=='synthetic' else None
    print(suite,len(out),'/',len(jobs),'errors',errors,'accuracy_so_far',round(acc,3) if acc is not None else 'unscored replay',flush=True)
 out.sort(key=lambda r:r['number']);(S_OUT/(suite+'_results.json')).write_text(json.dumps(out,indent=2));return out
print('Prepared384 synthetic requests +32 real-context replays. No cache. Eight concurrent HTTP requests; diffusion and Qwen are not run.')


In [ ]:
# 62. Validate perturbations, then run all sensitivity checks
for family in S_FAMILIES:
 cs=[c for c in S_CASES if c['family']==family]
 print(family,':',s_truth(cs[0]['records'])[0],'->',s_truth(cs[1]['records'])[0])
for source in {j['source'] for j in S_REPLAY_JOBS}:
 base=next(j for j in S_REPLAY_JOBS if j['source']==source and j['id_variant']=='original' and j['order']=='AB' and j['repeat']==0)
 renamed=next(j for j in S_REPLAY_JOBS if j['source']==source and j['id_variant']=='renamed' and j['order']=='AB' and j['repeat']==0)
 assert base['payload']!=renamed['payload'],('Rename failed',source)
 assert set(base['inverse_options'].values())==set(renamed['inverse_options'].values())
print('Perturbation checks passed. Starting384 synthetic calls.',flush=True)
S_SYNTHETIC=s_run_suite(S_JOBS,'synthetic')
print('Starting32 real-context replay calls.',flush=True)
S_REAL=s_run_suite(S_REPLAY_JOBS,'real')
print('All planned diagnostics completed. HTTP attempts:',S_HTTP_ATTEMPTS)


In [ ]:
# 63. Score all three sensitivities, probabilities, repeatability, and actual-context invariance
import pandas as pd
S_VALID=[r for r in S_SYNTHETIC if 'error' not in r];R_VALID=[r for r in S_REAL if 'error' not in r]
assert len(S_VALID)==384 and len(R_VALID)==32,'Report transport failures separately before interpreting accuracy.'
def s_tv(a,b):return .5*sum(abs(a.get(k,0)-b.get(k,0)) for k in set(a)|set(b))
def s_pairs(records,vary,baseline,alternative,keys):
 grouped={}
 for r in records:grouped.setdefault(tuple(r[k] for k in keys),{})[r[vary]]=r
 out=[]
 for key,g in grouped.items():
  if baseline not in g or alternative not in g:continue
  a,b=g[baseline],g[alternative]
  out.append({'pair':key,'same_choice':a['choice']==b['choice'],'tv_distance':s_tv(a['probabilities'],b['probabilities']),
              'before':a['choice'],'after':b['choice'],'before_correct':a.get('correct'),'after_correct':b.get('correct'),
              'before_prob':a['probabilities'],'after_prob':b['probabilities']})
 return out
S_PAIR_RESULTS={
 'id_rename':s_pairs(S_VALID,'id_variant','original','renamed',['family','evidence','representation','order','repeat']),
 'order_reverse':s_pairs(S_VALID,'order','AB','BA',['family','evidence','representation','id_variant','repeat']),
 'repeat':s_pairs(S_VALID,'repeat',0,1,['family','evidence','representation','id_variant','order']),
 'raw_vs_computed':s_pairs(S_VALID,'representation','raw','computed',['family','evidence','id_variant','order','repeat']),
 'raw_vs_both':s_pairs(S_VALID,'representation','raw','both',['family','evidence','id_variant','order','repeat']),
 'computed_vs_both':s_pairs(S_VALID,'representation','computed','both',['family','evidence','id_variant','order','repeat']),
 'real_id_rename':s_pairs(R_VALID,'id_variant','original','renamed',['source','order','repeat']),
 'real_order_reverse':s_pairs(R_VALID,'order','AB','BA',['source','id_variant','repeat']),
 'real_repeat':s_pairs(R_VALID,'repeat',0,1,['source','id_variant','order'])}
_evidence=s_pairs(S_VALID,'evidence',0,1,['family','representation','id_variant','order','repeat'])
S_PAIR_RESULTS['evidence_flip']=_evidence
S_METRICS={}
for name,pairs in S_PAIR_RESULTS.items():
 metric={'pairs':len(pairs),'same_choice':sum(p['same_choice'] for p in pairs),'mean_probability_TV':statistics.mean(p['tv_distance'] for p in pairs),'max_probability_TV':max(p['tv_distance'] for p in pairs)}
 if name=='evidence_flip':metric.update(correct_both=sum(p['before_correct'] and p['after_correct'] for p in pairs),changed_choice=sum(not p['same_choice'] for p in pairs))
 S_METRICS[name]=metric
S_ACCURACY=pd.DataFrame([{'representation':rep,'correct':sum(r['correct'] for r in S_VALID if r['representation']==rep),'n':sum(r['representation']==rep for r in S_VALID)} for rep in S_REPRESENTATIONS])
S_FAMILY_TABLE=pd.DataFrame([{'family':family,**{rep:sum(r['correct'] for r in S_VALID if r['family']==family and r['representation']==rep) for rep in S_REPRESENTATIONS}} for family in S_FAMILIES])
S_ELIGIBLE_ACCURACY={rep:{'correct':sum((p>=.5)==r['eligible_truth'][c] for r in S_VALID if r['representation']==rep for c,p in r['eligibility'].items()),'n':sum(2 for r in S_VALID if r['representation']==rep)} for rep in S_REPRESENTATIONS}
S_INCONSISTENCY=[r['number'] for r in S_VALID if r['choice'] in ['A','B'] and r['eligibility'][r['choice']]<.5]
S_RESULT={'selection_accuracy':S_ACCURACY.to_dict('records'),'eligibility_accuracy_threshold_0_5':S_ELIGIBLE_ACCURACY,'metrics':S_METRICS,
 'selection_eligibility_contradictions':S_INCONSISTENCY,'models_returned':sorted(set(r['model_returned'] for r in S_VALID+R_VALID)),
 'requests_completed':len(S_VALID)+len(R_VALID),'http_attempts':S_HTTP_ATTEMPTS,'transport_errors':0,
 'limits':S_PROTOCOL['limits'],'context_sources':[p.name for p in _replay_paths]}
(S_OUT/'summary.json').write_text(json.dumps(S_RESULT,indent=2));(S_OUT/'paired_results.json').write_text(json.dumps(S_PAIR_RESULTS,indent=2))
S_ACCURACY.to_csv(S_OUT/'accuracy.csv',index=False);pd.DataFrame(S_METRICS).T.to_csv(S_OUT/'sensitivity.csv')
print('SELECTION ACCURACY');display(S_ACCURACY)
print('ACCURACY BY FAMILY (16 cases per representation per family)');display(S_FAMILY_TABLE)
print('PAIRWISE SENSITIVITY');display(pd.DataFrame(S_METRICS).T)
print('ELIGIBILITY SUBQUESTIONS:',S_ELIGIBLE_ACCURACY,'Inconsistent selection/eligibility:',len(S_INCONSISTENCY))
print('ACTUAL-CONTEXT CHOICE CHANGES')
for name in ['real_id_rename','real_order_reverse','real_repeat']:
 for pair in S_PAIR_RESULTS[name]:
  if not pair['same_choice']:print(name,pair['pair'],pair['before'],'->',pair['after'],'TV',round(pair['tv_distance'],4))
print('All results:',S_OUT.resolve())


In [ ]:
# Readout: sensitivity checks (no additional API calls)
from IPython.display import Markdown
S_READOUT = """
## Jev sensitivity checks — completed

**416 uncached requests to jev-1.13.0: 384 synthetic cases and 32 saved-context replays. No transport errors.**

| Check | Compact synthetic cases | Actual diffusion prompts |
|---|---|---|
| Change evidence so the correct answer changes | 192/192 pairs changed correctly | Not tested: no ground-truth best operation |
| Rename candidate IDs consistently | 192/192 preserved their choice | 12/16 preserved; **4 changed** |
| Reverse option order | 192/192 preserved their choice | 11/16 preserved; **5 changed** |
| Repeat identical request | 192/192 preserved their choice | 15/16 preserved; **1 changed** |
| Raw numbers versus code-computed comparisons | 128/128 agreed; both 128/128 correct | Not tested |

**Representation affected probabilities even when the selected option stayed correct.** Raw-versus-computed mean total variation was 0.134 (maximum 0.43), versus 0.010 mean for exact repeats. Total variation measures the amount of probability mass redistributed, from 0 (identical) to 1 (disjoint).

For an actual decision-60 prompt, reversing order changed return_local into inspect:L3G5C5. In a feature-group prompt, consistently renaming IDs changed unresolved into canonical group L1G3. The underlying evidence was held fixed.

**Interpretation:** Jev can use compact numerical evidence under the explicit tested rule. Our larger action/group contexts show sensitivity to labels and ordering, so a plausible explanation or a confident choice alone is insufficient evidence that a diffusion action is well grounded. These tests do not identify its tokenizer or establish anatomical understanding, causal understanding, or image improvement.

**Limits:** Eight synthetic templates with correlated variants, and only four real contexts. Paired counts are not independent tasks. Real-context changes measure consistency, not correctness. The order test reverses option order and the region-list order together; it does not isolate their separate contributions. Evidence and representation sensitivity in full real contexts remain untested.

**Next implementation recommendation:** retain raw measurements alongside code-computed comparisons; verify hard numerical constraints in code; use compact evidence cards with opaque IDs whose metadata is explicit; check agreement under renaming/reordering before treating an uncertain architectural choice as established. Keep every disagreement in the log. These changes still need validation on real contexts before another long generation run.
"""
(S_OUT/'readout.md').write_text(S_READOUT)
display(Markdown(S_READOUT))
print('Exact prompts, raw responses, oracle, paired results and summary:', S_OUT.resolve())

In [ ]:
# Create a separate benchmark notebook; do not rerun prior experiments.
import nbformat, ast
BENCH_NOTEBOOK=Path('/workspace/crazy_exp/Jev_Checkpoint_Decision_Benchmark.ipynb')
assert not BENCH_NOTEBOOK.exists(), 'Benchmark notebook already exists; inspect before replacing.'
_bnb=nbformat.v4.new_notebook(cells=[nbformat.v4.new_markdown_cell('# Jev checkpoint decision benchmark\n\nIndependent record of paired interventions, evidence ablations, and blinded review. Reuses the existing SD1.5 kernel without rerunning prior experiments. All outcomes are retained.')],metadata={'kernelspec':{'display_name':'Python 3 (ipykernel)','language':'python','name':'python3'}})
nbformat.write(_bnb,BENCH_NOTEBOOK)
print('Created:',BENCH_NOTEBOOK)
_src_nb=nbformat.read('Jev_Attention_Downsampling_Hands.ipynb',as_version=4)
for i,c in enumerate(_src_nb.cells):
 if c.cell_type!='code':continue
 try:_tree=ast.parse(c.source)
 except SyntaxError:continue
 _defs=[n.name for n in _tree.body if isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef,ast.ClassDef))]
 if _defs:print(i, _defs)
print('GPU:',torch.cuda.get_device_name(), 'Allocated GiB:',round(torch.cuda.memory_allocated()/2**30,2))
print('Live relevant globals:',[k for k in list(globals()) if k.startswith(('v_','V_','qwen','QWEN'))])

In [ ]:
# Inspect reusable inference and observation interfaces, without credentials.
for _i in [60,61,62,63,66,69]:
 print('\nSOURCE CELL',_i)
 _c=_src_nb.cells[_i].source
 if _i==60:
  _t=ast.parse(_c);print('\n'.join(ast.get_source_segment(_c,n) for n in _t.body if isinstance(n,ast.FunctionDef)))
 else:print(_c)
print('ROOT_KEYS',list(V_ROOT));print('CONFIG',V_CONFIG)


In [ ]:
# 1. Inspect inherited interfaces; this notebook shares the source kernel.
assert 'pipe' in globals() and 'V_ROOT' in globals()
for _i in [10,34,64,65]:
 print('\nINTERFACE',_i);print(_src_nb.cells[_i].source)
print('Prompt:',PROMPT if 'PROMPT' in globals() else 'see source')
print('Scheduler:',type(A_SCHED).__name__,dict(A_SCHED.config))
print('Root:',V_ROOT['i'],V_ROOT['z'].shape,'Device:',pipe.device,'Targets:',V_TARGETS)
print('Observer globals:',[(k,type(v).__name__) for k,v in list(globals().items()) if any(s in k.lower() for s in ['qwen','vlm','processor','vision_model']) and not k.startswith('_')])
import transformers
print('Transformers:',transformers.__version__)
